# FINAL Kaggle Notebook for GRACE Baseline 2

This notebook is self-contained. All `baseline2` module and step sources are written directly into the notebook and then materialized into files under `/kaggle/working` using `%%writefile`.

What it does:
- configures the run in one place,
- installs missing packages,
- prepares the Kaggle workspace,
- writes all `baseline2` code from notebook cells into the working directory,
- auto-downloads `devign` when needed,
- restores prior artifacts from a previous Kaggle output dataset when available,
- runs steps `00` to `08`,
- prints the final evaluation summary.

The notebook is preconfigured for `devign` and chunked full-test execution on Kaggle.
It will automatically run all remaining chunks in sequence inside one session. If Kaggle stops the session early, save the output as a Kaggle dataset, attach that output dataset on the next run, and the notebook will auto-resume from the next unfinished chunk.


In [ ]:
from pathlib import Path

# =========================
# Preconfigured for Kaggle
# No edits needed for Devign
# =========================

DATASET_NAME = 'devign'  # 'devign' | 'bigvul' | 'reveal'

# Kaggle paths
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working/vulguardvn-final')

# Dataset sources from Kaggle Input.
# Set only the paths needed for the dataset you want to run.
DEVIGN_SOURCE_PATH = None
BIGVUL_SOURCE_PATH = None
REVEAL_SOURCE_DIR = None

# Optional direct download config for datasets when Kaggle Input is not mounted.
AUTO_DOWNLOAD_DATASET_IF_MISSING = True
DEVIGN_DOWNLOAD_URL = 'https://raw.githubusercontent.com/madlag/CodeXGLUE/main/Code-Code/Defect-detection/dataset/function.json'
DEVIGN_ARCHIVE_MEMBER = ''
BIGVUL_DOWNLOAD_URL = ''
BIGVUL_ARCHIVE_MEMBER = ''

# Optional local model folders uploaded as Kaggle datasets.
RETRIEVAL_MODEL_SOURCE_DIR = None
LOCAL_LLM_SOURCE_DIR = None

# Optional Hugging Face token. Public defaults work without it, but a token can make downloads more reliable.
HF_TOKEN = ''

# Staging toggles
USE_SYMLINKS_WHEN_POSSIBLE = True
COPY_MODELS_INSTEAD_OF_LINK = False
RESET_WORKING_ROOT = False
AUTO_DOWNLOAD_MISSING_MODELS = True
AUTO_DOWNLOAD_REVEAL_IF_MISSING = False
RESTORE_PREVIOUS_KAGGLE_OUTPUT = True
PREVIOUS_OUTPUT_SOURCE_DIR = None

# Core pipeline options
GRAPH_BACKEND = 'auto'            # 'auto' | 'heuristic' | 'joern'
RETRIEVAL_MODEL_ID = 'microsoft/unixcoder-base-nine'
LOCAL_LLM_MODEL_ID = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
PREFILTER_MODEL_NAME = 'hybrid_multiview_prefilter'

# LLM / inference options
# These defaults are tuned for better F1 with a controlled LLM budget on Kaggle T4.
LOAD_IN_4BIT = True
CALL_LLM_FOR_INSPECT = True
CALL_LLM_FOR_HIGH = False
MAX_TEST_SAMPLES = None
TEST_CHUNK_SIZE = 64
RUN_ALL_TEST_CHUNKS_IN_ONE_RUN = True
INSPECT_DEMOS = 2
HIGH_RISK_DEMOS = 1
MAX_NEW_TOKENS = 96
DEMO_CHAR_LIMIT = 800
PROMPT_CODE_CHAR_LIMIT = 1800
PROMPT_TOP_LINES_LIMIT = 3
PROMPT_TOP_LINE_CHAR_LIMIT = 120
PROMPT_SLICES_CHAR_LIMIT = 900
PROMPT_NODE_INFO_CHAR_LIMIT = 900
PROMPT_EDGE_INFO_CHAR_LIMIT = 900
RESUME_RUN = True

# Speed / training options
FEATURE_BATCH_SIZE = 16
FEATURE_PROGRESS_EVERY = 256
BUILD_PROGRESS_EVERY = 250
PREFILTER_BATCH_SIZE = 128
PREFILTER_EPOCHS = 10
PREFILTER_LEARNING_RATE = 7e-4
TARGET_RECALL = 0.995
DIRECT_ACCEPT_MIN_PROBABILITY = 0.20
HIGH_RISK_THRESHOLD_STRATEGY = 'f1'
HIGH_RISK_TARGET_PRECISION = 0.70

# Optional feature limits for debugging
GRACE_FEATURE_LIMIT = None
GRACE_FEATURE_LIMIT_TRAIN = None
GRACE_FEATURE_LIMIT_VAL = None
GRACE_FEATURE_LIMIT_TEST = None


## Variant B2

Calibration plus constrained routing

Adds threshold search with recall and budget constraints over the calibrated scores.


In [ ]:
# Variant override injected after the base Kaggle config.
EXPERIMENT_VARIANT = 'B2'
EXPERIMENT_NAME = 'Calibration plus constrained routing'
EXPERIMENT_NOTES = 'Adds threshold search with recall and budget constraints over the calibrated scores.'

BASE_PREFILTER_MODEL_NAME = PREFILTER_MODEL_NAME
VARIANT_OUTPUT_SUFFIX = EXPERIMENT_VARIANT.lower()
PREFILTER_MODEL_NAME = f"{BASE_PREFILTER_MODEL_NAME}_{VARIANT_OUTPUT_SUFFIX}"

GRACE_EXPERIMENT_VARIANT = EXPERIMENT_VARIANT
GRACE_VARIANT_OUTPUT_SUFFIX = VARIANT_OUTPUT_SUFFIX
GRACE_PREDICTION_FILE_STEM = f"grace_hybrid_predictions_{VARIANT_OUTPUT_SUFFIX}"
GRACE_RUN_STATE_FILE_STEM = f"grace_hybrid_run_state_{VARIANT_OUTPUT_SUFFIX}"
GRACE_EVALUATION_FILE_STEM = f"grace_hybrid_evaluation_summary_{VARIANT_OUTPUT_SUFFIX}"

TARGET_RECALL = 0.915
DIRECT_ACCEPT_MIN_PROBABILITY = 0.2
HIGH_RISK_THRESHOLD_STRATEGY = 'f1'
HIGH_RISK_TARGET_PRECISION = 0.7
GRAPH_BACKEND = 'stable'

GRACE_CALIBRATION_METHOD = 'auto'
GRACE_ROUTING_MODE = 'constrained'
GRACE_ROUTING_OBJECTIVE = 'accuracy'
GRACE_ROUTING_INSPECT_PROXY = 'probability'
GRACE_ROUTING_RECALL_FLOOR = 0.915
GRACE_LLM_BUDGET = 0.15
GRACE_TAU_NEG_MIN = 0.02
GRACE_TAU_NEG_MAX = 0.3
GRACE_TAU_NEG_STEPS = 15
GRACE_TAU_POS_MIN = 0.45
GRACE_TAU_POS_MAX = 0.9
GRACE_TAU_POS_STEPS = 19
GRACE_FORCE_REBUILD_FEATURES = False
GRACE_FEATURE_STORE_SUFFIX = ''
GRACE_HARD_NEGATIVE_MINING = False
GRACE_HARD_NEGATIVE_QUANTILE = 0.85
GRACE_HARD_NEGATIVE_WEIGHT = 2.5
GRACE_HARD_NEGATIVE_EPOCHS = 2
GRACE_PREFILTER_LOSS = 'bce'
GRACE_PREFILTER_FOCAL_GAMMA = 2.0
GRACE_TOKEN_MAX_TOKENS = 32000
GRACE_TOKEN_SEQUENCE_LENGTH = 384
GRACE_TOKEN_EMBEDDING_DIM = 96
GRACE_TOKEN_FILTERS = 96
GRACE_AST_MAX_TOKENS = 8000
GRACE_AST_SEQUENCE_LENGTH = 196
GRACE_AST_EMBEDDING_DIM = 64
GRACE_AST_FILTERS = 64
GRACE_PREFILTER_PROJECTION_DIM = 192
GRACE_PREFILTER_DENSE_UNITS = 192
GRACE_PREFILTER_DROPOUT = 0.25
GRACE_PREFILTER_EPOCHS = 10
GRACE_PREFILTER_LEARNING_RATE = 0.0007
GRACE_EVIDENCE_AWARE_VERIFIER = False
GRACE_MAX_NEW_TOKENS = 96
GRACE_PROMPT_TOP_LINES_LIMIT = 3
GRACE_PROMPT_TOP_LINE_CHAR_LIMIT = 120
GRACE_PROMPT_SLICES_CHAR_LIMIT = 900
GRACE_PROMPT_NODE_INFO_CHAR_LIMIT = 900
GRACE_PROMPT_EDGE_INFO_CHAR_LIMIT = 900
GRACE_CALL_LLM_FOR_HIGH = False


## Install Missing Packages

This installs only missing lightweight packages and leaves heavyweight packages such as `torch` and `tensorflow` to the Kaggle image.


In [ ]:
import importlib.util
import subprocess
import sys

required_pip_packages = {
    'dotenv': 'python-dotenv',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'pyarrow': 'pyarrow',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'huggingface_hub': 'huggingface_hub',
    'sentencepiece': 'sentencepiece',
    'bitsandbytes': 'bitsandbytes',
    'gdown': 'gdown',
    'requests': 'requests',
}

missing = [package for module_name, package in required_pip_packages.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required pip packages are already available.')

for heavy_module in ['torch', 'tensorflow']:
    if importlib.util.find_spec(heavy_module) is None:
        raise RuntimeError(f'Missing required runtime package on Kaggle image: {heavy_module}')


## Prepare Kaggle Workspace

This prepares the Kaggle workspace, stages datasets and model directories, restores prior artifacts, sets environment variables, and defines helpers to run the generated Python files.


In [ ]:
import gzip
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.parse
import zipfile
from pathlib import Path

import requests


def as_path(value):
    if value is None or value == '':
        return None
    if isinstance(value, Path):
        return value.expanduser().resolve()
    return Path(str(value)).expanduser().resolve()


def iter_dirs(root, max_depth=3):
    root = as_path(root)
    if root is None or not root.exists():
        return
    queue = [(root, 0)]
    seen = set()
    while queue:
        current, depth = queue.pop(0)
        key = str(current)
        if key in seen:
            continue
        seen.add(key)
        yield current
        if depth >= max_depth:
            continue
        try:
            children = sorted([child for child in current.iterdir() if child.is_dir()])
        except Exception:
            continue
        for child in children:
            queue.append((child, depth + 1))


def find_existing_file(candidates):
    for candidate in candidates:
        candidate = as_path(candidate)
        if candidate is not None and candidate.is_file():
            return candidate
    return None


def search_for_file(filename, search_roots, max_depth=4):
    for search_root in search_roots:
        search_root = as_path(search_root)
        if search_root is None or not search_root.exists():
            continue
        direct = search_root / filename
        if direct.is_file():
            return direct.resolve()
        for candidate in iter_dirs(search_root, max_depth=max_depth):
            path = candidate / filename
            if path.is_file():
                return path.resolve()
    return None


def search_for_dirname(dirname, search_roots, max_depth=4):
    for search_root in search_roots:
        search_root = as_path(search_root)
        if search_root is None or not search_root.exists():
            continue
        direct = search_root / dirname
        if direct.is_dir():
            return direct.resolve()
        for candidate in iter_dirs(search_root, max_depth=max_depth):
            if candidate.name == dirname and candidate.is_dir():
                return candidate.resolve()
    return None


def looks_like_model_dir(path):
    path = as_path(path)
    if path is None or not path.is_dir():
        return False
    has_config = (path / 'config.json').exists()
    has_weights = any(path.glob('*.safetensors')) or any(path.glob('pytorch_model*.bin'))
    return has_config and has_weights


def coerce_model_dir(explicit_path, expected_dir_name):
    explicit_path = as_path(explicit_path)
    if explicit_path is None:
        return None
    if looks_like_model_dir(explicit_path):
        return explicit_path
    nested = explicit_path / expected_dir_name
    if looks_like_model_dir(nested):
        return nested.resolve()
    found = search_for_dirname(expected_dir_name, [explicit_path], max_depth=3)
    if found is not None and looks_like_model_dir(found):
        return found
    return explicit_path if explicit_path.exists() else None


def looks_like_reveal_processed_dir(path):
    path = as_path(path)
    if path is None or not path.is_dir():
        return False
    required = ['train.jsonl', 'val.jsonl', 'test.jsonl']
    return all((path / name).is_file() for name in required)


def looks_like_reveal_parquet_dir(path):
    path = as_path(path)
    if path is None or not path.is_dir():
        return False
    required = [
        'train-00000-of-00001.parquet',
        'validation-00000-of-00001.parquet',
        'test-00000-of-00001.parquet',
    ]
    return all((path / name).is_file() for name in required)


def coerce_reveal_dir(explicit_path):
    explicit_path = as_path(explicit_path)
    if explicit_path is None:
        return None
    if looks_like_reveal_processed_dir(explicit_path) or looks_like_reveal_parquet_dir(explicit_path):
        return explicit_path
    for child_name in ['reveal', 'reveal_raw', 'reveal_ready', 'ReVeal', 'Reveal']:
        child = explicit_path / child_name
        if looks_like_reveal_processed_dir(child) or looks_like_reveal_parquet_dir(child) or child.is_dir():
            return child.resolve()
    return explicit_path if explicit_path.exists() else None


def normalize_previous_output_root(path):
    path = as_path(path)
    if path is None or not path.exists():
        return None
    if (path / 'GRACE-improve' / 'baseline' / 'baseline2' / 'artifacts').exists():
        return path.resolve()
    if path.name == 'GRACE-improve' and (path / 'baseline' / 'baseline2' / 'artifacts').exists():
        return path.parent.resolve()
    return None


def score_previous_output_root(root):
    code_root = root / 'GRACE-improve'
    baseline2_artifacts = code_root / 'baseline' / 'baseline2' / 'artifacts'
    shared_artifacts = code_root / 'baseline' / 'artifacts'
    dataset_predictions = baseline2_artifacts / 'predictions' / DATASET_NAME / 'grace_hybrid_predictions.jsonl'
    dataset_run_state = baseline2_artifacts / 'predictions' / DATASET_NAME / 'grace_hybrid_run_state.json'
    score = 0
    if baseline2_artifacts.exists():
        score += 10
    if shared_artifacts.exists():
        score += 5
    if dataset_predictions.exists():
        score += 20
    if dataset_run_state.exists():
        score += 20
    return score


def resolve_previous_output_root(explicit_path=None):
    explicit_root = normalize_previous_output_root(explicit_path)
    if explicit_root is not None:
        return explicit_root
    candidates = []
    if KAGGLE_INPUT_ROOT.exists():
        direct_root = normalize_previous_output_root(KAGGLE_INPUT_ROOT)
        if direct_root is not None:
            candidates.append(direct_root)
        for candidate in iter_dirs(KAGGLE_INPUT_ROOT, max_depth=5):
            normalized = normalize_previous_output_root(candidate)
            if normalized is not None:
                candidates.append(normalized)
    unique_candidates = []
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)
    if not unique_candidates:
        return None
    unique_candidates.sort(key=lambda path: (score_previous_output_root(path), len(str(path))), reverse=True)
    return unique_candidates[0]


def restore_previous_artifacts(previous_output_root):
    previous_output_root = normalize_previous_output_root(previous_output_root)
    if previous_output_root is None:
        print('No previous Kaggle output artifacts detected; starting from a clean workspace state.')
        return None
    previous_code_root = previous_output_root / 'GRACE-improve'
    restore_pairs = [
        (
            previous_code_root / 'baseline' / 'artifacts',
            WORKING_CODE_ROOT / 'baseline' / 'artifacts',
            'shared baseline artifacts',
        ),
        (
            previous_code_root / 'baseline' / 'baseline2' / 'artifacts',
            WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts',
            'baseline2 artifacts',
        ),
    ]
    restored = []
    for src, dst, label in restore_pairs:
        if src.exists():
            print(f'Restoring {label}: {src} -> {dst}')
            shutil.copytree(src, dst, dirs_exist_ok=True)
            restored.append({'label': label, 'source': str(src), 'destination': str(dst)})
    if not restored:
        print(f'Previous output root found but no restorable artifacts were detected: {previous_output_root}')
        return None
    print(json.dumps({'restored_from': str(previous_output_root), 'restored': restored}, indent=2))
    return previous_output_root


def infer_filename_from_url(url, fallback_name):
    parsed = urllib.parse.urlparse(url)
    name = Path(urllib.parse.unquote(parsed.path)).name
    return name or fallback_name


def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    headers = {}
    if HF_TOKEN and 'huggingface.co' in url:
        headers['Authorization'] = f'Bearer {HF_TOKEN.strip()}'
    with requests.get(url, stream=True, timeout=120, allow_redirects=True, headers=headers) as response:
        response.raise_for_status()
        with destination.open('wb') as handle:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return destination


def extract_downloaded_asset(archive_path, expected_filename, archive_member=''):
    archive_path = Path(archive_path)
    extract_dir = archive_path.parent / f'{archive_path.name}.extracted'
    extract_dir.mkdir(parents=True, exist_ok=True)
    archive_member = (archive_member or '').strip()

    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path, 'r') as zf:
            if archive_member:
                zf.extract(archive_member, extract_dir)
                candidate = extract_dir / archive_member
                if candidate.exists():
                    return candidate.resolve()
            else:
                zf.extractall(extract_dir)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, 'r:*') as tf:
            if archive_member:
                tf.extract(archive_member, extract_dir)
                candidate = extract_dir / archive_member
                if candidate.exists():
                    return candidate.resolve()
            else:
                tf.extractall(extract_dir)
    elif archive_path.suffix.lower() == '.gz' and not archive_path.name.endswith('.tar.gz'):
        target = extract_dir / (archive_member or expected_filename)
        with gzip.open(archive_path, 'rb') as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        if target.exists():
            return target.resolve()
    else:
        return None

    found = search_for_file(expected_filename, [extract_dir], max_depth=8)
    return found.resolve() if found is not None else None


def ensure_dataset_file(dataset_name, source_path, download_url, expected_filename, archive_member=''):
    source_path = as_path(source_path)
    if source_path is not None and source_path.exists():
        return source_path, None
    if not AUTO_DOWNLOAD_DATASET_IF_MISSING:
        return None, None
    if not download_url:
        raise FileNotFoundError(
            f'{dataset_name} source file is missing. Set the *_SOURCE_PATH or provide {dataset_name.upper()}_DOWNLOAD_URL.'
        )
    dataset_download_dir = WORKING_ROOT / '_downloads' / dataset_name
    dataset_download_dir.mkdir(parents=True, exist_ok=True)
    raw_name = infer_filename_from_url(download_url, expected_filename)
    raw_path = dataset_download_dir / raw_name
    if not raw_path.exists() or raw_path.stat().st_size == 0:
        print(f'Downloading {dataset_name} dataset from {download_url}')
        download_file(download_url, raw_path)
    else:
        print(f'Using cached downloaded asset for {dataset_name}: {raw_path}')

    if raw_path.name == expected_filename:
        return raw_path.resolve(), {'mode': 'download', 'source': download_url, 'path': str(raw_path.resolve())}

    extracted = extract_downloaded_asset(raw_path, expected_filename, archive_member=archive_member)
    if extracted is not None and extracted.exists():
        return extracted.resolve(), {'mode': 'download+extract', 'source': download_url, 'path': str(extracted.resolve())}

    located = search_for_file(expected_filename, [dataset_download_dir], max_depth=8)
    if located is not None:
        return located.resolve(), {'mode': 'download+locate', 'source': download_url, 'path': str(located.resolve())}

    raise FileNotFoundError(
        f'Could not locate {expected_filename} after downloading {dataset_name} from {download_url}. '
        'If the file is inside an archive, set the corresponding *_ARCHIVE_MEMBER in the config cell.'
    )


def remove_existing_target(target):
    target = Path(target)
    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.is_dir():
        shutil.rmtree(target)


def link_or_copy(source, target, prefer_symlink=True):
    source = as_path(source)
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        remove_existing_target(target)
    if prefer_symlink:
        try:
            os.symlink(str(source), str(target), target_is_directory=source.is_dir())
            return 'symlink'
        except Exception:
            pass
    if source.is_dir():
        shutil.copytree(source, target, dirs_exist_ok=True)
        return 'copytree'
    shutil.copy2(source, target)
    return 'copy'


def set_or_clear_env(name, value):
    if value is None or value == '':
        os.environ.pop(name, None)
    else:
        os.environ[name] = str(value)


KAGGLE_INPUT_ROOT = as_path(KAGGLE_INPUT_ROOT) or Path('/kaggle/input')
WORKING_ROOT = as_path(WORKING_ROOT) or Path('/kaggle/working/vulguardvn-final')
WORKING_CODE_ROOT = WORKING_ROOT / 'GRACE-improve'
BASELINE_DIR = WORKING_CODE_ROOT / 'baseline'
BASELINE2_DIR = BASELINE_DIR / 'baseline2'
WORKING_DATA_DIR = WORKING_CODE_ROOT / 'data'
WORKING_SHARED_ARTIFACTS_DIR = BASELINE_DIR / 'artifacts'
WORKING_SHARED_MODELS_DIR = WORKING_SHARED_ARTIFACTS_DIR / 'models'
WORKING_RETRIEVAL_TARGET_DIR = WORKING_SHARED_MODELS_DIR / 'retrieval' / RETRIEVAL_MODEL_ID.replace('/', '--')
WORKING_LOCAL_LLM_TARGET_DIR = WORKING_SHARED_MODELS_DIR / 'local_llm' / LOCAL_LLM_MODEL_ID.replace('/', '--')

if RESET_WORKING_ROOT and WORKING_ROOT.exists():
    print(f'Removing existing working root: {WORKING_ROOT}')
    shutil.rmtree(WORKING_ROOT)

WORKING_ROOT.mkdir(parents=True, exist_ok=True)
BASELINE2_DIR.mkdir(parents=True, exist_ok=True)
WORKING_DATA_DIR.mkdir(parents=True, exist_ok=True)
WORKING_SHARED_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
(WORKING_SHARED_MODELS_DIR / 'retrieval').mkdir(parents=True, exist_ok=True)
(WORKING_SHARED_MODELS_DIR / 'local_llm').mkdir(parents=True, exist_ok=True)
(BASELINE2_DIR / 'artifacts').mkdir(parents=True, exist_ok=True)

RETRIEVAL_MODEL_DIRNAME = RETRIEVAL_MODEL_ID.replace('/', '--')
LOCAL_LLM_DIRNAME = LOCAL_LLM_MODEL_ID.replace('/', '--')

DEVIGN_SOURCE_PATH = find_existing_file([DEVIGN_SOURCE_PATH]) or search_for_file('function.json', [KAGGLE_INPUT_ROOT])
BIGVUL_SOURCE_PATH = find_existing_file([BIGVUL_SOURCE_PATH]) or search_for_file('MSR_data_cleaned.csv', [KAGGLE_INPUT_ROOT])

REVEAL_SOURCE_DIR = coerce_reveal_dir(REVEAL_SOURCE_DIR)
if REVEAL_SOURCE_DIR is None:
    for candidate in iter_dirs(KAGGLE_INPUT_ROOT, max_depth=5):
        if looks_like_reveal_processed_dir(candidate) or looks_like_reveal_parquet_dir(candidate):
            REVEAL_SOURCE_DIR = candidate.resolve()
            break

RETRIEVAL_MODEL_SOURCE_DIR = coerce_model_dir(RETRIEVAL_MODEL_SOURCE_DIR, RETRIEVAL_MODEL_DIRNAME)
if RETRIEVAL_MODEL_SOURCE_DIR is None:
    found = search_for_dirname(RETRIEVAL_MODEL_DIRNAME, [KAGGLE_INPUT_ROOT], max_depth=5)
    if found is not None and looks_like_model_dir(found):
        RETRIEVAL_MODEL_SOURCE_DIR = found.resolve()

LOCAL_LLM_SOURCE_DIR = coerce_model_dir(LOCAL_LLM_SOURCE_DIR, LOCAL_LLM_DIRNAME)
if LOCAL_LLM_SOURCE_DIR is None:
    found = search_for_dirname(LOCAL_LLM_DIRNAME, [KAGGLE_INPUT_ROOT], max_depth=5)
    if found is not None and looks_like_model_dir(found):
        LOCAL_LLM_SOURCE_DIR = found.resolve()

discovery = {
    'kaggle_input_root': str(KAGGLE_INPUT_ROOT),
    'working_root': str(WORKING_ROOT),
    'dataset_name': DATASET_NAME,
    'devign_source_path': str(DEVIGN_SOURCE_PATH) if DEVIGN_SOURCE_PATH else None,
    'bigvul_source_path': str(BIGVUL_SOURCE_PATH) if BIGVUL_SOURCE_PATH else None,
    'reveal_source_dir': str(REVEAL_SOURCE_DIR) if REVEAL_SOURCE_DIR else None,
    'retrieval_model_source_dir': str(RETRIEVAL_MODEL_SOURCE_DIR) if RETRIEVAL_MODEL_SOURCE_DIR else None,
    'local_llm_source_dir': str(LOCAL_LLM_SOURCE_DIR) if LOCAL_LLM_SOURCE_DIR else None,
}
print(json.dumps(discovery, indent=2))

PREVIOUS_OUTPUT_ROOT = None
if RESTORE_PREVIOUS_KAGGLE_OUTPUT:
    PREVIOUS_OUTPUT_ROOT = restore_previous_artifacts(resolve_previous_output_root(PREVIOUS_OUTPUT_SOURCE_DIR))
else:
    print('RESTORE_PREVIOUS_KAGGLE_OUTPUT is disabled.')

dataset_download_summary = {}
if DATASET_NAME == 'devign':
    DEVIGN_SOURCE_PATH, devign_download_info = ensure_dataset_file(
        'devign',
        DEVIGN_SOURCE_PATH,
        DEVIGN_DOWNLOAD_URL,
        'function.json',
        archive_member=DEVIGN_ARCHIVE_MEMBER,
    )
    if devign_download_info is not None:
        dataset_download_summary['devign'] = devign_download_info
elif DATASET_NAME == 'bigvul':
    BIGVUL_SOURCE_PATH, bigvul_download_info = ensure_dataset_file(
        'bigvul',
        BIGVUL_SOURCE_PATH,
        BIGVUL_DOWNLOAD_URL,
        'MSR_data_cleaned.csv',
        archive_member=BIGVUL_ARCHIVE_MEMBER,
    )
    if bigvul_download_info is not None:
        dataset_download_summary['bigvul'] = bigvul_download_info

staging_summary = {}
if DEVIGN_SOURCE_PATH is not None:
    target = WORKING_DATA_DIR / 'function.json'
    mode = link_or_copy(DEVIGN_SOURCE_PATH, target, prefer_symlink=USE_SYMLINKS_WHEN_POSSIBLE)
    staging_summary['devign'] = {'mode': mode, 'target': str(target)}
if BIGVUL_SOURCE_PATH is not None:
    target = WORKING_DATA_DIR / 'MSR_data_cleaned.csv'
    mode = link_or_copy(BIGVUL_SOURCE_PATH, target, prefer_symlink=USE_SYMLINKS_WHEN_POSSIBLE)
    staging_summary['bigvul'] = {'mode': mode, 'target': str(target)}
if REVEAL_SOURCE_DIR is not None:
    reveal_target_name = 'reveal' if looks_like_reveal_processed_dir(REVEAL_SOURCE_DIR) else 'reveal_raw'
    reveal_target = WORKING_DATA_DIR / reveal_target_name
    mode = link_or_copy(REVEAL_SOURCE_DIR, reveal_target, prefer_symlink=USE_SYMLINKS_WHEN_POSSIBLE)
    staging_summary['reveal'] = {'mode': mode, 'target': str(reveal_target)}
if RETRIEVAL_MODEL_SOURCE_DIR is not None:
    mode = link_or_copy(
        RETRIEVAL_MODEL_SOURCE_DIR,
        WORKING_RETRIEVAL_TARGET_DIR,
        prefer_symlink=(USE_SYMLINKS_WHEN_POSSIBLE and not COPY_MODELS_INSTEAD_OF_LINK),
    )
    staging_summary['retrieval_model'] = {'mode': mode, 'target': str(WORKING_RETRIEVAL_TARGET_DIR)}
if LOCAL_LLM_SOURCE_DIR is not None:
    mode = link_or_copy(
        LOCAL_LLM_SOURCE_DIR,
        WORKING_LOCAL_LLM_TARGET_DIR,
        prefer_symlink=(USE_SYMLINKS_WHEN_POSSIBLE and not COPY_MODELS_INSTEAD_OF_LINK),
    )
    staging_summary['local_llm'] = {'mode': mode, 'target': str(WORKING_LOCAL_LLM_TARGET_DIR)}

if DATASET_NAME == 'devign' and not (WORKING_DATA_DIR / 'function.json').exists():
    raise FileNotFoundError('Devign file not found. Set DEVIGN_SOURCE_PATH or enable AUTO_DOWNLOAD_DATASET_IF_MISSING with DEVIGN_DOWNLOAD_URL.')
if DATASET_NAME == 'bigvul' and not (WORKING_DATA_DIR / 'MSR_data_cleaned.csv').exists():
    raise FileNotFoundError('BigVul file not found. Set BIGVUL_SOURCE_PATH or enable AUTO_DOWNLOAD_DATASET_IF_MISSING with BIGVUL_DOWNLOAD_URL.')
if DATASET_NAME == 'reveal':
    has_reveal_processed = (WORKING_DATA_DIR / 'reveal').exists()
    has_reveal_raw = (WORKING_DATA_DIR / 'reveal_raw').exists()
    if not has_reveal_processed and not has_reveal_raw:
        raise FileNotFoundError('ReVeal data was not staged. Set REVEAL_SOURCE_DIR or attach the dataset in /kaggle/input.')

HF_HOME = WORKING_ROOT / '.cache' / 'huggingface'
TORCH_HOME = WORKING_ROOT / '.cache' / 'torch'
HF_HOME.mkdir(parents=True, exist_ok=True)
TORCH_HOME.mkdir(parents=True, exist_ok=True)

set_or_clear_env('PYTHONUNBUFFERED', '1')
set_or_clear_env('HF_HOME', HF_HOME)
set_or_clear_env('TORCH_HOME', TORCH_HOME)
set_or_clear_env('TRANSFORMERS_CACHE', HF_HOME / 'transformers')
set_or_clear_env('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
set_or_clear_env('HUGGINGFACE_HUB_TOKEN', HF_TOKEN.strip() if HF_TOKEN else None)
set_or_clear_env('HF_TOKEN', HF_TOKEN.strip() if HF_TOKEN else None)
set_or_clear_env('GRACE_DATASET', DATASET_NAME)
set_or_clear_env('GRACE_PREFILTER_MODEL_NAME', PREFILTER_MODEL_NAME)
set_or_clear_env('GRACE_RETRIEVAL_MODEL_ID', RETRIEVAL_MODEL_ID)
set_or_clear_env('GRACE_LOCAL_MODEL_ID', LOCAL_LLM_MODEL_ID)
set_or_clear_env('GRACE_GRAPH_BACKEND', GRAPH_BACKEND)
set_or_clear_env('GRACE_AUTO_DOWNLOAD_MISSING', int(bool(AUTO_DOWNLOAD_MISSING_MODELS)))
set_or_clear_env('GRACE_AUTO_DOWNLOAD_RETRIEVAL_MODEL', int(bool(AUTO_DOWNLOAD_MISSING_MODELS)))
set_or_clear_env('GRACE_AUTO_DOWNLOAD_MODEL', int(bool(AUTO_DOWNLOAD_MISSING_MODELS)))
set_or_clear_env('GRACE_LOAD_IN_4BIT', int(bool(LOAD_IN_4BIT)))
set_or_clear_env('GRACE_CALL_LLM_FOR_INSPECT', int(bool(CALL_LLM_FOR_INSPECT)))
set_or_clear_env('GRACE_CALL_LLM_FOR_HIGH', int(bool(CALL_LLM_FOR_HIGH)))
set_or_clear_env('GRACE_RESUME', int(bool(RESUME_RUN)))
set_or_clear_env('GRACE_MAX_TEST_SAMPLES', MAX_TEST_SAMPLES)
set_or_clear_env('GRACE_TEST_CHUNK_SIZE', TEST_CHUNK_SIZE)
set_or_clear_env('GRACE_INSPECT_DEMOS', INSPECT_DEMOS)
set_or_clear_env('GRACE_HIGH_RISK_DEMOS', HIGH_RISK_DEMOS)
set_or_clear_env('GRACE_MAX_NEW_TOKENS', MAX_NEW_TOKENS)
set_or_clear_env('GRACE_DEMO_CHAR_LIMIT', DEMO_CHAR_LIMIT)
set_or_clear_env('GRACE_PROMPT_CODE_CHAR_LIMIT', PROMPT_CODE_CHAR_LIMIT)
set_or_clear_env('GRACE_PROMPT_TOP_LINES_LIMIT', PROMPT_TOP_LINES_LIMIT)
set_or_clear_env('GRACE_PROMPT_TOP_LINE_CHAR_LIMIT', PROMPT_TOP_LINE_CHAR_LIMIT)
set_or_clear_env('GRACE_PROMPT_SLICES_CHAR_LIMIT', PROMPT_SLICES_CHAR_LIMIT)
set_or_clear_env('GRACE_PROMPT_NODE_INFO_CHAR_LIMIT', PROMPT_NODE_INFO_CHAR_LIMIT)
set_or_clear_env('GRACE_PROMPT_EDGE_INFO_CHAR_LIMIT', PROMPT_EDGE_INFO_CHAR_LIMIT)
set_or_clear_env('GRACE_FEATURE_BATCH_SIZE', FEATURE_BATCH_SIZE)
set_or_clear_env('GRACE_FEATURE_PROGRESS_EVERY', FEATURE_PROGRESS_EVERY)
set_or_clear_env('GRACE_BUILD_PROGRESS_EVERY', BUILD_PROGRESS_EVERY)
set_or_clear_env('GRACE_PREFILTER_BATCH_SIZE', PREFILTER_BATCH_SIZE)
set_or_clear_env('GRACE_PREFILTER_EPOCHS', PREFILTER_EPOCHS)
set_or_clear_env('GRACE_PREFILTER_LEARNING_RATE', PREFILTER_LEARNING_RATE)
set_or_clear_env('GRACE_TARGET_RECALL', TARGET_RECALL)
set_or_clear_env('GRACE_DIRECT_ACCEPT_MIN_PROBABILITY', DIRECT_ACCEPT_MIN_PROBABILITY)
set_or_clear_env('GRACE_HIGH_RISK_THRESHOLD_STRATEGY', HIGH_RISK_THRESHOLD_STRATEGY)
set_or_clear_env('GRACE_HIGH_RISK_TARGET_PRECISION', HIGH_RISK_TARGET_PRECISION)
set_or_clear_env('GRACE_FEATURE_LIMIT', GRACE_FEATURE_LIMIT)
set_or_clear_env('GRACE_FEATURE_LIMIT_TRAIN', GRACE_FEATURE_LIMIT_TRAIN)
set_or_clear_env('GRACE_FEATURE_LIMIT_VAL', GRACE_FEATURE_LIMIT_VAL)
set_or_clear_env('GRACE_FEATURE_LIMIT_TEST', GRACE_FEATURE_LIMIT_TEST)
set_or_clear_env('GRACE_EXPERIMENT_VARIANT', globals().get('GRACE_EXPERIMENT_VARIANT'))
set_or_clear_env('GRACE_VARIANT_OUTPUT_SUFFIX', globals().get('GRACE_VARIANT_OUTPUT_SUFFIX'))
set_or_clear_env('GRACE_PREDICTION_FILE_STEM', globals().get('GRACE_PREDICTION_FILE_STEM'))
set_or_clear_env('GRACE_RUN_STATE_FILE_STEM', globals().get('GRACE_RUN_STATE_FILE_STEM'))
set_or_clear_env('GRACE_EVALUATION_FILE_STEM', globals().get('GRACE_EVALUATION_FILE_STEM'))
set_or_clear_env('GRACE_CALIBRATION_METHOD', globals().get('GRACE_CALIBRATION_METHOD'))
set_or_clear_env('GRACE_ROUTING_MODE', globals().get('GRACE_ROUTING_MODE'))
set_or_clear_env('GRACE_ROUTING_OBJECTIVE', globals().get('GRACE_ROUTING_OBJECTIVE'))
set_or_clear_env('GRACE_ROUTING_INSPECT_PROXY', globals().get('GRACE_ROUTING_INSPECT_PROXY'))
set_or_clear_env('GRACE_ROUTING_RECALL_FLOOR', globals().get('GRACE_ROUTING_RECALL_FLOOR'))
set_or_clear_env('GRACE_LLM_BUDGET', globals().get('GRACE_LLM_BUDGET'))
set_or_clear_env('GRACE_TAU_NEG_MIN', globals().get('GRACE_TAU_NEG_MIN'))
set_or_clear_env('GRACE_TAU_NEG_MAX', globals().get('GRACE_TAU_NEG_MAX'))
set_or_clear_env('GRACE_TAU_NEG_STEPS', globals().get('GRACE_TAU_NEG_STEPS'))
set_or_clear_env('GRACE_TAU_POS_MIN', globals().get('GRACE_TAU_POS_MIN'))
set_or_clear_env('GRACE_TAU_POS_MAX', globals().get('GRACE_TAU_POS_MAX'))
set_or_clear_env('GRACE_TAU_POS_STEPS', globals().get('GRACE_TAU_POS_STEPS'))
set_or_clear_env('GRACE_FORCE_REBUILD_FEATURES', globals().get('GRACE_FORCE_REBUILD_FEATURES'))
set_or_clear_env('GRACE_FEATURE_STORE_SUFFIX', globals().get('GRACE_FEATURE_STORE_SUFFIX'))
set_or_clear_env('GRACE_HARD_NEGATIVE_MINING', globals().get('GRACE_HARD_NEGATIVE_MINING'))
set_or_clear_env('GRACE_HARD_NEGATIVE_QUANTILE', globals().get('GRACE_HARD_NEGATIVE_QUANTILE'))
set_or_clear_env('GRACE_HARD_NEGATIVE_WEIGHT', globals().get('GRACE_HARD_NEGATIVE_WEIGHT'))
set_or_clear_env('GRACE_HARD_NEGATIVE_EPOCHS', globals().get('GRACE_HARD_NEGATIVE_EPOCHS'))
set_or_clear_env('GRACE_PREFILTER_LOSS', globals().get('GRACE_PREFILTER_LOSS'))
set_or_clear_env('GRACE_PREFILTER_FOCAL_GAMMA', globals().get('GRACE_PREFILTER_FOCAL_GAMMA'))
set_or_clear_env('GRACE_TOKEN_MAX_TOKENS', globals().get('GRACE_TOKEN_MAX_TOKENS'))
set_or_clear_env('GRACE_TOKEN_SEQUENCE_LENGTH', globals().get('GRACE_TOKEN_SEQUENCE_LENGTH'))
set_or_clear_env('GRACE_TOKEN_EMBEDDING_DIM', globals().get('GRACE_TOKEN_EMBEDDING_DIM'))
set_or_clear_env('GRACE_TOKEN_FILTERS', globals().get('GRACE_TOKEN_FILTERS'))
set_or_clear_env('GRACE_AST_MAX_TOKENS', globals().get('GRACE_AST_MAX_TOKENS'))
set_or_clear_env('GRACE_AST_SEQUENCE_LENGTH', globals().get('GRACE_AST_SEQUENCE_LENGTH'))
set_or_clear_env('GRACE_AST_EMBEDDING_DIM', globals().get('GRACE_AST_EMBEDDING_DIM'))
set_or_clear_env('GRACE_AST_FILTERS', globals().get('GRACE_AST_FILTERS'))
set_or_clear_env('GRACE_PREFILTER_PROJECTION_DIM', globals().get('GRACE_PREFILTER_PROJECTION_DIM'))
set_or_clear_env('GRACE_PREFILTER_DENSE_UNITS', globals().get('GRACE_PREFILTER_DENSE_UNITS'))
set_or_clear_env('GRACE_PREFILTER_DROPOUT', globals().get('GRACE_PREFILTER_DROPOUT'))
set_or_clear_env('GRACE_PREFILTER_EPOCHS', globals().get('GRACE_PREFILTER_EPOCHS'))
set_or_clear_env('GRACE_PREFILTER_LEARNING_RATE', globals().get('GRACE_PREFILTER_LEARNING_RATE'))
set_or_clear_env('GRACE_EVIDENCE_AWARE_VERIFIER', globals().get('GRACE_EVIDENCE_AWARE_VERIFIER'))

os.chdir(WORKING_ROOT)
if str(BASELINE2_DIR) not in sys.path:
    sys.path.insert(0, str(BASELINE2_DIR))

def run_python_file(path, extra_env=None):
    path = Path(path)
    env = os.environ.copy()
    if extra_env:
        for key, value in extra_env.items():
            if value is None or value == '':
                env.pop(key, None)
            else:
                env[key] = str(value)
    command = [sys.executable, str(path)]
    print('Running:', ' '.join(command))
    process = subprocess.Popen(
        command,
        cwd=str(WORKING_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{path.name} failed with exit code {return_code}')

def run_baseline2(script_name, extra_env=None):
    return run_python_file(BASELINE2_DIR / script_name, extra_env=extra_env)

summary = {
    'working_root': str(WORKING_ROOT),
    'baseline2_dir': str(BASELINE2_DIR),
    'working_data_dir': str(WORKING_DATA_DIR),
    'working_shared_models_dir': str(WORKING_SHARED_MODELS_DIR),
    'dataset_download_summary': dataset_download_summary,
    'staging_summary': staging_summary,
    'previous_output_root': str(PREVIOUS_OUTPUT_ROOT) if PREVIOUS_OUTPUT_ROOT else None,
}
print(json.dumps(summary, indent=2))


## Materialize Baseline2 Source Files

The next cells write the exact `baseline2` source files into `/kaggle/working` before the pipeline runs.


### File `common.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/common.py
import csv
import hashlib
import json
import math
import os
import re
from pathlib import Path
from typing import Iterable

from dotenv import load_dotenv


BASELINE_DIR = Path(__file__).resolve().parent
BASELINE_ROOT_DIR = BASELINE_DIR.parent
ROOT_DIR = BASELINE_ROOT_DIR.parent
DATA_DIR = ROOT_DIR / "data"

# Shared baseline assets already prepared by previous runs live here.
SHARED_ARTIFACTS_DIR = BASELINE_ROOT_DIR / "artifacts"
SHARED_PROCESSED_DIR = SHARED_ARTIFACTS_DIR / "processed"
SHARED_SPLITS_DIR = SHARED_ARTIFACTS_DIR / "splits"
SHARED_MODELS_DIR = SHARED_ARTIFACTS_DIR / "models"
SHARED_GRAPH_DIR = SHARED_ARTIFACTS_DIR / "graphs"
SHARED_CACHE_DIR = SHARED_ARTIFACTS_DIR / "cache"
SHARED_METRICS_DIR = SHARED_ARTIFACTS_DIR / "metrics"
SHARED_PREDICTIONS_DIR = SHARED_ARTIFACTS_DIR / "predictions"
SHARED_RETRIEVAL_DIR = SHARED_ARTIFACTS_DIR / "retrieval"

# Baseline2 writes its own derived artifacts here to avoid clobbering baseline1.
ARTIFACTS_DIR = BASELINE_DIR / "artifacts"
FEATURES_DIR = ARTIFACTS_DIR / "features"
MODELS_DIR = ARTIFACTS_DIR / "models"
RETRIEVAL_DIR = ARTIFACTS_DIR / "retrieval"
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
CACHE_DIR = ARTIFACTS_DIR / "cache"

# Keep aliases for code that expects dataset and graph assets.
PROCESSED_DIR = SHARED_PROCESSED_DIR
SPLITS_DIR = SHARED_SPLITS_DIR
GRAPH_DIR = SHARED_GRAPH_DIR
ENV_PATH = ROOT_DIR / ".env"

load_dotenv(ENV_PATH)
csv.field_size_limit(10**9)

C_KEYWORDS = {
    "auto",
    "break",
    "case",
    "char",
    "const",
    "continue",
    "default",
    "do",
    "double",
    "else",
    "enum",
    "extern",
    "float",
    "for",
    "goto",
    "if",
    "inline",
    "int",
    "long",
    "register",
    "restrict",
    "return",
    "short",
    "signed",
    "sizeof",
    "static",
    "struct",
    "switch",
    "typedef",
    "union",
    "unsigned",
    "void",
    "volatile",
    "while",
    "class",
    "namespace",
    "new",
    "delete",
    "public",
    "private",
    "protected",
    "template",
    "typename",
    "try",
    "catch",
    "throw",
    "using",
    "virtual",
    "bool",
    "true",
    "false",
    "nullptr",
    "null",
}

CONTROL_KEYWORDS = ["if", "else", "switch", "case", "for", "while", "do", "goto", "return", "break", "continue"]
RISKY_APIS = {
    "strcpy",
    "strncpy",
    "strcat",
    "strncat",
    "sprintf",
    "snprintf",
    "vsprintf",
    "scanf",
    "sscanf",
    "fscanf",
    "gets",
    "memcpy",
    "memmove",
    "memset",
    "malloc",
    "calloc",
    "realloc",
    "free",
    "new",
    "delete",
    "read",
    "write",
    "recv",
    "send",
    "open",
    "close",
    "fopen",
    "fclose",
    "strtok",
    "system",
    "exec",
    "popen",
}

STRING_PATTERN = re.compile(r'"(?:\\.|[^"\\])*"', re.DOTALL)
CHAR_PATTERN = re.compile(r"'(?:\\.|[^'\\])*'", re.DOTALL)
NUMBER_PATTERN = re.compile(r"\b(?:0x[0-9a-fA-F]+|\d+\.\d+|\d+)\b")
TOKEN_PATTERN = re.compile(r"[A-Za-z_]\w*|==|!=|<=|>=|->|\+\+|--|&&|\|\||[{}\[\]();,.*&|^~!<>%/\-+=?:]")
SIGNATURE_PATTERN = re.compile(r"([A-Za-z_]\w*)\s*\((.*?)\)\s*\{", re.DOTALL)
CAMEL_PATTERN = re.compile(r"([a-z0-9])([A-Z])")


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()


def normalize_code(code: str) -> str:
    text = (code or "").replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ")
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _sanitize_literals(code: str) -> str:
    text = STRING_PATTERN.sub(" STR_LIT ", code)
    text = CHAR_PATTERN.sub(" CHAR_LIT ", text)
    return NUMBER_PATTERN.sub(" NUM_LIT ", text)


def _split_identifier(token: str) -> list[str]:
    token = CAMEL_PATTERN.sub(r"\1_\2", token).replace("__", "_").strip("_")
    parts = [part.lower() for part in token.split("_") if part]
    return parts or [token.lower()]


def tokenize_code(code: str) -> list[str]:
    text = _sanitize_literals(normalize_code(code))
    tokens = TOKEN_PATTERN.findall(text)
    results: list[str] = []
    for token in tokens:
        if token in {"STR_LIT", "CHAR_LIT"}:
            results.append("str_lit")
        elif token == "NUM_LIT":
            results.append("num_lit")
        elif re.match(r"[A-Za-z_]\w*$", token):
            lowered = token.lower()
            if lowered in C_KEYWORDS:
                results.append(lowered)
            else:
                results.extend(_split_identifier(token))
        else:
            results.append(token)
    return results


def build_skeleton(code: str) -> str:
    text = _sanitize_literals(normalize_code(code))
    tokens = TOKEN_PATTERN.findall(text)
    skeleton: list[str] = []
    for index, token in enumerate(tokens):
        lowered = token.lower()
        next_token = tokens[index + 1] if index + 1 < len(tokens) else ""
        if token in {"STR_LIT", "CHAR_LIT"}:
            skeleton.append("lit")
        elif token == "NUM_LIT":
            skeleton.append("num")
        elif re.match(r"[A-Za-z_]\w*$", token):
            if lowered in C_KEYWORDS:
                skeleton.append(lowered)
            elif next_token == "(":
                skeleton.append("call")
            else:
                skeleton.append("id")
        else:
            skeleton.append(token)
    return " ".join(skeleton)


def extract_function_name(code: str) -> str:
    match = SIGNATURE_PATTERN.search(normalize_code(code)[:1200])
    if not match:
        return "unknown"
    name = match.group(1)
    return name if name.lower() not in CONTROL_KEYWORDS else "unknown"


def estimate_parameter_count(code: str) -> int:
    match = SIGNATURE_PATTERN.search(normalize_code(code)[:1200])
    if not match:
        return 0
    params = match.group(2).strip()
    if not params or params == "void":
        return 0
    return len([part for part in params.split(",") if part.strip()])


def extract_calls(code: str) -> list[str]:
    calls = re.findall(r"\b([A-Za-z_]\w*)\s*\(", normalize_code(code))
    seen = set()
    results = []
    for call in calls:
        lowered = call.lower()
        if lowered in C_KEYWORDS or lowered in seen:
            continue
        seen.add(lowered)
        results.append(call)
    return results


def build_structure_summary(code: str) -> str:
    text = normalize_code(code)
    tokens = tokenize_code(text)
    token_set = set(tokens)
    controls = {name: tokens.count(name) for name in CONTROL_KEYWORDS if tokens.count(name)}
    risky = [call for call in extract_calls(text) if call.lower() in RISKY_APIS]
    memory_ops = [name for name in ["malloc", "calloc", "realloc", "free", "new", "delete", "memcpy", "memmove"] if name in token_set]
    lines = [
        f"function={extract_function_name(text)}",
        f"params={estimate_parameter_count(text)}",
        f"lines={len([line for line in text.splitlines() if line.strip()])}",
        f"calls={', '.join(extract_calls(text)[:8]) or 'none'}",
        f"control={json.dumps(controls, ensure_ascii=True) if controls else '{}'}",
        f"risky_apis={', '.join(risky) or 'none'}",
        f"memory_ops={', '.join(memory_ops) or 'none'}",
        f"pointer_ops={text.count('->') + text.count('*') + text.count('&')}",
        f"array_accesses={text.count('[')}",
    ]
    return "\n".join(lines)


def truncate_text(text: str, limit: int) -> str:
    value = normalize_code(text)
    if len(value) <= limit:
        return value
    return value[: limit - 3].rstrip() + "..."


def get_record_code(record: dict) -> str:
    for key in ["code", "func", "functionSource", "source", "raw_code", "func_before", "before"]:
        value = record.get(key)
        if value is None:
            continue
        text = normalize_code(str(value))
        if text:
            return text
    return ""


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def iter_jsonl(path: Path) -> Iterable[dict]:
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                yield json.loads(line)


def write_jsonl(path: Path, rows: Iterable[dict]) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def dump_json(path: Path, payload: dict) -> None:
    ensure_dir(path.parent)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def load_json(path: Path, default: dict | None = None) -> dict:
    if not path.exists():
        return {} if default is None else default
    return json.loads(path.read_text(encoding="utf-8"))


def resolve_gemini_api_key() -> str:
    key = os.getenv("API_GEMINI") or os.getenv("GOOGLE_API_KEY") or os.getenv("API_KEY")
    if not key:
        raise RuntimeError(f"Missing Gemini API key. Set API_GEMINI in {ENV_PATH}.")
    return key.strip().strip('"').strip("'")


def sigmoid(value: float) -> float:
    if value >= 0:
        z = math.exp(-value)
        return 1.0 / (1.0 + z)
    z = math.exp(value)
    return z / (1.0 + z)


### File `datasets.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/datasets.py
import csv
import json
from pathlib import Path
from typing import Iterable

import pandas as pd

from common import DATA_DIR, get_record_code, normalize_code, stable_hash


DEVIGN_SOURCE = DATA_DIR / "function.json"
BIGVUL_SOURCE = DATA_DIR / "MSR_data_cleaned.csv"
REVEAL_SPLIT_FILES = {
    "train": DATA_DIR / "reveal" / "train.jsonl",
    "val": DATA_DIR / "reveal" / "val.jsonl",
    "test": DATA_DIR / "reveal" / "test.jsonl",
}
REVEAL_CANDIDATE_DIRS = [
    DATA_DIR / "reveal",
    DATA_DIR / "reveal_ready",
    DATA_DIR / "reveal_raw",
    DATA_DIR / "ReVeal",
    DATA_DIR / "Reveal",
]

PROJECT_FIELDS = ["project", "project_before", "repo", "repository"]
CWE_FIELDS = ["cwe_id", "cwe", "CWE ID", "cweID"]
CODE_FIELDS = ["code", "func", "functionSource", "source", "raw_code", "func_before", "before"]
LABEL_FIELDS = ["target", "label", "vul", "is_vul", "is_vulnerable"]


def list_available_datasets() -> list[str]:
    datasets = []
    if DEVIGN_SOURCE.exists():
        datasets.append("devign")
    if BIGVUL_SOURCE.exists():
        datasets.append("bigvul")
    if discover_reveal_root():
        datasets.append("reveal")
    return datasets


def discover_reveal_root() -> Path | None:
    if has_reveal_official_splits():
        return REVEAL_SPLIT_FILES["train"].parent
    for candidate in REVEAL_CANDIDATE_DIRS:
        if candidate.exists():
            return candidate
    return None


def has_reveal_official_splits() -> bool:
    return all(path.exists() for path in REVEAL_SPLIT_FILES.values())


def _choose_field(row: dict, names: list[str], default: str = "") -> str:
    for name in names:
        value = row.get(name)
        if value is None:
            continue
        text = str(value).strip()
        if text:
            return text
    return default


def _parse_label(value) -> int | None:
    if value is None:
        return None
    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "vulnerable", "positive"}:
        return 1
    if text in {"0", "false", "no", "non-vulnerable", "benign", "negative"}:
        return 0
    return None


def _project_from_row(row: dict, default: str = "") -> str:
    return _choose_field(row, PROJECT_FIELDS, default=default)


def _cwe_from_row(row: dict) -> str:
    return _choose_field(row, CWE_FIELDS)


def _canonical_record(
    *,
    dataset: str,
    record_id: str,
    code: str,
    label: int,
    project: str = "",
    commit_id: str = "",
    cwe_id: str = "",
    source_path: str = "",
    split: str = "",
    extra: dict | None = None,
) -> dict:
    record = {
        "record_id": record_id,
        "dataset": dataset,
        "project": project,
        "label": int(label),
        "code": normalize_code(code),
        "commit_id": commit_id,
        "cwe_id": cwe_id,
        "source_path": source_path,
        "code_hash": stable_hash(normalize_code(code)),
    }
    if split:
        record["split"] = split
    if extra:
        record.update(extra)
    return record


def get_dataset_iterator(dataset_name: str):
    if dataset_name == "devign":
        return iter_devign_records
    if dataset_name == "bigvul":
        return iter_bigvul_records
    if dataset_name == "reveal":
        return iter_reveal_records
    raise ValueError(f"Unsupported dataset: {dataset_name}")


def iter_devign_records() -> Iterable[dict]:
    data = json.loads(DEVIGN_SOURCE.read_text(encoding="utf-8"))
    for index, row in enumerate(data):
        code = normalize_code(str(row.get("func", "") or ""))
        label = _parse_label(row.get("target"))
        if not code or label is None:
            continue
        yield _canonical_record(
            dataset="devign",
            record_id=f"devign-{index}",
            code=code,
            label=label,
            project=str(row.get("project", "") or ""),
            commit_id=str(row.get("commit_id", "") or ""),
            source_path=str(DEVIGN_SOURCE.name),
            extra={"source_row": index},
        )


def iter_bigvul_records() -> Iterable[dict]:
    with BIGVUL_SOURCE.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for index, row in enumerate(reader):
            code = normalize_code(str(row.get("func_before", "") or ""))
            label = _parse_label(row.get("vul"))
            if not code or label is None:
                continue
            yield _canonical_record(
                dataset="bigvul",
                record_id=f"bigvul-{index}",
                code=code,
                label=label,
                project=_project_from_row(row),
                commit_id=str(row.get("commit_id", "") or ""),
                cwe_id=_cwe_from_row(row),
                source_path=str(BIGVUL_SOURCE.name),
                extra={"source_row": index},
            )


def _iter_reveal_jsonl_split(split_name: str, path: Path) -> Iterable[dict]:
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            code = get_record_code(row)
            label = _parse_label(_choose_field(row, LABEL_FIELDS))
            if not code or label is None:
                continue
            project = _project_from_row(row, default=path.parent.name)
            record_id = str(row.get("record_id", "") or f"reveal-{split_name}-{index}")
            extra = {}
            for key in ["hash", "size", "source_format"]:
                if key in row and row[key] is not None:
                    extra[key] = row[key]
            yield _canonical_record(
                dataset="reveal",
                record_id=record_id,
                code=code,
                label=label,
                project=project,
                source_path=str(path.relative_to(DATA_DIR)),
                split=split_name,
                extra=extra,
            )


def iter_reveal_split_records(split_name: str) -> Iterable[dict]:
    path = REVEAL_SPLIT_FILES[split_name]
    if not path.exists():
        return
    yield from _iter_reveal_jsonl_split(split_name, path)


def _iter_json_file(path: Path, dataset_name: str) -> Iterable[dict]:
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return
    if isinstance(payload, dict):
        for key in ["data", "records", "items"]:
            if isinstance(payload.get(key), list):
                payload = payload[key]
                break
    if not isinstance(payload, list):
        return
    for index, row in enumerate(payload):
        if not isinstance(row, dict):
            continue
        code = get_record_code(row)
        label = _parse_label(_choose_field(row, LABEL_FIELDS))
        if not code or label is None:
            continue
        split_hint = str(row.get("split", "") or "")
        yield _canonical_record(
            dataset=dataset_name,
            record_id=f"{dataset_name}-{path.stem}-{index}",
            code=code,
            label=label,
            project=_project_from_row(row, default=path.parent.name),
            commit_id=str(row.get("commit_id", "") or ""),
            cwe_id=_cwe_from_row(row),
            source_path=str(path.relative_to(DATA_DIR)),
            split=split_hint,
        )


def _iter_jsonl_file(path: Path, dataset_name: str) -> Iterable[dict]:
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            if not isinstance(row, dict):
                continue
            code = get_record_code(row)
            label = _parse_label(_choose_field(row, LABEL_FIELDS))
            if not code or label is None:
                continue
            split_hint = str(row.get("split", "") or "")
            record_id = str(row.get("record_id", "") or f"{dataset_name}-{path.stem}-{index}")
            extra = {}
            for key in ["hash", "size", "source_format"]:
                if key in row and row[key] is not None:
                    extra[key] = row[key]
            yield _canonical_record(
                dataset=dataset_name,
                record_id=record_id,
                code=code,
                label=label,
                project=_project_from_row(row, default=path.parent.name),
                commit_id=str(row.get("commit_id", "") or ""),
                cwe_id=_cwe_from_row(row),
                source_path=str(path.relative_to(DATA_DIR)),
                split=split_hint,
                extra=extra,
            )


def _iter_table_file(path: Path, dataset_name: str, delimiter: str) -> Iterable[dict]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle, delimiter=delimiter)
        for index, row in enumerate(reader):
            code = get_record_code(row)
            label = _parse_label(_choose_field(row, LABEL_FIELDS))
            if not code or label is None:
                continue
            split_hint = str(row.get("split", "") or "")
            yield _canonical_record(
                dataset=dataset_name,
                record_id=f"{dataset_name}-{path.stem}-{index}",
                code=code,
                label=label,
                project=_project_from_row(row, default=path.parent.name),
                commit_id=str(row.get("commit_id", "") or ""),
                cwe_id=_cwe_from_row(row),
                source_path=str(path.relative_to(DATA_DIR)),
                split=split_hint,
            )


def _iter_parquet_file(path: Path, dataset_name: str) -> Iterable[dict]:
    frame = pd.read_parquet(path)
    rows = frame.to_dict(orient="records")
    split_hint = "train" if "train" in path.stem else "val" if "validation" in path.stem else "test" if "test" in path.stem else ""
    for index, row in enumerate(rows):
        code = get_record_code(row)
        label = _parse_label(_choose_field(row, LABEL_FIELDS))
        if not code or label is None:
            continue
        extra = {}
        for key in ["hash", "size"]:
            if key in row and row[key] is not None:
                extra[key] = row[key]
        yield _canonical_record(
            dataset=dataset_name,
            record_id=f"{dataset_name}-{path.stem}-{index}",
            code=code,
            label=label,
            project=_project_from_row(row, default=path.parent.name),
            source_path=str(path.relative_to(DATA_DIR)),
            split=split_hint,
            extra=extra,
        )


def _parse_label_from_path(path: Path) -> int | None:
    parts = [part.lower() for part in path.parts]
    for part in reversed(parts):
        if part in {"1", "vulnerable", "positive", "bad"}:
            return 1
        if part in {"0", "non-vulnerable", "benign", "negative", "good"}:
            return 0
    stem = path.stem.lower()
    if stem.endswith("_1") or stem.endswith("-1"):
        return 1
    if stem.endswith("_0") or stem.endswith("-0"):
        return 0
    return None


def _iter_source_files(root: Path, dataset_name: str) -> Iterable[dict]:
    allowed_suffixes = {".c", ".cc", ".cpp", ".cxx", ".h", ".hpp"}
    for path in root.rglob("*"):
        if path.suffix.lower() not in allowed_suffixes:
            continue
        label = _parse_label_from_path(path)
        if label is None:
            continue
        code = normalize_code(path.read_text(encoding="utf-8", errors="ignore"))
        if not code:
            continue
        yield _canonical_record(
            dataset=dataset_name,
            record_id=f"{dataset_name}-{stable_hash(str(path.relative_to(root)))}",
            code=code,
            label=label,
            project=path.parent.name,
            source_path=str(path.relative_to(DATA_DIR)),
        )


def iter_reveal_records() -> Iterable[dict]:
    if has_reveal_official_splits():
        for split_name in ["train", "val", "test"]:
            yield from iter_reveal_split_records(split_name)
        return
    root = discover_reveal_root()
    if root is None:
        return
    seen_ids = set()
    for path in root.rglob("*"):
        if path.is_dir():
            continue
        suffix = path.suffix.lower()
        if suffix == ".json":
            iterator = _iter_json_file(path, "reveal")
        elif suffix == ".jsonl":
            iterator = _iter_jsonl_file(path, "reveal")
        elif suffix == ".csv":
            iterator = _iter_table_file(path, "reveal", ",")
        elif suffix == ".tsv":
            iterator = _iter_table_file(path, "reveal", "\t")
        elif suffix == ".parquet":
            iterator = _iter_parquet_file(path, "reveal")
        else:
            iterator = []
        for row in iterator:
            if row["record_id"] in seen_ids:
                continue
            seen_ids.add(row["record_id"])
            yield row
    for row in _iter_source_files(root, "reveal"):
        if row["record_id"] in seen_ids:
            continue
        seen_ids.add(row["record_id"])
        yield row


### File `graphs.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/graphs.py
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

from common import (
    CONTROL_KEYWORDS,
    GRAPH_DIR,
    ensure_dir,
    estimate_parameter_count,
    extract_function_name,
    normalize_code,
    stable_hash,
    truncate_text,
)


GRAPH_SCHEMA_VERSION = 1
DEFAULT_MAX_NODES = 72
DEFAULT_MAX_EDGES = 96
MAX_AST_TOKENS = 512
GRAPH_FILE_NAME = "graph_features.json"
JOERN_INSTALL_ROOT = GRAPH_DIR / "tools" / "joern"
JOERN_HOME_DIR = JOERN_INSTALL_ROOT / "joern-cli"

DOT_NODE_PATTERN = re.compile(r'^\s*"?(?P<node_id>[^"\s]+)"?\s*\[\s*label\s*=\s*"(?P<label>.*)"\s*\]\s*;?\s*$')
DOT_EDGE_PATTERN = re.compile(
    r'^\s*"?(?P<src>[^"\s]+)"?\s*->\s*"?(?P<dst>[^"\s]+)"?(?:\s*\[(?P<attrs>.*?)\])?\s*;?\s*$'
)
DOT_LABEL_ATTR_PATTERN = re.compile(r'label\s*=\s*"(?P<label>[^"]+)"')
IDENTIFIER_PATTERN = re.compile(r"\b([A-Za-z_]\w*)\b")
ASSIGNMENT_PATTERN = re.compile(r"\b([A-Za-z_]\w*)\s*=")

EDGE_LABEL_ALIASES = {
    "AST": "IS_AST_PARENT",
    "CFG": "FLOWS_TO",
    "CDG": "CONTROLS",
    "DDG": "REACHES",
    "DOMINATE": "DOM",
    "POST_DOMINATE": "POST_DOM",
    "REACHING_DEF": "REACHES",
}


def default_graph_cache_dir(dataset_name: str) -> Path:
    return GRAPH_DIR / dataset_name


def graph_cache_path(dataset_name: str, code_hash: str) -> Path:
    return default_graph_cache_dir(dataset_name) / code_hash / GRAPH_FILE_NAME


def resolve_graph_backend(preferred: str = "auto") -> str:
    resolved, _ = resolve_graph_backend_with_notice(preferred)
    return resolved


def resolve_graph_backend_with_notice(preferred: str = "auto") -> tuple[str, str | None]:
    lowered = (preferred or "auto").strip().lower()
    if lowered == "stable":
        lowered = "auto"
    if lowered not in {"auto", "joern", "heuristic", "stable"}:
        raise ValueError(f"Unsupported graph backend: {preferred}")
    if lowered != "auto":
        return lowered, None
    joern_ok, joern_notice = _probe_joern_backend()
    if joern_ok:
        return "joern", None
    return "heuristic", joern_notice


def default_joern_install_dir() -> Path:
    return JOERN_HOME_DIR


def resolve_joern_command(executable_name: str) -> str:
    env_name = "GRACE_JOERN_PARSE" if executable_name == "joern-parse" else "GRACE_JOERN_EXPORT"
    configured = os.getenv(env_name)
    if configured:
        return configured
    for candidate in _candidate_joern_commands(executable_name):
        if shutil.which(candidate):
            return candidate
        if Path(candidate).exists():
            return str(Path(candidate))
    return executable_name


def get_graph_features(
    record: dict,
    *,
    dataset_name: str | None = None,
    graph_backend: str = "auto",
    force_rebuild: bool = False,
    max_nodes: int = DEFAULT_MAX_NODES,
    max_edges: int = DEFAULT_MAX_EDGES,
    cache_dir: Path | None = None,
) -> dict:
    dataset = dataset_name or record.get("dataset", "default")
    code = normalize_code(record.get("code", ""))
    if not code:
        raise ValueError("Graph extraction requires a non-empty `code` field.")
    code_hash = record.get("code_hash") or stable_hash(code)
    backend_suffix = f"__{graph_backend}" if graph_backend else ""
    target_path = cache_dir or graph_cache_path(dataset, f"{code_hash}{backend_suffix}")
    if not force_rebuild and target_path.exists():
        cached = json.loads(target_path.read_text(encoding="utf-8"))
        if cached.get("schema_version") == GRAPH_SCHEMA_VERSION:
            return cached
    started = time.perf_counter()
    artifact = build_graph_features(
        code,
        record_id=str(record.get("record_id", "")),
        code_hash=code_hash,
        graph_backend=graph_backend,
        max_nodes=max_nodes,
        max_edges=max_edges,
    )
    artifact.update(
        {
            "schema_version": GRAPH_SCHEMA_VERSION,
            "dataset": dataset,
            "record_id": str(record.get("record_id", "")),
            "code_hash": code_hash,
            "cache_path": str(target_path),
            "build_seconds": round(time.perf_counter() - started, 6),
        }
    )
    ensure_dir(target_path.parent)
    target_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
    return artifact


def build_graph_features(
    code: str,
    *,
    record_id: str = "",
    code_hash: str = "",
    graph_backend: str = "auto",
    max_nodes: int = DEFAULT_MAX_NODES,
    max_edges: int = DEFAULT_MAX_EDGES,
) -> dict:
    requested_backend = (graph_backend or "auto").strip().lower()
    resolved_backend = resolve_graph_backend(requested_backend)
    last_error = None
    if resolved_backend == "joern":
        try:
            artifact = _build_joern_graph_features(code, max_nodes=max_nodes, max_edges=max_edges)
            artifact["requested_backend"] = requested_backend
            artifact["record_id"] = record_id
            artifact["code_hash"] = code_hash
            return artifact
        except Exception as exc:
            last_error = str(exc)
            if requested_backend == "joern":
                raise
    artifact = _build_heuristic_graph_features(code, max_nodes=max_nodes, max_edges=max_edges)
    artifact["requested_backend"] = requested_backend
    artifact["record_id"] = record_id
    artifact["code_hash"] = code_hash
    if last_error:
        artifact["backend_notice"] = f"Falling back to heuristic graph extraction because Joern failed: {last_error}"
    return artifact


def _build_joern_graph_features(code: str, *, max_nodes: int, max_edges: int) -> dict:
    joern_parse = resolve_joern_command("joern-parse")
    joern_export = resolve_joern_command("joern-export")
    if not _command_exists(joern_parse):
        raise FileNotFoundError(f"Missing Joern parser executable: {joern_parse}")
    if not _command_exists(joern_export):
        raise FileNotFoundError(f"Missing Joern export executable: {joern_export}")

    temp_dir = Path(tempfile.mkdtemp(prefix="grace_joern_"))
    try:
        source_dir = ensure_dir(temp_dir / "src")
        (source_dir / "snippet.c").write_text(normalize_code(code) + "\n", encoding="utf-8")
        _run_joern_command([joern_parse, str(source_dir)], cwd=temp_dir)

        merged_nodes = {}
        merged_edges = []
        export_specs = [
            ("ast", "IS_AST_PARENT"),
            ("cfg", "FLOWS_TO"),
            ("pdg", "REACHES"),
        ]
        for representation, default_edge_type in export_specs:
            output_dir = temp_dir / representation
            try:
                _run_joern_command([joern_export, "--repr", representation, "--out", str(output_dir)], cwd=temp_dir)
            except RuntimeError:
                if representation == "pdg":
                    continue
                raise
            dot_path = _choose_dot_file(output_dir)
            if dot_path is None:
                if representation == "pdg":
                    continue
                raise FileNotFoundError(f"Joern did not export a {representation.upper()} dot file.")
            nodes, edges = _parse_dot_graph(dot_path, default_edge_type=default_edge_type)
            merged_nodes.update(nodes)
            merged_edges.extend(edges)
        if not merged_nodes:
            raise RuntimeError("Joern export returned no nodes.")
        return _finalize_graph_artifact(
            merged_nodes,
            merged_edges,
            backend="joern",
            max_nodes=max_nodes,
            max_edges=max_edges,
        )
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)


def _run_joern_command(command: list[str], *, cwd: Path) -> None:
    result = subprocess.run(
        command,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        timeout=180,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(command)}"
            f" | stdout={truncate_text(result.stdout or '', 240)}"
            f" | stderr={truncate_text(result.stderr or '', 240)}"
        )


def _choose_dot_file(output_dir: Path) -> Path | None:
    dot_files = sorted(output_dir.rglob("*.dot"), key=lambda path: path.stat().st_size if path.exists() else 0, reverse=True)
    return dot_files[0] if dot_files else None


def _parse_dot_graph(dot_path: Path, *, default_edge_type: str) -> tuple[dict[str, dict], list[dict]]:
    nodes: dict[str, dict] = {}
    edges: list[dict] = []
    for line in dot_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        node_match = DOT_NODE_PATTERN.match(line)
        if node_match:
            node_id = node_match.group("node_id")
            node_type, code = _parse_dot_node_label(node_match.group("label"))
            nodes[node_id] = {
                "id": node_id,
                "type": node_type or "UNKNOWN",
                "code": truncate_text(code or node_type or "", 120),
            }
            continue
        edge_match = DOT_EDGE_PATTERN.match(line)
        if edge_match:
            edge_type = _canonicalize_edge_label(_extract_edge_label(edge_match.group("attrs")), default_edge_type)
            edges.append(
                {
                    "source": edge_match.group("src"),
                    "target": edge_match.group("dst"),
                    "type": edge_type,
                }
            )
    return nodes, edges


def _parse_dot_node_label(label: str) -> tuple[str, str]:
    inner = (label or "").strip()
    if inner.startswith("(") and inner.endswith(")"):
        inner = inner[1:-1]
    first = inner.find(",")
    if first < 0:
        return inner or "UNKNOWN", ""
    second = inner.find(",", first + 1)
    if second < 0:
        return inner[:first].strip() or "UNKNOWN", inner[first + 1 :].strip()
    node_type = inner[:first].strip() or "UNKNOWN"
    primary_code = inner[first + 1 : second].strip()
    fallback_code = inner[second + 1 :].strip()
    return node_type, primary_code or fallback_code


def _extract_edge_label(attrs: str | None) -> str | None:
    if not attrs:
        return None
    match = DOT_LABEL_ATTR_PATTERN.search(attrs)
    return match.group("label") if match else None


def _canonicalize_edge_label(label: str | None, default: str) -> str:
    if not label:
        return default
    candidate = label.strip().replace("-", "_").replace(" ", "_").upper()
    candidate = candidate.split(":")[0]
    return EDGE_LABEL_ALIASES.get(candidate, candidate or default)


def _build_heuristic_graph_features(code: str, *, max_nodes: int, max_edges: int) -> dict:
    text = normalize_code(code)
    function_name = extract_function_name(text)
    param_count = estimate_parameter_count(text)
    nodes: dict[str, dict] = {}
    edges: list[dict] = []
    next_node_id = 1

    def add_node(node_type: str, snippet: str) -> str:
        nonlocal next_node_id
        node_id = str(next_node_id)
        next_node_id += 1
        nodes[node_id] = {
            "id": node_id,
            "type": node_type,
            "code": truncate_text(snippet, 120),
        }
        return node_id

    method_id = add_node("METHOD", f"{function_name}()")
    for param_index in range(param_count):
        param_id = add_node("PARAM", f"param_{param_index + 1}")
        edges.append({"source": method_id, "target": param_id, "type": "IS_AST_PARENT"})

    statement_ids: list[str] = []
    last_definition_for_name: dict[str, str] = {}
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in lines[:40]:
        node_type = _heuristic_node_type(line)
        statement_id = add_node(node_type, line)
        statement_ids.append(statement_id)
        edges.append({"source": method_id, "target": statement_id, "type": "IS_AST_PARENT"})
        for call_name in _extract_line_calls(line):
            call_id = add_node("CALL", f"{call_name}(...)")
            edges.append({"source": statement_id, "target": call_id, "type": "IS_AST_PARENT"})
        for identifier in _extract_line_identifiers(line):
            if identifier in last_definition_for_name:
                edges.append({"source": last_definition_for_name[identifier], "target": statement_id, "type": "REACHES"})
        for assigned_name in _extract_assignment_targets(line):
            last_definition_for_name[assigned_name] = statement_id
    for left, right in zip(statement_ids, statement_ids[1:]):
        edges.append({"source": left, "target": right, "type": "FLOWS_TO"})

    return _finalize_graph_artifact(
        nodes,
        edges,
        backend="heuristic",
        max_nodes=max_nodes,
        max_edges=max_edges,
    )


def _heuristic_node_type(line: str) -> str:
    lowered = line.lower()
    for keyword in CONTROL_KEYWORDS:
        if lowered.startswith(keyword) or f"{keyword} (" in lowered or f"{keyword}(" in lowered:
            return "CONTROL_STRUCTURE"
    if "=" in line:
        return "CALL" if "(" in line and ")" in line else "EXPRESSION"
    if "(" in line and ")" in line:
        return "CALL"
    if lowered.startswith("return"):
        return "RETURN"
    return "STATEMENT"


def _extract_line_calls(line: str) -> list[str]:
    matches = re.findall(r"\b([A-Za-z_]\w*)\s*\(", line)
    results = []
    seen = set()
    for name in matches:
        lowered = name.lower()
        if lowered in CONTROL_KEYWORDS or lowered in seen:
            continue
        seen.add(lowered)
        results.append(name)
    return results


def _extract_line_identifiers(line: str) -> list[str]:
    identifiers = []
    for match in IDENTIFIER_PATTERN.findall(line):
        lowered = match.lower()
        if lowered in CONTROL_KEYWORDS:
            continue
        identifiers.append(lowered)
    return identifiers


def _extract_assignment_targets(line: str) -> list[str]:
    return [match.lower() for match in ASSIGNMENT_PATTERN.findall(line)]


def _finalize_graph_artifact(
    nodes: dict[str, dict],
    edges: list[dict],
    *,
    backend: str,
    max_nodes: int,
    max_edges: int,
) -> dict:
    deduped_edges = []
    seen_edges = set()
    for edge in edges:
        key = (str(edge["source"]), str(edge["target"]), str(edge["type"]))
        if key in seen_edges:
            continue
        seen_edges.add(key)
        deduped_edges.append({"source": key[0], "target": key[1], "type": key[2]})
    sorted_nodes = sorted(nodes.values(), key=lambda row: _node_sort_key(row["id"]))
    sorted_edges = sorted(
        deduped_edges,
        key=lambda row: (_node_sort_key(row["source"]), _node_sort_key(row["target"]), row["type"]),
    )
    ast_sequence = _build_ast_sequence(sorted_nodes, sorted_edges)
    node_rows = sorted_nodes[:max_nodes]
    edge_rows = sorted_edges[:max_edges]
    node_type_counts = Counter(row["type"] for row in sorted_nodes)
    edge_type_counts = Counter(row["type"] for row in sorted_edges)
    return {
        "backend": backend,
        "ast_sequence": ast_sequence,
        "node_rows": node_rows,
        "edge_rows": edge_rows,
        "node_info": _format_node_rows(node_rows),
        "edge_info": _format_edge_rows(edge_rows),
        "graph_summary": {
            "nodes": len(sorted_nodes),
            "edges": len(sorted_edges),
            "node_types": dict(node_type_counts.most_common(10)),
            "edge_types": dict(edge_type_counts.most_common(10)),
        },
    }


def _build_ast_sequence(node_rows: list[dict], edge_rows: list[dict]) -> str:
    nodes_by_id = {str(row["id"]): row for row in node_rows}
    ast_children: dict[str, list[str]] = defaultdict(list)
    for edge in edge_rows:
        if edge["type"] != "IS_AST_PARENT":
            continue
        if edge["source"] in nodes_by_id and edge["target"] in nodes_by_id:
            ast_children[str(edge["source"])].append(str(edge["target"]))
    for children in ast_children.values():
        children.sort(key=_node_sort_key)

    method_nodes = [row["id"] for row in node_rows if row["type"] == "METHOD"]
    if method_nodes:
        root_id = str(method_nodes[0])
    elif node_rows:
        root_id = str(node_rows[0]["id"])
    else:
        return ""

    sequence: list[str] = []
    visited: set[str] = set()

    def visit(node_id: str) -> None:
        if node_id in visited or node_id not in nodes_by_id:
            return
        visited.add(node_id)
        node_type = nodes_by_id[node_id]["type"]
        sequence.append("(")
        sequence.append(node_type)
        for child_id in ast_children.get(node_id, []):
            visit(child_id)
        sequence.append(")")
        sequence.append(node_type)

    visit(root_id)
    if not sequence:
        sequence = [row["type"] for row in node_rows]
    return " ".join(sequence[:MAX_AST_TOKENS])


def _format_node_rows(rows: list[dict]) -> str:
    lines = ["NodeID\tNodeType\tCode"]
    for row in rows:
        lines.append(f"{row['id']}\t{row['type']}\t{truncate_text(row['code'], 90)}")
    return "\n".join(lines)


def _format_edge_rows(rows: list[dict]) -> str:
    lines = ["Node1\tNode2\tEdgeType"]
    for row in rows:
        lines.append(f"{row['source']}\t{row['target']}\t{row['type']}")
    return "\n".join(lines)


def _node_sort_key(value: str) -> tuple[int, str]:
    text = str(value)
    return (0, f"{int(text):020d}") if text.isdigit() else (1, text)


def _has_joern_tools() -> bool:
    joern_parse = resolve_joern_command("joern-parse")
    joern_export = resolve_joern_command("joern-export")
    return _command_exists(joern_parse) and _command_exists(joern_export)


@lru_cache(maxsize=1)
def _probe_joern_backend() -> tuple[bool, str | None]:
    if not _has_joern_tools():
        return False, "Joern executables were not found."
    try:
        artifact = _build_joern_graph_features(
            "int grace_probe(int value) { return value + 1; }",
            max_nodes=16,
            max_edges=16,
        )
    except Exception as exc:
        return False, f"Joern health probe failed: {exc}"
    if not artifact.get("node_rows"):
        return False, "Joern health probe returned no nodes."
    return True, None


def _command_exists(command: str) -> bool:
    return shutil.which(command) is not None or Path(command).exists()


def _candidate_joern_commands(executable_name: str) -> list[str]:
    if os.name == "nt":
        suffixes = [".bat", ".cmd", ".exe", ""]
    else:
        suffixes = ["", ".sh", ".bat", ".cmd", ".exe"]
    candidates = []
    for suffix in suffixes:
        candidates.append(executable_name + suffix)
    for suffix in suffixes:
        candidates.append(str(JOERN_HOME_DIR / (executable_name + suffix)))
        candidates.append(str(JOERN_INSTALL_ROOT / (executable_name + suffix)))
        candidates.append(str(JOERN_HOME_DIR / "bin" / (executable_name + suffix)))
    return candidates


### File `metrics.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/metrics.py
from typing import Any

import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

try:
    from scipy.stats import binomtest
except Exception:
    binomtest = None


def _safe_logit(probabilities: np.ndarray) -> np.ndarray:
    clipped = np.clip(probabilities.astype(float), 1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped))


def _safe_probabilities(probabilities: list[float] | np.ndarray) -> np.ndarray:
    return np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)


def _sigmoid(values: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-values))


def brier_score(labels: list[int] | np.ndarray, probabilities: list[float] | np.ndarray) -> float:
    y = np.asarray(labels, dtype=float)
    p = _safe_probabilities(probabilities)
    return float(np.mean((p - y) ** 2))


def negative_log_likelihood(labels: list[int] | np.ndarray, probabilities: list[float] | np.ndarray) -> float:
    y = np.asarray(labels, dtype=float)
    p = _safe_probabilities(probabilities)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))


def expected_calibration_error(labels: list[int] | np.ndarray, probabilities: list[float] | np.ndarray, *, bins: int = 10) -> float:
    y = np.asarray(labels, dtype=int)
    p = _safe_probabilities(probabilities)
    if len(y) == 0:
        return 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = float(len(y))
    error = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        if high < 1.0:
            mask = (p >= low) & (p < high)
        else:
            mask = (p >= low) & (p <= high)
        if not np.any(mask):
            continue
        accuracy = float(np.mean(y[mask]))
        confidence = float(np.mean(p[mask]))
        error += abs(accuracy - confidence) * (float(np.sum(mask)) / total)
    return float(error)


def _calibration_metrics(labels: np.ndarray, probabilities: np.ndarray) -> dict[str, float]:
    return {
        "brier": brier_score(labels, probabilities),
        "nll": negative_log_likelihood(labels, probabilities),
        "ece": expected_calibration_error(labels, probabilities),
    }


def fit_platt_scaler(probabilities: list[float] | np.ndarray, labels: list[int] | np.ndarray) -> dict[str, float]:
    y = np.asarray(labels, dtype=int)
    x = _safe_logit(np.asarray(probabilities, dtype=float)).reshape(-1, 1)
    if len(np.unique(y)) < 2:
        return {"coef": 1.0, "intercept": 0.0}
    model = LogisticRegression(max_iter=2000, solver="lbfgs")
    model.fit(x, y)
    return {
        "coef": float(model.coef_[0][0]),
        "intercept": float(model.intercept_[0]),
    }


def apply_platt_scaler(probabilities: list[float] | np.ndarray, calibration: dict[str, float]) -> np.ndarray:
    logits = _safe_logit(np.asarray(probabilities, dtype=float))
    calibrated = calibration["coef"] * logits + calibration["intercept"]
    return _sigmoid(calibrated)


def fit_temperature_scaler(probabilities: list[float] | np.ndarray, labels: list[int] | np.ndarray) -> dict[str, float]:
    probs = _safe_probabilities(probabilities)
    y = np.asarray(labels, dtype=int)
    logits = _safe_logit(probs)
    if len(np.unique(y)) < 2:
        return {"temperature": 1.0}
    best_temperature = 1.0
    best_nll = float("inf")
    for temperature in np.linspace(0.5, 5.0, 91):
        calibrated = _sigmoid(logits / float(temperature))
        score = negative_log_likelihood(y, calibrated)
        if score < best_nll:
            best_nll = score
            best_temperature = float(temperature)
    return {"temperature": best_temperature}


def apply_temperature_scaler(probabilities: list[float] | np.ndarray, calibration: dict[str, float]) -> np.ndarray:
    logits = _safe_logit(np.asarray(probabilities, dtype=float))
    temperature = float(calibration.get("temperature", 1.0))
    return _sigmoid(logits / max(temperature, 1e-6))


def fit_beta_calibration(probabilities: list[float] | np.ndarray, labels: list[int] | np.ndarray) -> dict[str, float]:
    probs = _safe_probabilities(probabilities)
    y = np.asarray(labels, dtype=int)
    if len(np.unique(y)) < 2:
        return {"coef_pos": 1.0, "coef_neg": 0.0, "intercept": 0.0}
    features = np.column_stack([np.log(probs), np.log1p(-probs)])
    model = LogisticRegression(max_iter=4000, solver="lbfgs")
    model.fit(features, y)
    return {
        "coef_pos": float(model.coef_[0][0]),
        "coef_neg": float(model.coef_[0][1]),
        "intercept": float(model.intercept_[0]),
    }


def apply_beta_calibration(probabilities: list[float] | np.ndarray, calibration: dict[str, float]) -> np.ndarray:
    probs = _safe_probabilities(probabilities)
    features = np.column_stack([np.log(probs), np.log1p(-probs)])
    logits = (
        float(calibration.get("coef_pos", 1.0)) * features[:, 0]
        + float(calibration.get("coef_neg", 0.0)) * features[:, 1]
        + float(calibration.get("intercept", 0.0))
    )
    return _sigmoid(logits)


def fit_isotonic_calibration(probabilities: list[float] | np.ndarray, labels: list[int] | np.ndarray) -> dict[str, Any]:
    probs = _safe_probabilities(probabilities)
    y = np.asarray(labels, dtype=int)
    if len(np.unique(y)) < 2:
        return {"thresholds": [0.0, 1.0], "values": [0.0, 1.0]}
    model = IsotonicRegression(out_of_bounds="clip")
    model.fit(probs, y)
    return {
        "thresholds": [float(value) for value in model.X_thresholds_.tolist()],
        "values": [float(value) for value in model.y_thresholds_.tolist()],
    }


def apply_isotonic_calibration(probabilities: list[float] | np.ndarray, calibration: dict[str, Any]) -> np.ndarray:
    probs = _safe_probabilities(probabilities)
    thresholds = np.asarray(calibration.get("thresholds", [0.0, 1.0]), dtype=float)
    values = np.asarray(calibration.get("values", [0.0, 1.0]), dtype=float)
    if len(thresholds) == 0 or len(values) == 0:
        return probs
    if len(thresholds) == 1:
        return np.full_like(probs, float(values[0]), dtype=float)
    return np.interp(probs, thresholds, values, left=float(values[0]), right=float(values[-1]))


def fit_calibrator(probabilities: list[float] | np.ndarray, labels: list[int] | np.ndarray, method: str = "auto") -> dict[str, Any]:
    requested = (method or "platt").strip().lower()
    probs = _safe_probabilities(probabilities)
    y = np.asarray(labels, dtype=int)
    if requested == "platt":
        return {"method": "platt", "parameters": fit_platt_scaler(probs, y)}
    if requested == "temperature":
        return {"method": "temperature", "parameters": fit_temperature_scaler(probs, y)}
    if requested == "beta":
        return {"method": "beta", "parameters": fit_beta_calibration(probs, y)}
    if requested == "isotonic":
        return {"method": "isotonic", "parameters": fit_isotonic_calibration(probs, y)}
    if requested != "auto":
        raise ValueError(f"Unsupported calibration method: {method}")

    candidates = [
        {"method": "platt", "parameters": fit_platt_scaler(probs, y)},
        {"method": "temperature", "parameters": fit_temperature_scaler(probs, y)},
        {"method": "isotonic", "parameters": fit_isotonic_calibration(probs, y)},
        {"method": "beta", "parameters": fit_beta_calibration(probs, y)},
    ]
    scored = []
    for candidate in candidates:
        calibrated = apply_calibrator(probs, candidate)
        metrics = _calibration_metrics(y, calibrated)
        scored_candidate = dict(candidate)
        scored_candidate["metrics"] = metrics
        scored.append(scored_candidate)
    best = min(scored, key=lambda item: (float(item["metrics"]["nll"]), float(item["metrics"]["ece"]), float(item["metrics"]["brier"])))
    best["method_requested"] = "auto"
    best["candidate_metrics"] = {item["method"]: item["metrics"] for item in scored}
    return best


def apply_calibrator(probabilities: list[float] | np.ndarray, calibration: dict[str, Any]) -> np.ndarray:
    method = str(calibration.get("method", "platt")).strip().lower()
    params = calibration.get("parameters", calibration)
    if method == "platt":
        return apply_platt_scaler(probabilities, params)
    if method == "temperature":
        return apply_temperature_scaler(probabilities, params)
    if method == "isotonic":
        return apply_isotonic_calibration(probabilities, params)
    if method == "beta":
        return apply_beta_calibration(probabilities, params)
    if method == "identity":
        return np.asarray(probabilities, dtype=float)
    raise ValueError(f"Unsupported calibration method: {method}")


def choose_low_threshold(probabilities: list[float] | np.ndarray, labels: list[int] | np.ndarray, target_recall: float) -> float:
    probs = np.asarray(probabilities, dtype=float)
    y = np.asarray(labels, dtype=int)
    positives = probs[y == 1]
    if len(positives) == 0:
        return 0.0
    best = 0.0
    for threshold in np.unique(np.round(np.sort(probs), 6)):
        recall = float(np.mean(positives > threshold))
        if recall >= target_recall:
            best = float(threshold)
        else:
            break
    return best


def choose_high_threshold(probabilities: list[float] | np.ndarray, labels: list[int] | np.ndarray, minimum: float) -> tuple[float, float]:
    probs = np.asarray(probabilities, dtype=float)
    y = np.asarray(labels, dtype=int)
    candidates = np.unique(np.round(np.sort(probs), 6))
    best_threshold = max(minimum + 0.05, 0.5)
    best_f1 = -1.0
    for threshold in candidates:
        if threshold <= minimum:
            continue
        predictions = (probs >= threshold).astype(int)
        score = f1_score(y, predictions, zero_division=0)
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)
    if best_f1 < 0:
        predictions = (probs >= 0.5).astype(int)
        best_threshold = max(minimum + 0.05, 0.5)
        best_f1 = float(f1_score(y, predictions, zero_division=0))
    return best_threshold, best_f1


def choose_best_f1_threshold(
    probabilities: list[float] | np.ndarray,
    labels: list[int] | np.ndarray,
    minimum: float = 0.0,
) -> tuple[float, float]:
    probs = np.asarray(probabilities, dtype=float)
    y = np.asarray(labels, dtype=int)
    candidates = np.unique(np.round(np.sort(probs), 6))
    best_threshold = float(minimum)
    best_f1 = -1.0
    for threshold in candidates:
        if threshold < minimum:
            continue
        predictions = (probs >= threshold).astype(int)
        score = f1_score(y, predictions, zero_division=0)
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)
    if best_f1 < 0:
        predictions = (probs >= minimum).astype(int)
        best_f1 = float(f1_score(y, predictions, zero_division=0))
    return best_threshold, best_f1


def compute_binary_metrics(labels: list[int] | np.ndarray, predictions: list[int] | np.ndarray, probabilities: list[float] | np.ndarray | None = None) -> dict[str, Any]:
    y_true = np.asarray(labels, dtype=int)
    y_pred = np.asarray(predictions, dtype=int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }
    if probabilities is not None:
        probs = np.asarray(probabilities, dtype=float)
        try:
            metrics["roc_auc"] = float(roc_auc_score(y_true, probs))
        except Exception:
            metrics["roc_auc"] = None
        try:
            metrics["pr_auc"] = float(average_precision_score(y_true, probs))
        except Exception:
            metrics["pr_auc"] = None
        metrics["brier"] = brier_score(y_true, probs)
        metrics["nll"] = negative_log_likelihood(y_true, probs)
        metrics["ece"] = expected_calibration_error(y_true, probs)
    return metrics


def bootstrap_f1_interval(labels: list[int] | np.ndarray, predictions: list[int] | np.ndarray, iterations: int = 1000, seed: int = 42) -> dict[str, float]:
    y_true = np.asarray(labels, dtype=int)
    y_pred = np.asarray(predictions, dtype=int)
    if len(y_true) == 0:
        return {"mean": 0.0, "low": 0.0, "high": 0.0}
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(iterations):
        sample = rng.integers(0, len(y_true), size=len(y_true))
        scores.append(f1_score(y_true[sample], y_pred[sample], zero_division=0))
    low, high = np.percentile(scores, [2.5, 97.5]).tolist()
    return {
        "mean": float(np.mean(scores)),
        "low": float(low),
        "high": float(high),
    }


def mcnemar_exact(labels: list[int] | np.ndarray, predictions_a: list[int] | np.ndarray, predictions_b: list[int] | np.ndarray) -> dict[str, float | int | None]:
    y_true = np.asarray(labels, dtype=int)
    a = np.asarray(predictions_a, dtype=int)
    b = np.asarray(predictions_b, dtype=int)
    a_correct = a == y_true
    b_correct = b == y_true
    n01 = int(np.sum(a_correct & ~b_correct))
    n10 = int(np.sum(~a_correct & b_correct))
    total = n01 + n10
    if total == 0:
        return {"n01": n01, "n10": n10, "p_value": 1.0}
    if binomtest is not None:
        p_value = float(binomtest(min(n01, n10), total, 0.5, alternative="two-sided").pvalue)
    else:
        p_value = None
    return {"n01": n01, "n10": n10, "p_value": p_value}


### File `localizer.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/localizer.py
from collections import Counter

from common import RISKY_APIS, extract_calls, normalize_code, tokenize_code, truncate_text


MEMORY_TERMS = {
    "malloc",
    "calloc",
    "realloc",
    "free",
    "memcpy",
    "memmove",
    "memset",
    "new",
    "delete",
}
VALIDATION_TERMS = {
    "if",
    "while",
    "for",
    "assert",
    "check",
    "validate",
    "verify",
    "length",
    "size",
    "limit",
    "bound",
    "guard",
}
POINTER_TOKENS = {"->", "*", "&", "[", "]"}


def _line_tokens(line: str) -> list[str]:
    return tokenize_code(normalize_code(line))


def _line_score(line: str, *, branch_scale: float) -> tuple[float, list[str]]:
    text = normalize_code(line)
    tokens = _line_tokens(text)
    calls = extract_calls(text)
    reasons: list[str] = []
    score = 0.0

    risky_calls = [call for call in calls if call.lower() in RISKY_APIS]
    if risky_calls:
        score += 3.0 + 0.4 * len(risky_calls)
        reasons.append(f"risky_api={','.join(risky_calls[:3])}")
    memory_terms = [token for token in tokens if token in MEMORY_TERMS]
    if memory_terms:
        score += 1.6
        reasons.append("memory_op")
    if any(token in POINTER_TOKENS for token in tokens) or "->" in text:
        score += 1.2
        reasons.append("pointer_or_array")
    if any(op in text for op in ["+", "-", "*", "/", "%"]) and any(token in tokens for token in ["size", "len", "offset", "index"]):
        score += 1.0
        reasons.append("index_arithmetic")
    if any(term in tokens for term in VALIDATION_TERMS):
        score += 0.8
        reasons.append("control_or_validation")
    if "=" in text and any(term in tokens for term in ["ptr", "buf", "dst", "src", "data"]):
        score += 0.8
        reasons.append("buffer_assignment")
    if "return" in tokens and any(token in tokens for token in ["error", "fail", "null", "nullptr", "invalid"]):
        score += 0.5
        reasons.append("error_path")

    if text.count("(") != text.count(")") or text.count("{") != text.count("}"):
        score += 0.4
        reasons.append("unbalanced_structure")
    return score * branch_scale, reasons


def locate_suspicious_slices(
    code: str,
    *,
    semantic_score: float,
    graph_score: float,
    fusion_score: float,
    risk_band: str,
    top_k: int | None = None,
    context_radius: int = 1,
) -> dict:
    text = normalize_code(code)
    raw_lines = text.splitlines()
    numbered_lines = [(index + 1, line.rstrip()) for index, line in enumerate(raw_lines) if line.strip()]
    if not numbered_lines:
        return {
            "top_lines": [],
            "slices": [],
            "slices_text": "No suspicious slices could be extracted.",
            "token_highlights": [],
        }

    branch_scale = 0.35 + 0.3 * float(semantic_score) + 0.35 * float(graph_score) + 0.2 * float(fusion_score)
    scored_lines = []
    for line_number, line_text in numbered_lines:
        score, reasons = _line_score(line_text, branch_scale=branch_scale)
        if score <= 0:
            continue
        scored_lines.append(
            {
                "line_number": line_number,
                "score": float(round(score, 4)),
                "code": line_text,
                "reasons": reasons,
            }
        )

    if not scored_lines:
        scored_lines = [
            {
                "line_number": line_number,
                "score": float(round(0.1 * branch_scale, 4)),
                "code": line_text,
                "reasons": ["fallback_context"],
            }
            for line_number, line_text in numbered_lines[:3]
        ]

    scored_lines.sort(key=lambda item: (item["score"], -item["line_number"]), reverse=True)
    if top_k is None:
        top_k = 2 if risk_band == "inspect" else 4 if risk_band == "high" else 1
    top_lines = scored_lines[:top_k]

    selected_line_numbers = sorted({row["line_number"] for row in top_lines})
    slices: list[dict] = []
    occupied = set()
    for center in selected_line_numbers:
        start = max(1, center - context_radius)
        end = min(len(raw_lines), center + context_radius)
        if any(line in occupied for line in range(start, end + 1)):
            continue
        for line in range(start, end + 1):
            occupied.add(line)
        slice_lines = [f"{line_no:>4}: {raw_lines[line_no - 1]}" for line_no in range(start, end + 1)]
        slice_score = max((row["score"] for row in top_lines if start <= row["line_number"] <= end), default=0.0)
        slice_reasons = []
        for row in top_lines:
            if start <= row["line_number"] <= end:
                slice_reasons.extend(row["reasons"])
        slices.append(
            {
                "start_line": start,
                "end_line": end,
                "score": float(round(slice_score, 4)),
                "reasons": sorted(set(slice_reasons)),
                "text": "\n".join(slice_lines),
            }
        )

    token_counter = Counter()
    for row in top_lines:
        for token in _line_tokens(row["code"]):
            if token.isidentifier() and token not in {"if", "for", "while", "return"}:
                token_counter[token] += 1
    token_highlights = [token for token, _ in token_counter.most_common(8)]
    slices_text = "\n\n".join(
        [
            "\n".join(
                [
                    f"Slice score={item['score']:.4f} | lines {item['start_line']}-{item['end_line']} | reasons={','.join(item['reasons']) or 'n/a'}",
                    item["text"],
                ]
            )
            for item in slices
        ]
    )
    return {
        "top_lines": top_lines,
        "slices": slices,
        "slices_text": truncate_text(slices_text, 2200) if slices_text else "No suspicious slices.",
        "token_highlights": token_highlights,
    }


__all__ = ["locate_suspicious_slices"]


### File `retrieval.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/retrieval.py
import json
import os
import time
from pathlib import Path
from typing import Any

import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

from common import SHARED_MODELS_DIR, ensure_dir, tokenize_code, truncate_text
from graphs import get_graph_features, resolve_graph_backend_with_notice


DEMO_BANK_SCHEMA_VERSION = 3
UNIXCODER_RETRIEVAL_MODEL_REPO_ID = "microsoft/unixcoder-base-nine"
CODET5_RETRIEVAL_MODEL_REPO_ID = "Salesforce/codet5-base"
DEFAULT_RETRIEVAL_MODEL_REPO_ID = UNIXCODER_RETRIEVAL_MODEL_REPO_ID
DEFAULT_EMBEDDING_MAX_LENGTH = 512
DEFAULT_EMBEDDING_BATCH_SIZE = 16
DEFAULT_LEXICAL_WEIGHT = 0.7
DEFAULT_SYNTACTIC_WEIGHT = 0.3
AST_SIMILARITY_MAX_TOKENS = 192
DEFAULT_BUILD_PROGRESS_EVERY = 250

_ENCODER_CACHE: dict[tuple[str, str], "SemanticRetrievalEncoder"] = {}


def default_retrieval_model_dir(repo_id: str = DEFAULT_RETRIEVAL_MODEL_REPO_ID) -> Path:
    return SHARED_MODELS_DIR / "retrieval" / repo_id.replace("/", "--")


def is_retrieval_model_downloaded(model_dir: Path) -> bool:
    required = ["config.json"]
    tokenizers = ["tokenizer.json", "spiece.model", "vocab.json"]
    weights = list(model_dir.glob("*.safetensors")) or list(model_dir.glob("pytorch_model*.bin"))
    return all((model_dir / name).exists() for name in required) and any((model_dir / name).exists() for name in tokenizers) and bool(weights)


def download_retrieval_model_snapshot(repo_id: str, local_dir: Path | None = None) -> Path:
    try:
        from huggingface_hub import snapshot_download
    except Exception as exc:
        raise RuntimeError(
            "Missing dependency `huggingface_hub`. Install it before downloading the retrieval model."
        ) from exc
    target_dir = Path(local_dir or default_retrieval_model_dir(repo_id))
    ensure_dir(target_dir)
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(target_dir),
        token=_resolve_hf_token(),
        allow_patterns=[
            "*.json",
            "*.txt",
            "*.model",
            "tokenizer*",
            "spiece.model",
            "*.safetensors",
            "pytorch_model*.bin",
        ],
    )
    return target_dir


class SemanticRetrievalEncoder:
    def __init__(
        self,
        *,
        model_name: str = DEFAULT_RETRIEVAL_MODEL_REPO_ID,
        model_dir: Path | None = None,
        max_length: int = DEFAULT_EMBEDDING_MAX_LENGTH,
        batch_size: int = DEFAULT_EMBEDDING_BATCH_SIZE,
        auto_download: bool = False,
    ) -> None:
        self.model_name = model_name
        self.model_dir = Path(model_dir or default_retrieval_model_dir(model_name))
        self.max_length = int(max_length)
        self.batch_size = int(batch_size)
        self.auto_download = auto_download
        self._runtime: dict[str, Any] | None = None
        self._tokenizer = None
        self._model = None
        self._device = None

    def export_config(self) -> dict[str, Any]:
        lowered = self.model_name.lower()
        semantic_backend = "unixcoder" if "unixcoder" in lowered else "codet5" if "codet5" in lowered else "hf_encoder"
        return {
            "semantic_backend": semantic_backend,
            "model_name": self.model_name,
            "model_dir": str(self.model_dir),
            "max_length": self.max_length,
            "batch_size": self.batch_size,
            "auto_download": self.auto_download,
        }

    def prepare(self) -> None:
        if self._model is not None and self._tokenizer is not None:
            return
        if self.auto_download and not is_retrieval_model_downloaded(self.model_dir):
            download_retrieval_model_snapshot(self.model_name, self.model_dir)
        if not is_retrieval_model_downloaded(self.model_dir):
            raise FileNotFoundError(
                f"Retrieval model directory is missing or incomplete: {self.model_dir}. "
                "Download the required semantic encoder checkpoint or enable auto-download."
            )
        runtime = self._load_runtime()
        torch = runtime["torch"]
        AutoModel = runtime["AutoModel"]
        AutoTokenizer = runtime["AutoTokenizer"]
        RobertaTokenizer = runtime["RobertaTokenizer"]
        T5EncoderModel = runtime["T5EncoderModel"]
        tokenizer = None
        vocab_path = self.model_dir / "vocab.json"
        merges_path = self.model_dir / "merges.txt"
        if vocab_path.exists() and merges_path.exists():
            tokenizer = RobertaTokenizer(
                vocab_file=str(vocab_path),
                merges_file=str(merges_path),
                errors="replace",
                bos_token="<s>",
                eos_token="</s>",
                sep_token="</s>",
                cls_token="<s>",
                unk_token="<unk>",
                pad_token="<pad>",
                mask_token="<mask>",
                add_prefix_space=False,
            )
            tokenizer.model_max_length = self.max_length
            tokenizer.additional_special_tokens = [f"<extra_id_{index}>" for index in range(99, -1, -1)]
        if tokenizer is None:
            try:
                tokenizer = AutoTokenizer.from_pretrained(
                    self.model_dir,
                    local_files_only=True,
                    trust_remote_code=False,
                    use_fast=False,
                )
            except Exception:
                tokenizer = AutoTokenizer.from_pretrained(
                    self.model_dir,
                    local_files_only=True,
                    trust_remote_code=False,
                    use_fast=True,
                )
        config_payload = json.loads((self.model_dir / "config.json").read_text(encoding="utf-8"))
        if str(config_payload.get("model_type", "")).strip().lower() == "t5":
            model = T5EncoderModel.from_pretrained(self.model_dir, local_files_only=True, trust_remote_code=False)
        else:
            model = AutoModel.from_pretrained(self.model_dir, local_files_only=True, trust_remote_code=False)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
        self._runtime = runtime
        self._tokenizer = tokenizer
        self._model = model
        self._device = device

    def encode_texts(self, texts: list[str]) -> np.ndarray:
        self.prepare()
        if not texts:
            return np.zeros((0, 0), dtype=np.float32)
        torch = self._runtime["torch"]
        embeddings: list[np.ndarray] = []
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start : start + self.batch_size]
            encoded = self._tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            encoded = {name: tensor.to(self._device) for name, tensor in encoded.items()}
            with torch.inference_mode():
                outputs = self._model(**encoded)
            hidden = outputs.last_hidden_state
            attention_mask = encoded["attention_mask"].unsqueeze(-1).expand(hidden.size()).float()
            pooled = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1.0)
            normalized = torch.nn.functional.normalize(pooled, p=2, dim=1)
            embeddings.append(normalized.detach().cpu().numpy().astype(np.float32))
        return np.vstack(embeddings)

    def _load_runtime(self) -> dict[str, Any]:
        if self._runtime is not None:
            return self._runtime
        try:
            import torch
            from transformers import AutoModel, AutoTokenizer, RobertaTokenizer, T5EncoderModel
        except Exception as exc:
            raise RuntimeError(
                "Missing retrieval dependencies. Install `torch`, `transformers`, and tokenizer dependencies "
                "such as `sentencepiece` before using semantic retrieval."
            ) from exc
        self._runtime = {
            "torch": torch,
            "AutoModel": AutoModel,
            "AutoTokenizer": AutoTokenizer,
            "RobertaTokenizer": RobertaTokenizer,
            "T5EncoderModel": T5EncoderModel,
        }
        return self._runtime


def build_demo_bank(
    records: list[dict],
    max_examples_per_label: int,
    max_features: int,
    random_seed: int,
    *,
    semantic_backend: str = "auto",
    semantic_model_name: str = DEFAULT_RETRIEVAL_MODEL_REPO_ID,
    semantic_model_dir: Path | None = None,
    semantic_batch_size: int = DEFAULT_EMBEDDING_BATCH_SIZE,
    semantic_max_length: int = DEFAULT_EMBEDDING_MAX_LENGTH,
    auto_download_semantic_model: bool = False,
    graph_backend: str = "auto",
    progress_every: int = DEFAULT_BUILD_PROGRESS_EVERY,
) -> dict:
    rng = np.random.default_rng(random_seed)
    grouped = {0: [], 1: []}
    for record in records:
        grouped[int(record["label"])].append(record)

    sampled = []
    graph_backend_counts = {}
    graph_backend_resolved, graph_backend_notice = resolve_graph_backend_with_notice(graph_backend)
    total_records = sum(min(len(items), max_examples_per_label) for items in grouped.values())
    started = time.perf_counter()
    print(
        f"[demo-bank] Building graph features for {total_records} records "
        f"(graph={graph_backend_resolved}, semantic={semantic_backend})"
    )
    if graph_backend_notice:
        print(f"[demo-bank] Graph backend notice: {graph_backend_notice}")
    processed = 0
    for label, items in grouped.items():
        if len(items) > max_examples_per_label:
            indices = rng.choice(len(items), size=max_examples_per_label, replace=False)
            items = [items[index] for index in sorted(indices)]
        for record in items:
            graph_features = get_graph_features(record, graph_backend=graph_backend)
            graph_backend_counts[graph_features["backend"]] = graph_backend_counts.get(graph_features["backend"], 0) + 1
            tokens = tokenize_code(record["code"])
            ast_sequence = graph_features.get("ast_sequence", "")
            sampled.append(
                {
                    "record_id": record["record_id"],
                    "label": int(record["label"]),
                    "project": record.get("project", ""),
                    "dataset": record.get("dataset", ""),
                    "code": record["code"],
                    "tokenized_text": " ".join(tokens),
                    "tokens": tokens,
                    "ast_sequence": ast_sequence,
                    "ast_tokens": ast_sequence.split(),
                    "graph_backend": graph_features["backend"],
                }
            )
            processed += 1
            if progress_every and (processed % progress_every == 0 or processed == total_records):
                elapsed = time.perf_counter() - started
                print(f"[demo-bank] Graph features ready: {processed}/{total_records} in {elapsed:.1f}s")

    print(f"[demo-bank] Building semantic store with backend request={semantic_backend}")
    semantic_backend_used, semantic_payload, semantic_notice = _build_semantic_store(
        sampled,
        semantic_backend=semantic_backend,
        semantic_model_name=semantic_model_name,
        semantic_model_dir=semantic_model_dir,
        semantic_batch_size=semantic_batch_size,
        semantic_max_length=semantic_max_length,
        max_features=max_features,
        auto_download_semantic_model=auto_download_semantic_model,
    )
    print(f"[demo-bank] Semantic store ready with backend={semantic_backend_used}")
    label_indices = {
        0: [index for index, row in enumerate(sampled) if row["label"] == 0],
        1: [index for index, row in enumerate(sampled) if row["label"] == 1],
    }
    return {
        "schema_version": DEMO_BANK_SCHEMA_VERSION,
        "records": sampled,
        "label_indices": label_indices,
        "semantic_backend": semantic_backend_used,
        "semantic_notice": semantic_notice,
        "semantic_config": semantic_payload["config"],
        "semantic_store": semantic_payload["store"],
        "graph_backend_requested": graph_backend,
        "graph_backend_resolved": graph_backend_resolved,
        "graph_backend_notice": graph_backend_notice,
        "graph_backend_counts": graph_backend_counts,
        "rerank": {
            "lexical_weight": DEFAULT_LEXICAL_WEIGHT,
            "syntactic_weight": DEFAULT_SYNTACTIC_WEIGHT,
        },
    }


def _build_semantic_store(
    sampled: list[dict],
    *,
    semantic_backend: str,
    semantic_model_name: str,
    semantic_model_dir: Path | None,
    semantic_batch_size: int,
    semantic_max_length: int,
    max_features: int,
    auto_download_semantic_model: bool,
) -> tuple[str, dict, str | None]:
    requested = (semantic_backend or "auto").strip().lower()
    if requested not in {"auto", "unixcoder", "codet5", "tfidf"}:
        raise ValueError(f"Unsupported retrieval backend: {semantic_backend}")
    if requested in {"auto", "unixcoder", "codet5"}:
        try:
            encoder = SemanticRetrievalEncoder(
                model_name=semantic_model_name,
                model_dir=semantic_model_dir,
                max_length=semantic_max_length,
                batch_size=semantic_batch_size,
                auto_download=auto_download_semantic_model,
            )
            embeddings = encoder.encode_texts([row["code"] for row in sampled])
            exported = encoder.export_config()
            return str(exported.get("semantic_backend", "hf_encoder")), {"config": exported, "store": {"embeddings": embeddings}}, None
        except Exception as exc:
            if requested in {"unixcoder", "codet5"}:
                raise
            semantic_notice = f"Falling back to TF-IDF retrieval because the semantic encoder could not be loaded: {exc}"
            vectorizer = TfidfVectorizer(
                tokenizer=str.split,
                preprocessor=None,
                token_pattern=None,
                lowercase=False,
                ngram_range=(1, 2),
                max_features=max_features,
                min_df=2,
                sublinear_tf=True,
            )
            matrix = vectorizer.fit_transform([row["tokenized_text"] for row in sampled])
            return "tfidf", {"config": {"semantic_backend": "tfidf", "max_features": max_features}, "store": {"vectorizer": vectorizer, "matrix": matrix}}, semantic_notice
    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        max_features=max_features,
        min_df=2,
        sublinear_tf=True,
    )
    matrix = vectorizer.fit_transform([row["tokenized_text"] for row in sampled])
    return "tfidf", {"config": {"semantic_backend": "tfidf", "max_features": max_features}, "store": {"vectorizer": vectorizer, "matrix": matrix}}, None


def save_demo_bank(path, bank: dict) -> None:
    ensure_dir(path.parent)
    joblib.dump(bank, path)


def load_demo_bank(path) -> dict:
    return joblib.load(path)


def retrieve_examples(
    query_code: str,
    bank: dict,
    total_k: int,
    calibrated_probability: float,
    candidate_pool_size: int = 24,
    demo_char_limit: int = 1800,
    *,
    query_record: dict | None = None,
    graph_backend: str | None = None,
    query_graph_features: dict | None = None,
) -> list[dict]:
    query_record = dict(query_record or {})
    query_record["code"] = query_code
    graph_features = query_graph_features or get_graph_features(
        query_record,
        dataset_name=query_record.get("dataset"),
        graph_backend=graph_backend or bank.get("graph_backend_requested", "auto"),
    )
    query_tokens = tokenize_code(query_code)
    query_ast_tokens = graph_features.get("ast_sequence", "").split()
    semantic = _semantic_scores(query_code, query_tokens, bank)

    label_indices = bank["label_indices"]
    records = bank["records"]
    rerank = bank.get("rerank", {})
    lexical_weight = float(rerank.get("lexical_weight", DEFAULT_LEXICAL_WEIGHT))
    syntactic_weight = float(rerank.get("syntactic_weight", DEFAULT_SYNTACTIC_WEIGHT))

    vulnerable_ratio = 0.5 if total_k <= 2 else max(0.5, min(0.75, calibrated_probability))
    vulnerable_k = min(total_k, max(1, int(round(total_k * vulnerable_ratio))))
    benign_k = max(0, total_k - vulnerable_k)

    def rank_label(label: int, needed: int) -> list[dict]:
        if needed <= 0 or not label_indices[label]:
            return []
        candidates = label_indices[label]
        candidate_scores = semantic[candidates]
        top_local = np.argsort(candidate_scores)[-candidate_pool_size:][::-1]
        scored = []
        for local_index in top_local:
            bank_index = candidates[int(local_index)]
            record = records[bank_index]
            lexical = _jaccard(query_tokens, record["tokens"])
            syntactic = _syntactic_similarity(query_ast_tokens, record["ast_tokens"])
            mixed = lexical_weight * lexical + syntactic_weight * syntactic
            scored.append((mixed, float(semantic[bank_index]), lexical, syntactic, record))
        scored.sort(key=lambda item: (item[0], item[1]), reverse=True)
        selected = []
        for mixed, semantic_score, lexical, syntactic, record in scored[:needed]:
            selected.append(
                {
                    "record_id": record["record_id"],
                    "label": record["label"],
                    "project": record["project"],
                    "code": truncate_text(record["code"], demo_char_limit),
                    "mixed_score": mixed,
                    "semantic_score": semantic_score,
                    "lexical_score": lexical,
                    "syntactic_score": syntactic,
                }
            )
        return selected

    vulnerable = rank_label(1, vulnerable_k)
    benign = rank_label(0, benign_k)
    results = _interleave(vulnerable, benign, total_k)
    if len(results) < total_k:
        spill = rank_label(1, total_k - len(results)) + rank_label(0, total_k - len(results))
        for record in spill:
            if all(existing["record_id"] != record["record_id"] for existing in results):
                results.append(record)
            if len(results) >= total_k:
                break
    return results


def _semantic_scores(query_code: str, query_tokens: list[str], bank: dict) -> np.ndarray:
    backend = bank.get("semantic_backend", "tfidf")
    store = bank.get("semantic_store", {})
    if backend in {"codet5", "unixcoder", "hf_encoder"}:
        encoder = _get_encoder(bank.get("semantic_config", {}))
        query_embedding = encoder.encode_texts([query_code])
        return np.dot(store["embeddings"], query_embedding[0])
    vectorizer = store["vectorizer"]
    matrix = store["matrix"]
    query_vector = vectorizer.transform([" ".join(query_tokens)])
    return linear_kernel(query_vector, matrix).ravel()


def _get_encoder(config: dict) -> SemanticRetrievalEncoder:
    model_name = config.get("model_name", DEFAULT_RETRIEVAL_MODEL_REPO_ID)
    model_dir = Path(config.get("model_dir") or default_retrieval_model_dir(model_name))
    cache_key = (model_name, str(model_dir))
    encoder = _ENCODER_CACHE.get(cache_key)
    if encoder is None:
        encoder = SemanticRetrievalEncoder(
            model_name=model_name,
            model_dir=model_dir,
            max_length=int(config.get("max_length", DEFAULT_EMBEDDING_MAX_LENGTH)),
            batch_size=int(config.get("batch_size", DEFAULT_EMBEDDING_BATCH_SIZE)),
            auto_download=bool(config.get("auto_download", False)),
        )
        _ENCODER_CACHE[cache_key] = encoder
    return encoder


def _jaccard(left: list[str], right: list[str]) -> float:
    left_set = set(left)
    right_set = set(right)
    if not left_set and not right_set:
        return 1.0
    union = left_set | right_set
    if not union:
        return 0.0
    return len(left_set & right_set) / len(union)


def _syntactic_similarity(left: list[str], right: list[str]) -> float:
    left_seq = left[:AST_SIMILARITY_MAX_TOKENS]
    right_seq = right[:AST_SIMILARITY_MAX_TOKENS]
    if not left_seq and not right_seq:
        return 1.0
    denominator = len(left_seq) + len(right_seq)
    if denominator == 0:
        return 0.0
    distance = _levenshtein_distance(left_seq, right_seq)
    return max(0.0, (denominator - distance) / denominator)


def _levenshtein_distance(left: list[str], right: list[str]) -> int:
    if left == right:
        return 0
    if not left:
        return len(right)
    if not right:
        return len(left)
    previous = list(range(len(right) + 1))
    for row_index, left_token in enumerate(left, start=1):
        current = [row_index]
        for col_index, right_token in enumerate(right, start=1):
            insert_cost = current[col_index - 1] + 1
            delete_cost = previous[col_index] + 1
            substitute_cost = previous[col_index - 1] + (0 if left_token == right_token else 1)
            current.append(min(insert_cost, delete_cost, substitute_cost))
        previous = current
    return previous[-1]


def _interleave(primary: list[dict], secondary: list[dict], total_k: int) -> list[dict]:
    results = []
    while len(results) < total_k and (primary or secondary):
        if primary:
            results.append(primary.pop(0))
        if len(results) >= total_k:
            break
        if secondary:
            results.append(secondary.pop(0))
    while len(results) < total_k and primary:
        results.append(primary.pop(0))
    while len(results) < total_k and secondary:
        results.append(secondary.pop(0))
    return results


def _resolve_hf_token() -> str | None:
    for env_name in ["HUGGINGFACE_HUB_TOKEN", "HF_TOKEN"]:
        value = os.getenv(env_name)
        if value:
            return value.strip().strip('"').strip("'")
    return None


### File `hybrid_prefilter.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/hybrid_prefilter.py
import math
import os
import time
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import tensorflow as tf
from tensorflow import keras

from common import (
    FEATURES_DIR,
    MODELS_DIR,
    RISKY_APIS,
    SPLITS_DIR,
    build_skeleton,
    dump_json,
    ensure_dir,
    extract_calls,
    get_record_code,
    iter_jsonl,
    normalize_code,
    read_jsonl,
    tokenize_code,
)
from graphs import get_graph_features
from retrieval import DEFAULT_RETRIEVAL_MODEL_REPO_ID, SemanticRetrievalEncoder, default_retrieval_model_dir


FEATURE_STORE_SCHEMA_VERSION = 1
PREFILTER_MODEL_SCHEMA_VERSION = 1
DEFAULT_PREFILTER_MODEL_NAME = "hybrid_multiview_prefilter"
DEFAULT_FEATURE_PROGRESS_EVERY = 256

NUMERIC_FEATURE_NAMES = [
    "log_token_count",
    "unique_token_ratio",
    "log_line_count",
    "parameter_count",
    "log_call_count",
    "risky_call_count",
    "risky_call_ratio",
    "control_density",
    "pointer_density",
    "array_access_density",
    "numeric_literal_density",
    "memory_ops_count",
    "backend_is_joern",
    "log_graph_nodes",
    "log_graph_edges",
    "graph_avg_degree",
    "graph_call_ratio",
    "graph_control_ratio",
    "graph_expression_ratio",
    "graph_cfg_ratio",
    "graph_ast_ratio",
    "graph_reaches_ratio",
    "graph_max_out_degree",
    "log_ast_token_count",
]

MEMORY_KEYWORDS = {
    "malloc",
    "calloc",
    "realloc",
    "free",
    "new",
    "delete",
    "memcpy",
    "memmove",
    "memset",
}
CONTROL_TOKENS = {"if", "else", "switch", "case", "for", "while", "do", "goto", "return", "break", "continue"}


def _feature_store_suffix() -> str:
    suffix = os.getenv("GRACE_FEATURE_STORE_SUFFIX", "").strip()
    if not suffix:
        return ""
    return suffix if suffix.startswith("_") else f"_{suffix}"


def feature_store_path(dataset_name: str, split_name: str) -> Path:
    return FEATURES_DIR / dataset_name / f"{split_name}_features{_feature_store_suffix()}.joblib"


def _safe_div(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def _count_numeric_literals(code: str) -> int:
    total = 0
    for token in tokenize_code(code):
        if token == "num_lit":
            total += 1
    return total


def _compute_numeric_features(code: str, graph_features: dict) -> np.ndarray:
    text = normalize_code(code)
    tokens = tokenize_code(text)
    token_count = max(len(tokens), 1)
    unique_token_ratio = _safe_div(len(set(tokens)), token_count)
    line_count = len([line for line in text.splitlines() if line.strip()])
    calls = extract_calls(text)
    risky_call_count = sum(1 for call in calls if call.lower() in RISKY_APIS)
    control_count = sum(tokens.count(token) for token in CONTROL_TOKENS)
    pointer_ops = text.count("->") + text.count("*") + text.count("&")
    array_accesses = text.count("[")
    numeric_literals = _count_numeric_literals(text)
    memory_ops_count = sum(1 for token in tokens if token in MEMORY_KEYWORDS)

    summary = graph_features.get("graph_summary", {})
    node_types = summary.get("node_types", {})
    edge_types = summary.get("edge_types", {})
    node_count = int(summary.get("nodes", 0))
    edge_count = int(summary.get("edges", 0))
    graph_call_ratio = _safe_div(float(node_types.get("CALL", 0)), node_count)
    graph_control_ratio = _safe_div(float(node_types.get("CONTROL_STRUCTURE", 0)), node_count)
    graph_expression_ratio = _safe_div(float(node_types.get("EXPRESSION", 0)), node_count)
    graph_cfg_ratio = _safe_div(float(edge_types.get("FLOWS_TO", 0)), edge_count)
    graph_ast_ratio = _safe_div(float(edge_types.get("IS_AST_PARENT", 0)), edge_count)
    graph_reaches_ratio = _safe_div(float(edge_types.get("REACHES", 0)), edge_count)
    ast_token_count = len((graph_features.get("ast_sequence") or "").split())

    out_degree: dict[str, int] = {}
    for edge in graph_features.get("edge_rows", []):
        source = str(edge.get("source"))
        out_degree[source] = out_degree.get(source, 0) + 1
    graph_max_out_degree = max(out_degree.values()) if out_degree else 0
    graph_avg_degree = _safe_div(float(sum(out_degree.values())), max(len(out_degree), 1))

    values = [
        math.log1p(token_count),
        unique_token_ratio,
        math.log1p(line_count),
        float(_estimate_parameter_count(text)),
        math.log1p(len(calls)),
        float(risky_call_count),
        _safe_div(float(risky_call_count), max(len(calls), 1)),
        _safe_div(float(control_count), token_count),
        _safe_div(float(pointer_ops), max(line_count, 1)),
        _safe_div(float(array_accesses), max(line_count, 1)),
        _safe_div(float(numeric_literals), token_count),
        float(memory_ops_count),
        1.0 if graph_features.get("backend") == "joern" else 0.0,
        math.log1p(node_count),
        math.log1p(edge_count),
        graph_avg_degree,
        graph_call_ratio,
        graph_control_ratio,
        graph_expression_ratio,
        graph_cfg_ratio,
        graph_ast_ratio,
        graph_reaches_ratio,
        float(graph_max_out_degree),
        math.log1p(ast_token_count),
    ]
    return np.asarray(values, dtype=np.float32)


def _estimate_parameter_count(code: str) -> int:
    signature_head = normalize_code(code)[:1000]
    start = signature_head.find("(")
    end = signature_head.find(")", start + 1)
    if start < 0 or end < 0 or end <= start:
        return 0
    inside = signature_head[start + 1 : end].strip()
    if not inside or inside == "void":
        return 0
    return len([part for part in inside.split(",") if part.strip()])


def _iter_records(split_path: Path, limit: int | None = None) -> Iterable[dict]:
    seen = 0
    for record in iter_jsonl(split_path):
        code = get_record_code(record)
        if not code:
            continue
        payload = dict(record)
        payload["code"] = code
        yield payload
        seen += 1
        if limit is not None and seen >= limit:
            break


def build_feature_store(
    dataset_name: str,
    split_name: str,
    *,
    semantic_model_name: str = DEFAULT_RETRIEVAL_MODEL_REPO_ID,
    semantic_model_dir: Path | None = None,
    graph_backend: str = "auto",
    force_rebuild: bool = False,
    auto_download_semantic_model: bool = False,
    batch_size: int = 16,
    limit: int | None = None,
    progress_every: int = DEFAULT_FEATURE_PROGRESS_EVERY,
) -> dict:
    output_path = feature_store_path(dataset_name, split_name)
    if output_path.exists() and not force_rebuild:
        payload = joblib.load(output_path)
        if payload.get("schema_version") == FEATURE_STORE_SCHEMA_VERSION:
            return payload

    split_path = SPLITS_DIR / dataset_name / f"{split_name}.jsonl"
    if not split_path.exists():
        raise FileNotFoundError(f"Missing split file: {split_path}")

    encoder = SemanticRetrievalEncoder(
        model_name=semantic_model_name,
        model_dir=semantic_model_dir or default_retrieval_model_dir(semantic_model_name),
        batch_size=batch_size,
        auto_download=auto_download_semantic_model,
    )

    rows = list(_iter_records(split_path, limit=limit))
    if not rows:
        raise RuntimeError(f"No records found for dataset={dataset_name} split={split_name}")

    print(
        f"[feature-store] dataset={dataset_name} split={split_name} "
        f"| rows={len(rows)} | semantic_model={semantic_model_name} | graph={graph_backend}"
    )
    started = time.perf_counter()
    token_texts: list[str] = []
    ast_texts: list[str] = []
    labels: list[int] = []
    record_ids: list[str] = []
    code_hashes: list[str] = []
    graph_backends: list[str] = []
    numeric_features: list[np.ndarray] = []
    codes: list[str] = []
    semantic_inputs: list[str] = []

    for index, record in enumerate(rows, start=1):
        graph_features = get_graph_features(record, graph_backend=graph_backend, force_rebuild=force_rebuild)
        code = record["code"]
        token_texts.append(" ".join(tokenize_code(code)))
        ast_text = graph_features.get("ast_sequence") or build_skeleton(code)
        ast_texts.append(ast_text)
        numeric_features.append(_compute_numeric_features(code, graph_features))
        labels.append(int(record["label"]))
        record_ids.append(str(record["record_id"]))
        code_hashes.append(str(record.get("code_hash") or ""))
        graph_backends.append(str(graph_features.get("backend") or "unknown"))
        semantic_inputs.append(code)
        codes.append(code)
        if progress_every and (index % progress_every == 0 or index == len(rows)):
            elapsed = time.perf_counter() - started
            print(f"[feature-store] prepared graph view {index}/{len(rows)} in {elapsed:.1f}s")

    semantic_embeddings = encoder.encode_texts(semantic_inputs)
    payload = {
        "schema_version": FEATURE_STORE_SCHEMA_VERSION,
        "dataset": dataset_name,
        "split": split_name,
        "semantic_config": encoder.export_config(),
        "graph_backend_requested": graph_backend,
        "record_ids": record_ids,
        "code_hashes": code_hashes,
        "graph_backends": graph_backends,
        "labels": np.asarray(labels, dtype=np.int32),
        "token_texts": token_texts,
        "ast_texts": ast_texts,
        "numeric_features": np.asarray(numeric_features, dtype=np.float32),
        "semantic_embeddings": np.asarray(semantic_embeddings, dtype=np.float32),
        "codes": codes,
        "feature_names": list(NUMERIC_FEATURE_NAMES),
    }
    ensure_dir(output_path.parent)
    joblib.dump(payload, output_path)
    print(f"[feature-store] saved to {output_path}")
    return payload


def load_feature_store(dataset_name: str, split_name: str) -> dict:
    path = feature_store_path(dataset_name, split_name)
    if not path.exists():
        raise FileNotFoundError(f"Missing feature store: {path}")
    payload = joblib.load(path)
    if payload.get("schema_version") != FEATURE_STORE_SCHEMA_VERSION:
        raise RuntimeError(f"Incompatible feature store schema for {path}")
    return payload


def _build_prefilter_model(
    *,
    token_max_tokens: int,
    token_sequence_length: int,
    token_embedding_dim: int,
    token_filters: int,
    ast_max_tokens: int,
    ast_sequence_length: int,
    ast_embedding_dim: int,
    ast_filters: int,
    semantic_dim: int,
    numeric_dim: int,
    projection_dim: int,
    dense_units: int,
    dropout_rate: float,
) -> keras.Model:
    token_vectorizer = keras.layers.TextVectorization(
        standardize=None,
        split="whitespace",
        output_mode="int",
        output_sequence_length=token_sequence_length,
        max_tokens=token_max_tokens,
        name="token_vectorizer",
    )
    ast_vectorizer = keras.layers.TextVectorization(
        standardize=None,
        split="whitespace",
        output_mode="int",
        output_sequence_length=ast_sequence_length,
        max_tokens=ast_max_tokens,
        name="ast_vectorizer",
    )

    token_input = keras.Input(shape=(), dtype=tf.string, name="token_text")
    ast_input = keras.Input(shape=(), dtype=tf.string, name="ast_text")
    semantic_input = keras.Input(shape=(semantic_dim,), dtype=tf.float32, name="semantic_embedding")
    numeric_input = keras.Input(shape=(numeric_dim,), dtype=tf.float32, name="numeric_features")

    token_branch = token_vectorizer(token_input)
    token_branch = keras.layers.Embedding(token_max_tokens, token_embedding_dim, name="token_embedding")(token_branch)
    token_branch = keras.layers.SpatialDropout1D(dropout_rate)(token_branch)
    token_branch = keras.layers.Conv1D(token_filters, 5, padding="same", activation="relu")(token_branch)
    token_branch = keras.layers.Conv1D(token_filters, 3, padding="same", activation="relu")(token_branch)
    token_branch = keras.layers.GlobalMaxPooling1D()(token_branch)

    ast_branch = ast_vectorizer(ast_input)
    ast_branch = keras.layers.Embedding(ast_max_tokens, ast_embedding_dim, name="ast_embedding")(ast_branch)
    ast_branch = keras.layers.SpatialDropout1D(dropout_rate)(ast_branch)
    ast_branch = keras.layers.Conv1D(ast_filters, 5, padding="same", activation="relu")(ast_branch)
    ast_branch = keras.layers.Conv1D(ast_filters, 3, padding="same", activation="relu")(ast_branch)
    ast_branch = keras.layers.GlobalMaxPooling1D()(ast_branch)

    semantic_branch = keras.layers.Dense(projection_dim, activation="relu")(semantic_input)
    semantic_branch = keras.layers.Dropout(dropout_rate)(semantic_branch)
    semantic_hidden = keras.layers.Dense(max(32, projection_dim // 2), activation="relu")(semantic_branch)
    semantic_score = keras.layers.Dense(1, activation="sigmoid", name="semantic_score")(semantic_hidden)

    graph_branch = keras.layers.Concatenate(name="graph_concat")([ast_branch, numeric_input])
    graph_branch = keras.layers.Dense(projection_dim, activation="relu")(graph_branch)
    graph_branch = keras.layers.Dropout(dropout_rate)(graph_branch)
    graph_hidden = keras.layers.Dense(max(32, projection_dim // 2), activation="relu")(graph_branch)
    graph_score = keras.layers.Dense(1, activation="sigmoid", name="graph_score")(graph_hidden)

    fusion_branch = keras.layers.Concatenate(name="fusion_concat")(
        [token_branch, ast_branch, semantic_branch, numeric_input, semantic_score, graph_score]
    )
    fusion_branch = keras.layers.Dense(dense_units, activation="relu")(fusion_branch)
    fusion_branch = keras.layers.Dropout(dropout_rate)(fusion_branch)
    fusion_branch = keras.layers.Dense(max(64, dense_units // 2), activation="relu")(fusion_branch)
    fusion_score = keras.layers.Dense(1, activation="sigmoid", name="fusion_score")(fusion_branch)

    model = keras.Model(
        inputs={
            "token_text": token_input,
            "ast_text": ast_input,
            "semantic_embedding": semantic_input,
            "numeric_features": numeric_input,
        },
        outputs={
            "fusion_score": fusion_score,
            "semantic_score": semantic_score,
            "graph_score": graph_score,
        },
    )
    model.token_vectorizer = token_vectorizer
    model.ast_vectorizer = ast_vectorizer
    return model


def _prepare_scaled_inputs(payload: dict, numeric_mean: np.ndarray, numeric_std: np.ndarray) -> dict[str, np.ndarray]:
    scaled_numeric = (payload["numeric_features"] - numeric_mean) / numeric_std
    return {
        "token_text": np.asarray(payload["token_texts"], dtype=object),
        "ast_text": np.asarray(payload["ast_texts"], dtype=object),
        "semantic_embedding": np.asarray(payload["semantic_embeddings"], dtype=np.float32),
        "numeric_features": np.asarray(scaled_numeric, dtype=np.float32),
    }


def _targets(labels: np.ndarray) -> dict[str, np.ndarray]:
    values = labels.astype(np.float32).reshape(-1, 1)
    return {
        "fusion_score": values,
        "semantic_score": values,
        "graph_score": values,
    }


def _sample_weights_from_array(weights: np.ndarray) -> dict[str, np.ndarray]:
    weights = np.asarray(weights, dtype=np.float32)
    return {
        "fusion_score": weights,
        "semantic_score": weights,
        "graph_score": weights,
    }


def _sample_weights(labels: np.ndarray, positive_weight: float, extra_weights: np.ndarray | None = None) -> dict[str, np.ndarray]:
    weights = np.where(labels.astype(int) == 1, positive_weight, 1.0).astype(np.float32)
    if extra_weights is not None:
        weights = weights * np.asarray(extra_weights, dtype=np.float32)
    return _sample_weights_from_array(weights)


def _format_metrics(logs: dict | None) -> str:
    if not logs:
        return "no metrics"
    keys = [
        "loss",
        "fusion_score_loss",
        "fusion_score_pr_auc",
        "fusion_score_recall",
        "val_loss",
        "val_fusion_score_loss",
        "val_fusion_score_pr_auc",
        "val_fusion_score_recall",
    ]
    parts = []
    for key in keys:
        value = logs.get(key)
        if value is not None:
            parts.append(f"{key}={float(value):.4f}")
    return " | ".join(parts) if parts else "no metrics"


class ProgressLogger(keras.callbacks.Callback):
    def __init__(self) -> None:
        super().__init__()
        self.total_epochs = "?"

    def on_train_begin(self, logs=None):
        self.total_epochs = self.params.get("epochs", "?")
        print(f"[train] started | epochs={self.total_epochs}")

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_started = time.time()
        print(f"[epoch {epoch + 1}/{self.total_epochs}] started")

    def on_epoch_end(self, epoch, logs=None):
        duration = time.time() - self.epoch_started
        print(f"[epoch {epoch + 1}/{self.total_epochs}] finished | duration={duration:.1f}s | {_format_metrics(logs)}")

    def on_train_end(self, logs=None):
        print("[train] finished")


def _binary_focal_loss(gamma: float = 2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-6, 1.0 - 1e-6)
        pt = tf.where(tf.equal(y_true, 1.0), y_pred, 1.0 - y_pred)
        ce = keras.backend.binary_crossentropy(y_true, y_pred)
        return tf.pow(1.0 - pt, gamma) * ce

    return loss


def _build_binary_loss(loss_name: str, focal_gamma: float) -> keras.losses.Loss | Any:
    requested = (loss_name or "bce").strip().lower()
    if requested in {"bce", "binary_crossentropy", "weighted_bce"}:
        return keras.losses.BinaryCrossentropy()
    if requested == "focal":
        return _binary_focal_loss(focal_gamma)
    raise ValueError(f"Unsupported loss: {loss_name}")


def train_hybrid_prefilter(
    dataset_name: str,
    *,
    model_name: str = DEFAULT_PREFILTER_MODEL_NAME,
    semantic_model_name: str = DEFAULT_RETRIEVAL_MODEL_REPO_ID,
    token_max_tokens: int = 32000,
    token_sequence_length: int = 384,
    token_embedding_dim: int = 96,
    token_filters: int = 96,
    ast_max_tokens: int = 8000,
    ast_sequence_length: int = 196,
    ast_embedding_dim: int = 64,
    ast_filters: int = 64,
    projection_dim: int = 192,
    dense_units: int = 192,
    dropout_rate: float = 0.25,
    batch_size: int = 128,
    epochs: int = 10,
    learning_rate: float = 7e-4,
    random_seed: int = 42,
    log_progress: bool = True,
    loss_name: str = "bce",
    focal_gamma: float = 2.0,
    hard_negative_mining: bool = False,
    hard_negative_quantile: float = 0.85,
    hard_negative_weight: float = 2.5,
    hard_negative_epochs: int = 2,
) -> dict:
    tf.keras.utils.set_random_seed(random_seed)
    train_payload = load_feature_store(dataset_name, "train")
    val_payload = load_feature_store(dataset_name, "val")

    semantic_dim = int(train_payload["semantic_embeddings"].shape[1])
    numeric_dim = int(train_payload["numeric_features"].shape[1])
    numeric_mean = train_payload["numeric_features"].mean(axis=0).astype(np.float32)
    numeric_std = train_payload["numeric_features"].std(axis=0).astype(np.float32)
    numeric_std = np.where(numeric_std < 1e-6, 1.0, numeric_std)

    model = _build_prefilter_model(
        token_max_tokens=token_max_tokens,
        token_sequence_length=token_sequence_length,
        token_embedding_dim=token_embedding_dim,
        token_filters=token_filters,
        ast_max_tokens=ast_max_tokens,
        ast_sequence_length=ast_sequence_length,
        ast_embedding_dim=ast_embedding_dim,
        ast_filters=ast_filters,
        semantic_dim=semantic_dim,
        numeric_dim=numeric_dim,
        projection_dim=projection_dim,
        dense_units=dense_units,
        dropout_rate=dropout_rate,
    )
    model.token_vectorizer.adapt(tf.data.Dataset.from_tensor_slices(train_payload["token_texts"]).batch(batch_size))
    model.ast_vectorizer.adapt(tf.data.Dataset.from_tensor_slices(train_payload["ast_texts"]).batch(batch_size))

    train_inputs = _prepare_scaled_inputs(train_payload, numeric_mean, numeric_std)
    val_inputs = _prepare_scaled_inputs(val_payload, numeric_mean, numeric_std)
    train_labels = np.asarray(train_payload["labels"], dtype=np.int32)
    val_labels = np.asarray(val_payload["labels"], dtype=np.int32)
    positive_weight = max(1.0, float(np.sum(train_labels == 0) / max(np.sum(train_labels == 1), 1)))

    if log_progress:
        print(
            f"[train] dataset={dataset_name} | train={len(train_labels)} | val={len(val_labels)} "
            f"| positive_weight={positive_weight:.4f} | semantic_model={semantic_model_name}"
        )

    binary_loss = _build_binary_loss(loss_name, focal_gamma)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss={
            "fusion_score": binary_loss,
            "semantic_score": binary_loss,
            "graph_score": binary_loss,
        },
        loss_weights={"fusion_score": 1.0, "semantic_score": 0.25, "graph_score": 0.25},
        metrics={
            "fusion_score": [
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.Precision(name="precision"),
                keras.metrics.Recall(name="recall"),
                keras.metrics.AUC(name="roc_auc"),
                keras.metrics.AUC(name="pr_auc", curve="PR"),
            ]
        },
    )

    callbacks: list[keras.callbacks.Callback] = [
        keras.callbacks.EarlyStopping(monitor="val_fusion_score_pr_auc", mode="max", patience=2, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_fusion_score_pr_auc", mode="max", factor=0.5, patience=1, min_lr=1e-5),
    ]
    if log_progress:
        callbacks.append(ProgressLogger())

    history = model.fit(
        train_inputs,
        _targets(train_labels),
        validation_data=(val_inputs, _targets(val_labels), _sample_weights(val_labels, 1.0)),
        sample_weight=_sample_weights(train_labels, positive_weight),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0,
    )

    hard_negative_summary = {
        "enabled": bool(hard_negative_mining),
        "applied": False,
        "quantile": float(hard_negative_quantile),
        "weight": float(hard_negative_weight),
        "epochs": int(hard_negative_epochs),
        "cutoff": None,
        "selected": 0,
    }
    if hard_negative_mining:
        train_outputs = model.predict(train_inputs, batch_size=batch_size, verbose=0)
        train_scores = np.asarray(train_outputs["fusion_score"], dtype=np.float32).reshape(-1)
        negative_scores = train_scores[train_labels == 0]
        if len(negative_scores) > 0:
            cutoff = float(np.quantile(negative_scores, np.clip(hard_negative_quantile, 0.5, 0.99)))
            hard_negative_mask = (train_labels == 0) & (train_scores >= cutoff)
            hard_negative_weights = np.ones_like(train_labels, dtype=np.float32)
            hard_negative_weights[hard_negative_mask] = float(hard_negative_weight)
            hard_negative_summary = {
                "enabled": True,
                "applied": True,
                "quantile": float(hard_negative_quantile),
                "weight": float(hard_negative_weight),
                "epochs": int(hard_negative_epochs),
                "cutoff": float(cutoff),
                "selected": int(np.sum(hard_negative_mask)),
            }
            if log_progress:
                print(
                    f"[train] hard-negative mining | selected={hard_negative_summary['selected']} | "
                    f"cutoff={hard_negative_summary['cutoff']:.4f} | weight={hard_negative_weight:.2f}"
                )
            hard_callbacks: list[keras.callbacks.Callback] = [ProgressLogger()] if log_progress else []
            hard_history = model.fit(
                train_inputs,
                _targets(train_labels),
                validation_data=(val_inputs, _targets(val_labels), _sample_weights(val_labels, 1.0)),
                sample_weight=_sample_weights(train_labels, positive_weight, extra_weights=hard_negative_weights),
                epochs=hard_negative_epochs,
                batch_size=batch_size,
                callbacks=hard_callbacks,
                verbose=0,
            )
            history.history["hard_negative_loss"] = [float(value) for value in hard_history.history.get("loss", [])]

    output_dir = ensure_dir(MODELS_DIR / dataset_name / model_name)
    model.save_weights(output_dir / "weights.weights.h5")
    (output_dir / "token_vocabulary.txt").write_text("\n".join(model.token_vectorizer.get_vocabulary()), encoding="utf-8")
    (output_dir / "ast_vocabulary.txt").write_text("\n".join(model.ast_vectorizer.get_vocabulary()), encoding="utf-8")

    config = {
        "schema_version": PREFILTER_MODEL_SCHEMA_VERSION,
        "architecture": "hybrid_multiview_prefilter",
        "token_max_tokens": token_max_tokens,
        "token_sequence_length": token_sequence_length,
        "token_embedding_dim": token_embedding_dim,
        "token_filters": token_filters,
        "ast_max_tokens": ast_max_tokens,
        "ast_sequence_length": ast_sequence_length,
        "ast_embedding_dim": ast_embedding_dim,
        "ast_filters": ast_filters,
        "projection_dim": projection_dim,
        "dense_units": dense_units,
        "dropout_rate": dropout_rate,
        "semantic_dim": semantic_dim,
        "numeric_dim": numeric_dim,
        "numeric_feature_names": NUMERIC_FEATURE_NAMES,
        "numeric_mean": numeric_mean.tolist(),
        "numeric_std": numeric_std.tolist(),
        "semantic_model_name": semantic_model_name,
        "loss_name": loss_name,
        "focal_gamma": float(focal_gamma),
        "hard_negative_mining": bool(hard_negative_mining),
        "hard_negative_quantile": float(hard_negative_quantile),
        "hard_negative_weight": float(hard_negative_weight),
        "hard_negative_epochs": int(hard_negative_epochs),
    }
    dump_json(output_dir / "config.json", config)

    summary = {
        "dataset": dataset_name,
        "model_name": model_name,
        "model_path": str(output_dir),
        "train_size": int(len(train_labels)),
        "val_size": int(len(val_labels)),
        "positive_class_weight": float(positive_weight),
        "best_val_pr_auc": float(max(history.history.get("val_fusion_score_pr_auc", [0.0]))),
        "best_val_recall": float(max(history.history.get("val_fusion_score_recall", [0.0]))),
        "history": {key: [float(value) for value in values] for key, values in history.history.items()},
        "feature_names": list(NUMERIC_FEATURE_NAMES),
        "semantic_model_name": semantic_model_name,
        "loss_name": loss_name,
        "focal_gamma": float(focal_gamma),
        "hard_negative_mining": bool(hard_negative_mining),
        "hard_negative_quantile": float(hard_negative_quantile),
        "hard_negative_weight": float(hard_negative_weight),
        "hard_negative_epochs": int(hard_negative_epochs),
        "hard_negative_summary": hard_negative_summary,
    }
    dump_json(MODELS_DIR / dataset_name / f"training_summary.{model_name}.json", summary)
    return summary


class HybridPrefilterBundle:
    def __init__(self, artifact_dir: Path) -> None:
        self.artifact_dir = Path(artifact_dir)
        config_path = self.artifact_dir / "config.json"
        if not config_path.exists():
            raise FileNotFoundError(f"Missing prefilter config: {config_path}")
        self.config = _load_json(config_path)
        self.model = _build_prefilter_model(
            token_max_tokens=int(self.config["token_max_tokens"]),
            token_sequence_length=int(self.config["token_sequence_length"]),
            token_embedding_dim=int(self.config["token_embedding_dim"]),
            token_filters=int(self.config["token_filters"]),
            ast_max_tokens=int(self.config["ast_max_tokens"]),
            ast_sequence_length=int(self.config["ast_sequence_length"]),
            ast_embedding_dim=int(self.config["ast_embedding_dim"]),
            ast_filters=int(self.config["ast_filters"]),
            semantic_dim=int(self.config["semantic_dim"]),
            numeric_dim=int(self.config["numeric_dim"]),
            projection_dim=int(self.config["projection_dim"]),
            dense_units=int(self.config["dense_units"]),
            dropout_rate=float(self.config["dropout_rate"]),
        )
        token_vocabulary = (self.artifact_dir / "token_vocabulary.txt").read_text(encoding="utf-8").splitlines()
        ast_vocabulary = (self.artifact_dir / "ast_vocabulary.txt").read_text(encoding="utf-8").splitlines()
        self.model.token_vectorizer.set_vocabulary(token_vocabulary)
        self.model.ast_vectorizer.set_vocabulary(ast_vocabulary)
        self.model.load_weights(self.artifact_dir / "weights.weights.h5")
        self.numeric_mean = np.asarray(self.config["numeric_mean"], dtype=np.float32)
        self.numeric_std = np.asarray(self.config["numeric_std"], dtype=np.float32)

    def predict_payload(self, payload: dict, batch_size: int = 128) -> dict[str, np.ndarray]:
        inputs = _prepare_scaled_inputs(payload, self.numeric_mean, self.numeric_std)
        outputs = self.model.predict(inputs, batch_size=batch_size, verbose=0)
        return {
            "fusion_score": outputs["fusion_score"].reshape(-1),
            "semantic_score": outputs["semantic_score"].reshape(-1),
            "graph_score": outputs["graph_score"].reshape(-1),
        }


def load_hybrid_prefilter_bundle(dataset_name: str, model_name: str = DEFAULT_PREFILTER_MODEL_NAME) -> HybridPrefilterBundle:
    return HybridPrefilterBundle(MODELS_DIR / dataset_name / model_name)


def predict_feature_store(
    dataset_name: str,
    split_name: str,
    *,
    model_name: str = DEFAULT_PREFILTER_MODEL_NAME,
    batch_size: int = 128,
) -> dict:
    bundle = load_hybrid_prefilter_bundle(dataset_name, model_name=model_name)
    payload = load_feature_store(dataset_name, split_name)
    predictions = bundle.predict_payload(payload, batch_size=batch_size)
    return {
        "record_ids": payload["record_ids"],
        "labels": payload["labels"],
        **predictions,
    }


def build_single_record_feature_payload(
    record: dict,
    *,
    semantic_encoder: SemanticRetrievalEncoder,
    graph_backend: str = "auto",
) -> dict:
    code = get_record_code(record)
    graph_features = get_graph_features({**record, "code": code}, graph_backend=graph_backend)
    payload = {
        "record_ids": [str(record.get("record_id", "record-0"))],
        "labels": np.asarray([int(record.get("label", 0))], dtype=np.int32),
        "token_texts": [" ".join(tokenize_code(code))],
        "ast_texts": [graph_features.get("ast_sequence") or build_skeleton(code)],
        "numeric_features": np.asarray([_compute_numeric_features(code, graph_features)], dtype=np.float32),
        "semantic_embeddings": np.asarray(semantic_encoder.encode_texts([code]), dtype=np.float32),
    }
    return payload


def _load_json(path: Path) -> dict:
    import json

    return json.loads(path.read_text(encoding="utf-8"))


__all__ = [
    "DEFAULT_PREFILTER_MODEL_NAME",
    "FEATURE_STORE_SCHEMA_VERSION",
    "NUMERIC_FEATURE_NAMES",
    "HybridPrefilterBundle",
    "build_feature_store",
    "build_single_record_feature_payload",
    "feature_store_path",
    "load_feature_store",
    "load_hybrid_prefilter_bundle",
    "predict_feature_store",
    "train_hybrid_prefilter",
]


### File `local_llm_client.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/local_llm_client.py
import json
import os
import re
import time
from pathlib import Path
from typing import Any

from common import CACHE_DIR, SHARED_MODELS_DIR, build_structure_summary, ensure_dir, load_json, stable_hash, truncate_text


EVIDENCE_SCHEMA_ENABLED = os.getenv("GRACE_EVIDENCE_AWARE_VERIFIER", "0").strip().lower() in {"1", "true", "yes", "on"}
CACHE_SCHEMA_VERSION = 2 if EVIDENCE_SCHEMA_ENABLED else 1
EXPECTED_RESPONSE_KEYS = {"label", "confidence", "reason"}
DEFAULT_MODEL_REPO_ID = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
DEFAULT_REASON_WORD_LIMIT = 48
DEFAULT_PROMPT_CODE_CHAR_LIMIT = int(os.getenv("GRACE_PROMPT_CODE_CHAR_LIMIT", "1800"))
DEFAULT_PROMPT_TOP_LINES_LIMIT = int(os.getenv("GRACE_PROMPT_TOP_LINES_LIMIT", "3"))
DEFAULT_PROMPT_TOP_LINE_CHAR_LIMIT = int(os.getenv("GRACE_PROMPT_TOP_LINE_CHAR_LIMIT", "120"))
DEFAULT_PROMPT_SLICES_CHAR_LIMIT = int(os.getenv("GRACE_PROMPT_SLICES_CHAR_LIMIT", "900"))
DEFAULT_PROMPT_NODE_INFO_CHAR_LIMIT = int(os.getenv("GRACE_PROMPT_NODE_INFO_CHAR_LIMIT", "900"))
DEFAULT_PROMPT_EDGE_INFO_CHAR_LIMIT = int(os.getenv("GRACE_PROMPT_EDGE_INFO_CHAR_LIMIT", "900"))
if EVIDENCE_SCHEMA_ENABLED:
    CHAT_SYSTEM_INSTRUCTION = (
        "You are a software vulnerability classifier. "
        "Do not output chain-of-thought, step-by-step reasoning, markdown, or any preamble. "
        'Your entire visible answer must be exactly one line that starts with FINAL_JSON: '
        "followed by a JSON object with keys label, confidence, reason, cwe_family, vulnerable_lines, sink_or_api, missing_guard."
    )
else:
    CHAT_SYSTEM_INSTRUCTION = (
        "You are a software vulnerability classifier. "
        "Do not output chain-of-thought, step-by-step reasoning, markdown, or any preamble. "
        'Your entire visible answer must be exactly one line that starts with FINAL_JSON: '
        "followed by a JSON object with keys label, confidence, reason."
    )


def default_local_model_dir(repo_id: str = DEFAULT_MODEL_REPO_ID) -> Path:
    return SHARED_MODELS_DIR / "local_llm" / repo_id.replace("/", "--")


def resolve_hf_token() -> str | None:
    for env_name in ["HUGGINGFACE_HUB_TOKEN", "HF_TOKEN"]:
        value = os.getenv(env_name)
        if value:
            return value.strip().strip('"').strip("'")
    return None


def is_model_downloaded(model_dir: Path) -> bool:
    has_weights = any(model_dir.glob("*.safetensors")) or any(model_dir.glob("pytorch_model*.bin"))
    return (model_dir / "config.json").exists() and has_weights


def download_model_snapshot(repo_id: str, local_dir: Path | None = None) -> Path:
    try:
        from huggingface_hub import snapshot_download
    except Exception as exc:
        raise RuntimeError(
            "Missing dependency `huggingface_hub`. Install it before downloading the local model."
        ) from exc

    target_dir = Path(local_dir or default_local_model_dir(repo_id))
    ensure_dir(target_dir)
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(target_dir),
        token=resolve_hf_token(),
        allow_patterns=[
            "*.json",
            "*.safetensors",
            "pytorch_model*.bin",
            "*.txt",
            "*.model",
            "tokenizer*",
            "merges.txt",
            "vocab.*",
        ],
    )
    return target_dir


class LocalVulnLLMClassifier:
    def __init__(
        self,
        model_name: str = DEFAULT_MODEL_REPO_ID,
        model_dir: Path | None = None,
        temperature: float = 0.0,
        max_new_tokens: int = 128,
        cache_path: Path | None = None,
        load_in_4bit: bool = True,
        device_map: str = "auto",
        auto_download: bool = False,
    ) -> None:
        self.model_name = model_name
        self.model_dir = Path(model_dir or default_local_model_dir(model_name))
        self.temperature = temperature
        self.max_new_tokens = max_new_tokens
        self.load_in_4bit = load_in_4bit
        self.device_map = device_map
        self.auto_download = auto_download
        self.cache_path = cache_path or (CACHE_DIR / "local_vulnllm_cache.json")
        ensure_dir(self.cache_path.parent)
        self.cache = load_json(self.cache_path, default={})
        self._runtime: dict[str, Any] | None = None
        self._model = None
        self._tokenizer = None

    def _save_cache(self) -> None:
        self.cache_path.write_text(json.dumps(self.cache, ensure_ascii=False, indent=2), encoding="utf-8")

    def _cache_key(self, prompt: str) -> str:
        return stable_hash(
            "\n".join(
                [
                    self.model_name,
                    str(self.model_dir),
                    f"load_in_4bit={self.load_in_4bit}",
                    f"max_new_tokens={self.max_new_tokens}",
                    f"temperature={self.temperature}",
                    CHAT_SYSTEM_INSTRUCTION,
                    prompt,
                ]
            )
        )

    def prompt_hash(self, prompt: str) -> str:
        return self._cache_key(prompt)

    def _invalidate_cache_key(self, cache_key: str) -> None:
        if cache_key in self.cache:
            self.cache.pop(cache_key, None)
            self._save_cache()

    def _get_cached_entry(self, prompt: str) -> dict | None:
        cache_key = self._cache_key(prompt)
        entry = self.cache.get(cache_key)
        if entry is None:
            return None
        try:
            normalized = _normalize_cache_entry(entry)
        except Exception:
            self._invalidate_cache_key(cache_key)
            return None
        if normalized != entry:
            self.cache[cache_key] = normalized
            self._save_cache()
        return normalized

    def is_cached(self, prompt: str) -> bool:
        return self._get_cached_entry(prompt) is not None

    def prepare(self) -> dict[str, Any]:
        self._ensure_loaded()
        return {
            "model_name": self.model_name,
            "model_dir": str(self.model_dir),
            "load_in_4bit": self.load_in_4bit,
            "device": str(self._model_input_device()),
        }

    def classify(self, prompt: str) -> dict:
        cached_entry = self._get_cached_entry(prompt)
        if cached_entry is not None:
            result = dict(cached_entry["parsed"])
            result["raw_text"] = cached_entry["raw_text"]
            result["cached"] = True
            result["finish_reason"] = cached_entry.get("finish_reason")
            result["usage_metadata"] = cached_entry.get("usage_metadata", {})
            result["device"] = cached_entry.get("device")
            return result

        cache_key = self._cache_key(prompt)
        response_payload = None
        try:
            response_payload = self._generate_response_payload(prompt)
            parsed = _parse_detection_payload(response_payload.get("raw_text", ""))
            cache_entry = {
                "schema_version": CACHE_SCHEMA_VERSION,
                "model_name": self.model_name,
                "raw_text": response_payload.get("raw_text", ""),
                "parsed": parsed,
                "finish_reason": response_payload.get("finish_reason"),
                "usage_metadata": response_payload.get("usage_metadata", {}),
                "device": response_payload.get("device"),
                "cached_at_unix": time.time(),
            }
            self.cache[cache_key] = cache_entry
            self._save_cache()
            result = dict(parsed)
            result["raw_text"] = cache_entry["raw_text"]
            result["cached"] = False
            result["finish_reason"] = cache_entry["finish_reason"]
            result["usage_metadata"] = cache_entry["usage_metadata"]
            result["device"] = cache_entry["device"]
            return result
        except Exception as exc:
            self._invalidate_cache_key(cache_key)
            if response_payload is not None:
                finish_reason = response_payload.get("finish_reason")
                raw_preview = (response_payload.get("raw_text", "") or "")[:200]
                raise RuntimeError(
                    f"{exc} | finish_reason={finish_reason} | raw_text_preview={raw_preview!r}"
                ) from exc
            raise

    def _ensure_loaded(self) -> None:
        if self._model is not None and self._tokenizer is not None:
            return
        if self.auto_download and not is_model_downloaded(self.model_dir):
            download_model_snapshot(self.model_name, self.model_dir)
        if not is_model_downloaded(self.model_dir):
            raise FileNotFoundError(
                f"Local model directory is missing or incomplete: {self.model_dir}. "
                "Run `python GRACE-improve/baseline/baseline2/00_verify_assets.py` first."
            )

        runtime = self._load_runtime()
        torch = runtime["torch"]
        AutoModelForCausalLM = runtime["AutoModelForCausalLM"]
        AutoTokenizer = runtime["AutoTokenizer"]
        BitsAndBytesConfig = runtime["BitsAndBytesConfig"]

        tokenizer = AutoTokenizer.from_pretrained(self.model_dir, local_files_only=True, trust_remote_code=False)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        config_payload = {}
        config_path = self.model_dir / "config.json"
        if config_path.exists():
            try:
                config_payload = json.loads(config_path.read_text(encoding="utf-8"))
            except Exception:
                config_payload = {}
        has_prequantized_config = bool(config_payload.get("quantization_config")) or "bnb-4bit" in self.model_name.lower()

        model_kwargs: dict[str, Any] = {
            "device_map": self.device_map,
            "low_cpu_mem_usage": True,
            "local_files_only": True,
            "trust_remote_code": False,
        }
        if self.load_in_4bit and not has_prequantized_config:
            model_kwargs["dtype"] = torch.float16
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
        else:
            model_kwargs["dtype"] = torch.float16

        model = AutoModelForCausalLM.from_pretrained(self.model_dir, **model_kwargs)
        model.eval()

        self._runtime = runtime
        self._tokenizer = tokenizer
        self._model = model

    def _load_runtime(self) -> dict[str, Any]:
        if self._runtime is not None:
            return self._runtime
        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        except Exception as exc:
            raise RuntimeError(
                "Missing local LLM dependencies. Install `torch`, `transformers`, `accelerate`, "
                "`bitsandbytes`, and `huggingface_hub` before running script 07."
            ) from exc
        self._runtime = {
            "torch": torch,
            "AutoModelForCausalLM": AutoModelForCausalLM,
            "AutoTokenizer": AutoTokenizer,
            "BitsAndBytesConfig": BitsAndBytesConfig,
        }
        return self._runtime

    def _model_input_device(self):
        if hasattr(self._model, "device") and self._model.device is not None:
            return self._model.device
        for parameter in self._model.parameters():
            return parameter.device
        raise RuntimeError("Could not determine local model device.")

    def _generate_response_payload(self, prompt: str) -> dict:
        self._ensure_loaded()
        tokenizer = self._tokenizer
        model = self._model
        torch = self._runtime["torch"]

        messages = [
            {
                "role": "system",
                "content": CHAT_SYSTEM_INSTRUCTION,
            },
            {"role": "user", "content": prompt},
        ]
        if hasattr(tokenizer, "apply_chat_template"):
            rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            rendered = prompt

        model_inputs = tokenizer([rendered], return_tensors="pt")
        input_device = self._model_input_device()
        model_inputs = {name: tensor.to(input_device) for name, tensor in model_inputs.items()}
        prompt_tokens = int(model_inputs["input_ids"].shape[-1])

        generation_kwargs = {
            "max_new_tokens": self.max_new_tokens,
            "do_sample": self.temperature > 0,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }
        if self.temperature > 0:
            generation_kwargs["temperature"] = self.temperature

        with torch.inference_mode():
            generated = model.generate(**model_inputs, **generation_kwargs)

        new_tokens = generated[:, model_inputs["input_ids"].shape[-1] :]
        raw_text = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
        generated_tokens = int(new_tokens.shape[-1])
        finish_reason = "length" if generated_tokens >= self.max_new_tokens else "stop"
        return {
            "raw_text": raw_text,
            "finish_reason": finish_reason,
            "usage_metadata": {
                "prompt_tokens": prompt_tokens,
                "generated_tokens": generated_tokens,
            },
            "device": str(input_device),
        }


def build_detection_prompt(
    record: dict,
    retrieved_examples: list[dict],
    calibrated_probability: float,
    risk_band: str,
    graph_features: dict | None = None,
    suspicious_context: dict | None = None,
    semantic_score: float | None = None,
    graph_score: float | None = None,
    fusion_score: float | None = None,
) -> str:
    graph_features = graph_features or {}
    suspicious_context = suspicious_context or {}
    node_info = graph_features.get("node_info") or build_structure_summary(record["code"])
    edge_info = graph_features.get("edge_info") or "Node1\tNode2\tEdgeType\nn/a\tn/a\tSTRUCTURE_SUMMARY_ONLY"
    node_info = truncate_text(node_info, DEFAULT_PROMPT_NODE_INFO_CHAR_LIMIT)
    edge_info = truncate_text(edge_info, DEFAULT_PROMPT_EDGE_INFO_CHAR_LIMIT)
    graph_backend = graph_features.get("backend", "summary")
    suspicious_slices_text = suspicious_context.get("slices_text") or "No suspicious slices."
    suspicious_slices_text = truncate_text(suspicious_slices_text, DEFAULT_PROMPT_SLICES_CHAR_LIMIT)
    top_lines = suspicious_context.get("top_lines") or []
    top_lines_text = "\n".join(
        [
            f"line {row['line_number']}: score={row['score']:.4f} | reasons={','.join(row.get('reasons', [])) or 'n/a'} | {truncate_text(row['code'], DEFAULT_PROMPT_TOP_LINE_CHAR_LIMIT)}"
            for row in top_lines[:DEFAULT_PROMPT_TOP_LINES_LIMIT]
        ]
    )
    token_highlights = ", ".join(suspicious_context.get("token_highlights") or []) or "none"
    example_blocks = []
    for index, example in enumerate(retrieved_examples, start=1):
        label_text = "Vulnerable" if int(example["label"]) == 1 else "Non-vulnerable"
        example_blocks.append(
            "\n".join(
                [
                    f"Reference Example {index}",
                    f"Label: {label_text}",
                    f"Project: {example.get('project', '') or 'unknown'}",
                    "```c",
                    example["code"],
                    "```",
                ]
            )
        )
    examples_text = "\n\n".join(example_blocks) if example_blocks else "No demonstrations."
    return "\n".join(
        [
            "You are auditing one C/C++ function for security vulnerabilities.",
            "Use concrete code evidence only.",
            "Be recall-oriented on real bug patterns, but do not invent vulnerabilities without supporting code evidence.",
            "Focus on memory safety, bounds checks, pointer misuse, lifetime bugs, unsafe APIs, integer overflow, race-prone state changes, and auth or validation flaws.",
            "The demonstrations are similar functions for in-context learning only. Do not copy their labels blindly.",
            f"Prefilter risk band: {risk_band}",
            f"Prefilter calibrated vulnerability probability: {calibrated_probability:.4f}",
            f"Fusion prefilter score: {float(fusion_score or calibrated_probability):.4f}",
            f"Semantic branch score: {float(semantic_score or 0.0):.4f}",
            f"Graph branch score: {float(graph_score or 0.0):.4f}",
            f"Graph backend used for structure extraction: {graph_backend}",
            "",
            "Code snippet:",
            "```c",
            truncate_text(record["code"], DEFAULT_PROMPT_CODE_CHAR_LIMIT),
            "```",
            "",
            "Suspicious lines ranked by the localizer:",
            top_lines_text or "No suspicious lines.",
            "",
            "Suspicious slices with short context:",
            suspicious_slices_text,
            "",
            f"Highlighted local tokens: {token_highlights}",
            "",
            "Use the suspicious slices as guidance, but verify against the full function before deciding.",
            "In the above code snippet, check for potential security vulnerabilities and output either 'Vulnerable' or 'Non-vulnerable'.",
            "The node information of the function is as follows:",
            node_info,
            "",
            "The edge information of the function is as follows:",
            edge_info,
            "",
            "The following are demonstrations retrieved from similar functions:",
            examples_text,
            "",
            "Do not output step-by-step reasoning or any extra commentary.",
            *(
                [
                    "Return exactly one line in this format:",
                    'FINAL_JSON: {"label":"Vulnerable"|"Non-vulnerable","confidence":0.0-1.0,"cwe_family":"CWE-xxx or empty","vulnerable_lines":["start-end"],"sink_or_api":"short name","missing_guard":"short phrase","reason":"short justification"}',
                    "If you decide Vulnerable, fill vulnerable_lines and sink_or_api with concrete evidence; otherwise keep them empty.",
                ]
                if EVIDENCE_SCHEMA_ENABLED
                else [
                    "Return exactly one line in this format:",
                    'FINAL_JSON: {"label":"Vulnerable"|"Non-vulnerable","confidence":0.0-1.0,"reason":"short justification"}',
                ]
            ),
            f"Keep `reason` under {DEFAULT_REASON_WORD_LIMIT} words.",
        ]
    )


def parse_detection_response(text: str) -> dict:
    return _parse_detection_payload(text)


def _parse_detection_payload(raw_text: str) -> dict:
    payload = _extract_json_payload(raw_text)
    if payload is not None:
        return _normalize_response_payload(payload)

    label = _extract_label_fallback(raw_text)
    if label is None:
        raise ValueError("Could not parse a vulnerability label from the local LLM output.")
    confidence = _extract_confidence_fallback(raw_text)
    reason = _extract_reason_fallback(raw_text)
    return _normalize_response_payload(
        {
            "label": label,
            "confidence": confidence,
            "reason": reason,
        }
    )


def _extract_json_payload(raw_text: str) -> dict | None:
    stripped = (raw_text or "").strip()
    if not stripped:
        return None

    candidates = []
    final_json_match = re.search(r"FINAL_JSON:\s*(\{.*\})", stripped, flags=re.IGNORECASE | re.DOTALL)
    if final_json_match:
        candidates.append(final_json_match.group(1).strip())
    candidates.append(stripped)

    for candidate in candidates:
        for value in _iter_json_candidates(candidate):
            if isinstance(value, dict) and EXPECTED_RESPONSE_KEYS.issubset(value.keys()):
                return value
    return None


def _iter_json_candidates(text: str):
    decoder = json.JSONDecoder()
    try:
        yield json.loads(text)
    except Exception:
        pass

    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text[match.start() :])
        except json.JSONDecodeError:
            continue
        yield value


def _normalize_line_spans(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        parts = [part.strip() for part in re.split(r"[;,]", text) if part.strip()]
        return parts or [text]
    if isinstance(value, (list, tuple, set)):
        items = []
        for item in value:
            text = str(item).strip()
            if text:
                items.append(text)
        return items
    text = str(value).strip()
    return [text] if text else []


def _normalize_response_payload(payload: dict) -> dict:
    label_text = str(payload.get("label", "")).strip()
    normalized = label_text.lower().replace("_", "-")
    if normalized in {"vulnerable", "1"}:
        label_int = 1
        canonical_label = "Vulnerable"
    elif normalized in {"non-vulnerable", "not vulnerable", "non vulnerable", "safe", "benign", "0"}:
        label_int = 0
        canonical_label = "Non-vulnerable"
    else:
        raise ValueError(f"Invalid label in local LLM response: {label_text!r}")

    try:
        confidence = float(payload.get("confidence"))
    except Exception as exc:
        raise ValueError(f"Invalid confidence in local LLM response: {payload.get('confidence')!r}") from exc
    confidence = max(0.0, min(1.0, confidence))

    reason = str(payload.get("reason") or payload.get("brief_reason") or "").strip()
    if not reason:
        raise ValueError("Local LLM response is missing a non-empty reason.")

    return {
        "label": canonical_label,
        "label_int": label_int,
        "confidence": confidence,
        "reason": reason,
        "cwe_family": str(payload.get("cwe_family", "")).strip(),
        "vulnerable_lines": _normalize_line_spans(payload.get("vulnerable_lines")),
        "sink_or_api": str(payload.get("sink_or_api", "")).strip(),
        "missing_guard": str(payload.get("missing_guard", "")).strip(),
    }


def _extract_label_fallback(raw_text: str) -> str | None:
    stripped = (raw_text or "").strip()
    if not stripped:
        return None
    patterns = [
        r"FINAL_LABEL\s*[:=]\s*(VULNERABLE|NON-VULNERABLE|NON VULNERABLE|SAFE|BENIGN)",
        r"final answer\s*[:=]\s*(vulnerable|non-vulnerable|non vulnerable|safe|benign)",
        r"verdict\s*[:=]\s*(vulnerable|non-vulnerable|non vulnerable|safe|benign)",
    ]
    for pattern in patterns:
        match = re.search(pattern, stripped, flags=re.IGNORECASE)
        if match:
            return match.group(1)
    tail = "\n".join(stripped.splitlines()[-8:])
    if re.search(r"\bnon[- ]vulnerable\b|\bsafe\b|\bbenign\b", tail, flags=re.IGNORECASE):
        return "Non-vulnerable"
    if re.search(r"\bvulnerable\b", tail, flags=re.IGNORECASE):
        return "Vulnerable"
    return None


def _extract_confidence_fallback(raw_text: str) -> float:
    match = re.search(r"confidence\s*[:=]\s*(0(?:\.\d+)?|1(?:\.0+)?)", raw_text or "", flags=re.IGNORECASE)
    if match:
        return float(match.group(1))
    return 0.5


def _extract_reason_fallback(raw_text: str) -> str:
    stripped = (raw_text or "").strip()
    if not stripped:
        return "fallback_parse_empty_output"
    reason_match = re.search(r"reason\s*[:=]\s*(.+)", stripped, flags=re.IGNORECASE)
    if reason_match:
        return truncate_text(reason_match.group(1).strip(), 280) or "fallback_parse_reason_line"
    final_json_prefix = re.sub(r"FINAL_JSON:.*", "", stripped, flags=re.IGNORECASE | re.DOTALL).strip()
    reason_source = final_json_prefix or stripped
    reason_source = re.sub(r"\s+", " ", reason_source).strip()
    return truncate_text(reason_source, 280) or "fallback_parse_short_output"


def _normalize_cache_entry(entry: Any) -> dict:
    if isinstance(entry, str):
        parsed = parse_detection_response(entry)
        return {
            "schema_version": CACHE_SCHEMA_VERSION,
            "model_name": None,
            "raw_text": entry,
            "parsed": parsed,
            "finish_reason": None,
            "usage_metadata": {},
            "device": None,
            "cached_at_unix": None,
        }
    if not isinstance(entry, dict):
        raise ValueError(f"Unsupported cache entry type: {type(entry).__name__}")
    parsed = _parse_detection_payload(entry.get("raw_text", ""))
    return {
        "schema_version": CACHE_SCHEMA_VERSION,
        "model_name": entry.get("model_name"),
        "raw_text": entry.get("raw_text", ""),
        "parsed": parsed,
        "finish_reason": entry.get("finish_reason"),
        "usage_metadata": entry.get("usage_metadata", {}),
        "device": entry.get("device"),
        "cached_at_unix": entry.get("cached_at_unix"),
    }


### File `evaluate_predictions.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/evaluate_predictions.py
import json
from pathlib import Path
from typing import Any

from common import METRICS_DIR, PREDICTIONS_DIR, dump_json
from metrics import bootstrap_f1_interval, compute_binary_metrics, mcnemar_exact


DEFAULT_DATASET_NAME = "devign"
DEFAULT_PREDICTIONS_FILENAME = "grace_hybrid_predictions.jsonl"
DEFAULT_RUN_STATE_FILENAME = "grace_hybrid_run_state.json"
EXPECTED_SCHEMA_VERSION = 1


def default_prediction_paths(dataset_name: str = DEFAULT_DATASET_NAME) -> tuple[Path, Path]:
    dataset_dir = PREDICTIONS_DIR / dataset_name
    return (
        dataset_dir / DEFAULT_PREDICTIONS_FILENAME,
        dataset_dir / DEFAULT_RUN_STATE_FILENAME,
    )


def load_predictions(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def _count_by_field(rows: list[dict[str, Any]], field: str) -> dict[str, int]:
    counts: dict[str, int] = {}
    for row in rows:
        key = str(row.get(field) or "unknown")
        counts[key] = counts.get(key, 0) + 1
    return counts


def validate_predictions(
    rows: list[dict[str, Any]],
    run_state: dict[str, Any],
    expected_schema_version: int = EXPECTED_SCHEMA_VERSION,
) -> None:
    if run_state.get("schema_version") != expected_schema_version:
        raise RuntimeError(
            f"Prediction schema mismatch. Expected {expected_schema_version}, got {run_state.get('schema_version')}."
        )
    if not run_state.get("complete"):
        raise RuntimeError(
            f"Run is incomplete: resolved_samples={run_state.get('resolved_samples')} / target_samples={run_state.get('target_samples')}."
        )
    if not run_state.get("evaluation_ready"):
        raise RuntimeError(
            "Run is not evaluation-ready. Resolve the issue in run_state and rerun script 07 before evaluating."
        )
    if len(rows) != int(run_state.get("target_samples", -1)):
        raise RuntimeError(
            f"Predictions count mismatch. Found {len(rows)} rows but target_samples={run_state.get('target_samples')}."
        )
    record_ids = set()
    for row in rows:
        if row.get("schema_version") != expected_schema_version:
            raise RuntimeError(f"Found incompatible prediction row for record {row.get('record_id')}.")
        if row.get("resolution_status") != "resolved":
            raise RuntimeError(f"Found unresolved prediction row for record {row.get('record_id')}.")
        record_id = row.get("record_id")
        if record_id in record_ids:
            raise RuntimeError(f"Duplicate prediction row for record {record_id}.")
        record_ids.add(record_id)


def build_evaluation_metrics(
    rows: list[dict[str, Any]],
    run_state: dict[str, Any],
    *,
    dataset_name: str | None = None,
    baseline_compare_path: Path | None = None,
    expected_schema_version: int = EXPECTED_SCHEMA_VERSION,
    bootstrap_iterations: int = 1000,
    run_state_path: Path | None = None,
) -> dict[str, Any]:
    resolved_dataset = dataset_name or run_state.get("dataset") or DEFAULT_DATASET_NAME

    labels = [int(row["ground_truth"]) for row in rows]
    predictions = [int(row["prediction"]) for row in rows]
    probabilities = [float(row["calibrated_probability"]) for row in rows]

    metrics = compute_binary_metrics(labels, predictions, probabilities)
    metrics["dataset"] = resolved_dataset
    metrics["model_name"] = run_state.get("model_name")
    metrics["experiment_mode"] = run_state.get("experiment_mode")
    metrics["schema_version"] = expected_schema_version
    metrics["samples"] = len(rows)
    metrics["llm_calls"] = sum(1 for row in rows if row.get("llm_called"))
    metrics["api_requests_made"] = sum(1 for row in rows if row.get("api_request_made"))
    metrics["llm_cache_hits"] = sum(1 for row in rows if row.get("llm_cache_hit"))
    metrics["llm_call_ratio"] = metrics["llm_calls"] / max(len(rows), 1)
    metrics["routing"] = _count_by_field(rows, "risk_band")
    metrics["decision_sources"] = _count_by_field(rows, "decision_source")
    metrics["retrieval_backend"] = run_state.get("retrieval_backend")
    metrics["graph_backend_requested"] = run_state.get("graph_backend_requested")
    metrics["graph_backend_counts"] = run_state.get("graph_backend_counts")
    metrics["timing_ms"] = run_state.get("timing_ms")
    metrics["run_signature"] = run_state.get("run_signature")
    metrics["bootstrap_f1"] = bootstrap_f1_interval(labels, predictions, iterations=bootstrap_iterations)
    metrics["config"] = run_state.get("config")
    metrics["predictions_path"] = run_state.get("predictions_path")
    if run_state_path is not None:
        metrics["run_state_path"] = str(run_state_path)

    if baseline_compare_path:
        baseline_rows = {row["record_id"]: row for row in load_predictions(baseline_compare_path)}
        aligned = [row for row in rows if row["record_id"] in baseline_rows]
        if aligned:
            base_predictions = [int(baseline_rows[row["record_id"]]["prediction"]) for row in aligned]
            aligned_labels = [int(row["ground_truth"]) for row in aligned]
            aligned_predictions = [int(row["prediction"]) for row in aligned]
            metrics["comparison"] = {
                "mcnemar": mcnemar_exact(aligned_labels, base_predictions, aligned_predictions),
                "baseline_f1": compute_binary_metrics(aligned_labels, base_predictions)["f1"],
                "current_f1": compute_binary_metrics(aligned_labels, aligned_predictions)["f1"],
                "aligned_samples": len(aligned),
                "baseline_compare_path": str(baseline_compare_path),
            }

    return metrics


def evaluate_prediction_artifacts(
    predictions_path: Path,
    run_state_path: Path,
    *,
    dataset_name: str | None = None,
    baseline_compare_path: Path | None = None,
    expected_schema_version: int = EXPECTED_SCHEMA_VERSION,
    bootstrap_iterations: int = 1000,
) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, Any]]:
    if not predictions_path.exists():
        raise FileNotFoundError(f"Missing predictions file: {predictions_path}")
    if not run_state_path.exists():
        raise FileNotFoundError(f"Missing run state file: {run_state_path}")

    rows = load_predictions(predictions_path)
    run_state = load_json(run_state_path)
    validate_predictions(rows, run_state, expected_schema_version=expected_schema_version)
    metrics = build_evaluation_metrics(
        rows,
        run_state,
        dataset_name=dataset_name,
        baseline_compare_path=baseline_compare_path,
        expected_schema_version=expected_schema_version,
        bootstrap_iterations=bootstrap_iterations,
        run_state_path=run_state_path,
    )
    return rows, run_state, metrics


def write_evaluation_summary(
    metrics: dict[str, Any],
    *,
    dataset_name: str | None = None,
    filename: str = "evaluation_summary.json",
) -> Path:
    resolved_dataset = dataset_name or metrics.get("dataset")
    if not resolved_dataset:
        raise ValueError("dataset_name is required when metrics do not include a dataset field.")
    output_path = METRICS_DIR / resolved_dataset / filename
    dump_json(output_path, metrics)
    return output_path


### File `00_verify_assets.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/00_verify_assets.py
import json
import os

from common import SHARED_SPLITS_DIR
from graphs import resolve_graph_backend_with_notice
from local_llm_client import DEFAULT_MODEL_REPO_ID, default_local_model_dir, download_model_snapshot, is_model_downloaded
from retrieval import DEFAULT_RETRIEVAL_MODEL_REPO_ID, default_retrieval_model_dir, download_retrieval_model_snapshot, is_retrieval_model_downloaded


DATASET_NAME = os.getenv("GRACE_DATASET", "devign")
AUTO_DOWNLOAD = os.getenv("GRACE_AUTO_DOWNLOAD_MISSING", "1").strip().lower() in {"1", "true", "yes", "on"}
SEMANTIC_MODEL_ID = os.getenv("GRACE_RETRIEVAL_MODEL_ID", DEFAULT_RETRIEVAL_MODEL_REPO_ID)
LLM_MODEL_ID = os.getenv("GRACE_LOCAL_MODEL_ID", DEFAULT_MODEL_REPO_ID)


def _split_status(dataset_name: str) -> dict:
    split_dir = SHARED_SPLITS_DIR / dataset_name
    files = {name: split_dir / f"{name}.jsonl" for name in ["train", "val", "test"]}
    return {
        "split_dir": str(split_dir),
        "available": all(path.exists() for path in files.values()),
        "files": {name: str(path) for name, path in files.items()},
    }


def main() -> None:
    semantic_dir = default_retrieval_model_dir(SEMANTIC_MODEL_ID)
    llm_dir = default_local_model_dir(LLM_MODEL_ID)
    semantic_ready = is_retrieval_model_downloaded(semantic_dir)
    llm_ready = is_model_downloaded(llm_dir)

    actions = []
    if AUTO_DOWNLOAD and not semantic_ready:
        download_retrieval_model_snapshot(SEMANTIC_MODEL_ID, semantic_dir)
        semantic_ready = is_retrieval_model_downloaded(semantic_dir)
        actions.append(f"downloaded semantic model: {SEMANTIC_MODEL_ID}")
    if AUTO_DOWNLOAD and not llm_ready:
        download_model_snapshot(LLM_MODEL_ID, llm_dir)
        llm_ready = is_model_downloaded(llm_dir)
        actions.append(f"downloaded llm model: {LLM_MODEL_ID}")

    graph_backend, graph_notice = resolve_graph_backend_with_notice("auto")
    payload = {
        "dataset": DATASET_NAME,
        "auto_download_missing": AUTO_DOWNLOAD,
        "actions": actions,
        "splits": _split_status(DATASET_NAME),
        "semantic_model": {
            "repo_id": SEMANTIC_MODEL_ID,
            "path": str(semantic_dir),
            "ready": semantic_ready,
        },
        "local_llm": {
            "repo_id": LLM_MODEL_ID,
            "path": str(llm_dir),
            "ready": llm_ready,
        },
        "graph_backend_auto": graph_backend,
        "graph_notice": graph_notice,
    }
    print(json.dumps(payload, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### File `01_prepare_datasets.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/01_prepare_datasets.py
import csv
import os
from collections import Counter

from common import PROCESSED_DIR, dump_json, ensure_dir
from datasets import discover_reveal_root, get_dataset_iterator


TARGET_DATASETS = [name.strip() for name in os.getenv("GRACE_DATASETS", os.getenv("GRACE_DATASET", "devign")).split(",") if name.strip()]


def prepare_dataset(dataset_name: str) -> None:
    if dataset_name == "reveal" and discover_reveal_root() is None:
        print("Skipping reveal because no raw files were found in data/reveal_raw or similar folders.")
        return
    output_dir = ensure_dir(PROCESSED_DIR / dataset_name)
    stale_records_path = output_dir / "records.jsonl"
    if stale_records_path.exists():
        stale_records_path.unlink()
    index_path = output_dir / "index.csv"
    stats = Counter()
    label_counter = Counter()
    project_counter = Counter()
    iterator = get_dataset_iterator(dataset_name)
    with index_path.open("w", encoding="utf-8", newline="") as index_handle:
        writer = csv.DictWriter(
            index_handle,
            fieldnames=["record_id", "dataset", "project", "label", "commit_id", "cwe_id", "code_hash", "source_path", "split"],
        )
        writer.writeheader()
        for record in iterator():
            writer.writerow(
                {
                    "record_id": record["record_id"],
                    "dataset": record["dataset"],
                    "project": record["project"],
                    "label": record["label"],
                    "commit_id": record["commit_id"],
                    "cwe_id": record["cwe_id"],
                    "code_hash": record["code_hash"],
                    "source_path": record["source_path"],
                    "split": record.get("split", ""),
                }
            )
            stats["records"] += 1
            label_counter[int(record["label"])] += 1
            project_counter[record["project"] or "unknown"] += 1
    payload = {
        "dataset": dataset_name,
        "records": int(stats["records"]),
        "positive": int(label_counter[1]),
        "negative": int(label_counter[0]),
        "positive_ratio": float(label_counter[1] / max(stats["records"], 1)),
        "top_projects": project_counter.most_common(10),
        "index_path": str(index_path),
        "materialization": "index_only",
    }
    dump_json(output_dir / "stats.json", payload)
    print(f"{dataset_name}: {payload['records']} rows indexed at {index_path}")


def main() -> None:
    for dataset_name in TARGET_DATASETS:
        prepare_dataset(dataset_name)


if __name__ == "__main__":
    main()


### File `02_create_splits.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/02_create_splits.py
import json
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from common import PROCESSED_DIR, SPLITS_DIR, dump_json, ensure_dir
from datasets import (
    get_dataset_iterator,
    has_reveal_official_splits,
    iter_reveal_split_records,
)


TARGET_DATASETS = [name.strip() for name in os.getenv("GRACE_DATASETS", os.getenv("GRACE_DATASET", "devign")).split(",") if name.strip()]
OUTER_SPLITS = 10
INNER_SPLITS = 9
SEED = 42


def _assign_splits(frame: pd.DataFrame) -> pd.DataFrame:
    x = np.zeros(len(frame))
    y = frame["label"].to_numpy()
    groups = frame["code_hash"].to_numpy()
    outer = StratifiedGroupKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=SEED)
    train_val_idx, test_idx = next(outer.split(x, y, groups))
    assignments = np.array([""] * len(frame), dtype=object)
    assignments[test_idx] = "test"
    inner_frame = frame.iloc[train_val_idx].reset_index(drop=True)
    inner_x = np.zeros(len(inner_frame))
    inner_y = inner_frame["label"].to_numpy()
    inner_groups = inner_frame["code_hash"].to_numpy()
    inner = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED + 1)
    train_idx, val_idx = next(inner.split(inner_x, inner_y, inner_groups))
    assignments[train_val_idx[train_idx]] = "train"
    assignments[train_val_idx[val_idx]] = "val"
    result = frame.copy()
    result["split"] = assignments
    return result


def _write_records_from_assignment(dataset_name: str, assigned: pd.DataFrame, output_dir) -> None:
    split_index_path = output_dir / "split_index.csv"
    assigned.to_csv(split_index_path, index=False)
    record_to_split = dict(zip(assigned["record_id"], assigned["split"]))
    handles = {
        "train": (output_dir / "train.jsonl").open("w", encoding="utf-8"),
        "val": (output_dir / "val.jsonl").open("w", encoding="utf-8"),
        "test": (output_dir / "test.jsonl").open("w", encoding="utf-8"),
    }
    iterator = get_dataset_iterator(dataset_name)
    try:
        for record in iterator():
            split = record_to_split.get(record["record_id"])
            if split in handles:
                handles[split].write(json.dumps(record, ensure_ascii=False) + "\n")
    finally:
        for handle in handles.values():
            handle.close()


def _summarize_assigned(dataset_name: str, assigned: pd.DataFrame, split_index_path) -> dict:
    summary = {}
    for split in ["train", "val", "test"]:
        subset = assigned[assigned["split"] == split]
        summary[split] = {
            "rows": int(len(subset)),
            "positive": int(subset["label"].sum()),
            "negative": int((1 - subset["label"]).sum()),
            "groups": int(subset["code_hash"].nunique()),
        }
    summary["dataset"] = dataset_name
    summary["split_index_path"] = str(split_index_path)
    return summary


def _create_random_group_splits(dataset_name: str) -> None:
    index_path = PROCESSED_DIR / dataset_name / "index.csv"
    if not index_path.exists():
        print(f"Skipping {dataset_name}: run 01_prepare_datasets.py first.")
        return
    frame = pd.read_csv(index_path)
    if frame.empty:
        print(f"Skipping {dataset_name}: empty index.")
        return
    frame["label"] = frame["label"].astype(int)
    assigned = _assign_splits(frame)
    output_dir = ensure_dir(SPLITS_DIR / dataset_name)
    _write_records_from_assignment(dataset_name, assigned, output_dir)
    split_index_path = output_dir / "split_index.csv"
    summary = _summarize_assigned(dataset_name, assigned, split_index_path)
    dump_json(output_dir / "split_summary.json", summary)
    print(f"{dataset_name}: split files written to {output_dir}")


def _create_reveal_official_splits() -> None:
    output_dir = ensure_dir(SPLITS_DIR / "reveal")
    split_rows = []
    stats = {}
    for split_name in ["train", "val", "test"]:
        rows = list(iter_reveal_split_records(split_name))
        output_path = output_dir / f"{split_name}.jsonl"
        with output_path.open("w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        for row in rows:
            split_rows.append(
                {
                    "record_id": row["record_id"],
                    "dataset": row["dataset"],
                    "project": row["project"],
                    "label": row["label"],
                    "commit_id": row.get("commit_id", ""),
                    "cwe_id": row.get("cwe_id", ""),
                    "code_hash": row["code_hash"],
                    "source_path": row["source_path"],
                    "split": split_name,
                }
            )
        stats[split_name] = {
            "rows": len(rows),
            "positive": sum(int(row["label"]) for row in rows),
            "negative": sum(1 - int(row["label"]) for row in rows),
            "groups": len({row["code_hash"] for row in rows}),
        }
    split_index = pd.DataFrame(split_rows)
    split_index_path = output_dir / "split_index.csv"
    split_index.to_csv(split_index_path, index=False)
    dump_json(
        output_dir / "split_summary.json",
        {
            "dataset": "reveal",
            "strategy": "official",
            "split_index_path": str(split_index_path),
            **stats,
        },
    )
    print(f"reveal: official split files written to {output_dir}")


def _create_hint_based_splits(dataset_name: str) -> bool:
    index_path = PROCESSED_DIR / dataset_name / "index.csv"
    if not index_path.exists():
        return False
    frame = pd.read_csv(index_path)
    if frame.empty or "split" not in frame.columns:
        return False
    frame["split"] = frame["split"].fillna("").astype(str)
    valid = frame["split"].isin(["train", "val", "test"]).all()
    if not valid:
        return False
    output_dir = ensure_dir(SPLITS_DIR / dataset_name)
    _write_records_from_assignment(dataset_name, frame, output_dir)
    split_index_path = output_dir / "split_index.csv"
    summary = _summarize_assigned(dataset_name, frame, split_index_path)
    summary["strategy"] = "source_split_hint"
    dump_json(output_dir / "split_summary.json", summary)
    print(f"{dataset_name}: split files written from source split hints to {output_dir}")
    return True


def create_dataset_splits(dataset_name: str) -> None:
    if dataset_name == "reveal" and has_reveal_official_splits():
        _create_reveal_official_splits()
        return
    if dataset_name == "reveal" and _create_hint_based_splits(dataset_name):
        return
    _create_random_group_splits(dataset_name)


def main() -> None:
    for dataset_name in TARGET_DATASETS:
        create_dataset_splits(dataset_name)


if __name__ == "__main__":
    main()


### File `03_build_feature_store.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/03_build_feature_store.py
import json
import os

from hybrid_prefilter import build_feature_store, feature_store_path
from retrieval import DEFAULT_RETRIEVAL_MODEL_REPO_ID, default_retrieval_model_dir


DATASET_NAME = os.getenv("GRACE_DATASET", "devign")
GRAPH_BACKEND = os.getenv("GRACE_GRAPH_BACKEND", "auto")
SEMANTIC_MODEL_ID = os.getenv("GRACE_RETRIEVAL_MODEL_ID", DEFAULT_RETRIEVAL_MODEL_REPO_ID)
SEMANTIC_MODEL_DIR = default_retrieval_model_dir(SEMANTIC_MODEL_ID)
AUTO_DOWNLOAD = os.getenv("GRACE_AUTO_DOWNLOAD_RETRIEVAL_MODEL", "0").strip().lower() in {"1", "true", "yes", "on"}
FORCE_REBUILD = os.getenv("GRACE_FORCE_REBUILD_FEATURES", "0").strip().lower() in {"1", "true", "yes", "on"}
BATCH_SIZE = int(os.getenv("GRACE_FEATURE_BATCH_SIZE", "16"))
PROGRESS_EVERY = int(os.getenv("GRACE_FEATURE_PROGRESS_EVERY", "256"))


def _env_int(name: str) -> int | None:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return None
    lowered = value.strip().lower()
    if lowered == "none":
        return None
    return int(lowered)


def main() -> None:
    summary = {"dataset": DATASET_NAME, "splits": {}}
    for split_name in ["train", "val", "test"]:
        split_limit = _env_int(f"GRACE_FEATURE_LIMIT_{split_name.upper()}") or _env_int("GRACE_FEATURE_LIMIT")
        payload = build_feature_store(
            DATASET_NAME,
            split_name,
            semantic_model_name=SEMANTIC_MODEL_ID,
            semantic_model_dir=SEMANTIC_MODEL_DIR,
            graph_backend=GRAPH_BACKEND,
            force_rebuild=FORCE_REBUILD,
            auto_download_semantic_model=AUTO_DOWNLOAD,
            batch_size=BATCH_SIZE,
            limit=split_limit,
            progress_every=PROGRESS_EVERY,
        )
        summary["splits"][split_name] = {
            "path": str(feature_store_path(DATASET_NAME, split_name)),
            "rows": len(payload["record_ids"]),
            "semantic_dim": int(payload["semantic_embeddings"].shape[1]),
            "numeric_dim": int(payload["numeric_features"].shape[1]),
            "graph_backends": sorted(set(payload["graph_backends"])),
        }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### File `04_train_hybrid_prefilter.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/04_train_hybrid_prefilter.py
import json
import os

from hybrid_prefilter import DEFAULT_PREFILTER_MODEL_NAME, train_hybrid_prefilter
from retrieval import DEFAULT_RETRIEVAL_MODEL_REPO_ID


DATASET_NAME = os.getenv("GRACE_DATASET", "devign")
MODEL_NAME = os.getenv("GRACE_PREFILTER_MODEL_NAME", DEFAULT_PREFILTER_MODEL_NAME)
SEMANTIC_MODEL_NAME = os.getenv("GRACE_RETRIEVAL_MODEL_ID", DEFAULT_RETRIEVAL_MODEL_REPO_ID)
BATCH_SIZE = int(os.getenv("GRACE_PREFILTER_BATCH_SIZE", "128"))
EPOCHS = int(os.getenv("GRACE_PREFILTER_EPOCHS", "10"))
LEARNING_RATE = float(os.getenv("GRACE_PREFILTER_LEARNING_RATE", "7e-4"))
RANDOM_SEED = int(os.getenv("GRACE_PREFILTER_RANDOM_SEED", "42"))
LOSS_NAME = os.getenv("GRACE_PREFILTER_LOSS", "bce")
FOCAL_GAMMA = float(os.getenv("GRACE_PREFILTER_FOCAL_GAMMA", "2.0"))
HARD_NEGATIVE_MINING = os.getenv("GRACE_HARD_NEGATIVE_MINING", "0").strip().lower() in {"1", "true", "yes", "on"}
HARD_NEGATIVE_QUANTILE = float(os.getenv("GRACE_HARD_NEGATIVE_QUANTILE", "0.85"))
HARD_NEGATIVE_WEIGHT = float(os.getenv("GRACE_HARD_NEGATIVE_WEIGHT", "2.5"))
HARD_NEGATIVE_EPOCHS = int(os.getenv("GRACE_HARD_NEGATIVE_EPOCHS", "2"))
TOKEN_MAX_TOKENS = int(os.getenv("GRACE_TOKEN_MAX_TOKENS", "32000"))
TOKEN_SEQUENCE_LENGTH = int(os.getenv("GRACE_TOKEN_SEQUENCE_LENGTH", "384"))
TOKEN_EMBEDDING_DIM = int(os.getenv("GRACE_TOKEN_EMBEDDING_DIM", "96"))
TOKEN_FILTERS = int(os.getenv("GRACE_TOKEN_FILTERS", "96"))
AST_MAX_TOKENS = int(os.getenv("GRACE_AST_MAX_TOKENS", "8000"))
AST_SEQUENCE_LENGTH = int(os.getenv("GRACE_AST_SEQUENCE_LENGTH", "196"))
AST_EMBEDDING_DIM = int(os.getenv("GRACE_AST_EMBEDDING_DIM", "64"))
AST_FILTERS = int(os.getenv("GRACE_AST_FILTERS", "64"))
PROJECTION_DIM = int(os.getenv("GRACE_PREFILTER_PROJECTION_DIM", "192"))
DENSE_UNITS = int(os.getenv("GRACE_PREFILTER_DENSE_UNITS", "192"))
DROPOUT_RATE = float(os.getenv("GRACE_PREFILTER_DROPOUT", "0.25"))


def main() -> None:
    summary = train_hybrid_prefilter(
        DATASET_NAME,
        model_name=MODEL_NAME,
        semantic_model_name=SEMANTIC_MODEL_NAME,
        token_max_tokens=TOKEN_MAX_TOKENS,
        token_sequence_length=TOKEN_SEQUENCE_LENGTH,
        token_embedding_dim=TOKEN_EMBEDDING_DIM,
        token_filters=TOKEN_FILTERS,
        ast_max_tokens=AST_MAX_TOKENS,
        ast_sequence_length=AST_SEQUENCE_LENGTH,
        ast_embedding_dim=AST_EMBEDDING_DIM,
        ast_filters=AST_FILTERS,
        projection_dim=PROJECTION_DIM,
        dense_units=DENSE_UNITS,
        dropout_rate=DROPOUT_RATE,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        random_seed=RANDOM_SEED,
        log_progress=True,
        loss_name=LOSS_NAME,
        focal_gamma=FOCAL_GAMMA,
        hard_negative_mining=HARD_NEGATIVE_MINING,
        hard_negative_quantile=HARD_NEGATIVE_QUANTILE,
        hard_negative_weight=HARD_NEGATIVE_WEIGHT,
        hard_negative_epochs=HARD_NEGATIVE_EPOCHS,
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### File `05_calibrate_budget_controller.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/05_calibrate_budget_controller.py
import json
import os

import numpy as np

from common import MODELS_DIR, dump_json
from hybrid_prefilter import DEFAULT_PREFILTER_MODEL_NAME, predict_feature_store
from metrics import (
    apply_calibrator,
    apply_platt_scaler,
    choose_best_f1_threshold,
    choose_low_threshold,
    compute_binary_metrics,
    fit_calibrator,
)


DATASET_NAME = os.getenv("GRACE_DATASET", "devign")
MODEL_NAME = os.getenv("GRACE_PREFILTER_MODEL_NAME", DEFAULT_PREFILTER_MODEL_NAME)
CALIBRATION_METHOD = os.getenv("GRACE_CALIBRATION_METHOD", "auto").strip().lower()
TARGET_RECALL = float(os.getenv("GRACE_TARGET_RECALL", "0.995"))
ROUTING_MODE = os.getenv("GRACE_ROUTING_MODE", "baseline").strip().lower()
ROUTING_OBJECTIVE = os.getenv("GRACE_ROUTING_OBJECTIVE", "f1").strip().lower()
ROUTING_INSPECT_PROXY = os.getenv("GRACE_ROUTING_INSPECT_PROXY", "probability").strip().lower()
ROUTING_RECALL_FLOOR = float(os.getenv("GRACE_ROUTING_RECALL_FLOOR", str(TARGET_RECALL)))
LLM_BUDGET = float(os.getenv("GRACE_LLM_BUDGET", "0.15"))
HIGH_RISK_TARGET_PRECISION = float(os.getenv("GRACE_HIGH_RISK_TARGET_PRECISION", "0.70"))
DIRECT_ACCEPT_MIN_PROBABILITY = float(os.getenv("GRACE_DIRECT_ACCEPT_MIN_PROBABILITY", "0.20"))
HIGH_RISK_THRESHOLD_STRATEGY = os.getenv("GRACE_HIGH_RISK_THRESHOLD_STRATEGY", "f1").strip().lower()
TAU_NEG_MIN = float(os.getenv("GRACE_TAU_NEG_MIN", "0.02"))
TAU_NEG_MAX = float(os.getenv("GRACE_TAU_NEG_MAX", "0.30"))
TAU_NEG_STEPS = int(os.getenv("GRACE_TAU_NEG_STEPS", "15"))
TAU_POS_MIN = float(os.getenv("GRACE_TAU_POS_MIN", "0.45"))
TAU_POS_MAX = float(os.getenv("GRACE_TAU_POS_MAX", "0.90"))
TAU_POS_STEPS = int(os.getenv("GRACE_TAU_POS_STEPS", "19"))


def _candidate_grid(low: float, high: float, steps: int) -> np.ndarray:
    if steps <= 1:
        return np.asarray([float(low)], dtype=np.float32)
    return np.unique(np.round(np.linspace(low, high, steps), 6))


def _choose_high_threshold(probabilities: np.ndarray, labels: np.ndarray, tau_low: float, target_precision: float, minimum: float) -> tuple[float, str]:
    best_precision_threshold = None
    for threshold in np.unique(np.round(np.sort(probabilities), 6)):
        if threshold <= max(tau_low, minimum):
            continue
        predictions = (probabilities >= threshold).astype(int)
        metrics = compute_binary_metrics(labels, predictions, probabilities)
        precision = float(metrics["precision"])
        coverage = float(np.mean(probabilities >= threshold))
        if precision >= target_precision and coverage > 0:
            best_precision_threshold = float(threshold)
            break
    if best_precision_threshold is not None:
        return best_precision_threshold, "target_precision"

    best_threshold = max(tau_low + 0.05, minimum)
    best_score = -1.0
    for threshold in np.unique(np.round(np.sort(probabilities), 6)):
        if threshold <= max(tau_low, minimum):
            continue
        predictions = (probabilities >= threshold).astype(int)
        metrics = compute_binary_metrics(labels, predictions, probabilities)
        precision = float(metrics["precision"])
        recall = float(metrics["recall"])
        beta_sq = 0.5 * 0.5
        denominator = beta_sq * precision + recall
        score = 0.0 if denominator == 0 else (1 + beta_sq) * precision * recall / denominator
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
    return best_threshold, "f0_5_fallback"


def _calibration_payload(probabilities: np.ndarray, labels: np.ndarray) -> dict:
    calibrator = fit_calibrator(probabilities, labels, method=CALIBRATION_METHOD)
    calibrated = apply_calibrator(probabilities, calibrator)
    return {
        "calibration_method_requested": CALIBRATION_METHOD,
        "calibrator": calibrator,
        "calibrated": calibrated,
        "calibration_metrics": {
            "brier": float(compute_binary_metrics(labels, (calibrated >= 0.5).astype(int), calibrated).get("brier") or 0.0),
            "nll": float(compute_binary_metrics(labels, (calibrated >= 0.5).astype(int), calibrated).get("nll") or 0.0),
            "ece": float(compute_binary_metrics(labels, (calibrated >= 0.5).astype(int), calibrated).get("ece") or 0.0),
        },
    }


def _routing_proxy_predictions(probabilities: np.ndarray, tau_low: float, tau_high: float) -> np.ndarray:
    if ROUTING_INSPECT_PROXY == "positive":
        inspect_pred = np.ones_like(probabilities, dtype=np.int32)
    elif ROUTING_INSPECT_PROXY == "negative":
        inspect_pred = np.zeros_like(probabilities, dtype=np.int32)
    else:
        inspect_pred = (probabilities >= 0.5).astype(np.int32)
    return np.where(probabilities <= tau_low, 0, np.where(probabilities >= tau_high, 1, inspect_pred)).astype(np.int32)


def _routing_stats(probabilities: np.ndarray, labels: np.ndarray, tau_low: float, tau_high: float) -> dict:
    proxy_predictions = _routing_proxy_predictions(probabilities, tau_low, tau_high)
    metrics = compute_binary_metrics(labels, proxy_predictions, probabilities)
    inspect_mask = (probabilities > tau_low) & (probabilities < tau_high)
    metrics["llm_call_rate"] = float(np.mean(inspect_mask))
    metrics["auto_positive_rate"] = float(np.mean(probabilities >= tau_high))
    metrics["auto_negative_rate"] = float(np.mean(probabilities <= tau_low))
    metrics["inspect_rate"] = float(np.mean(inspect_mask))
    return metrics


def _choose_routing_thresholds(probabilities: np.ndarray, labels: np.ndarray) -> tuple[float, float, str, dict]:
    neg_grid = _candidate_grid(TAU_NEG_MIN, TAU_NEG_MAX, TAU_NEG_STEPS)
    pos_grid = _candidate_grid(TAU_POS_MIN, TAU_POS_MAX, TAU_POS_STEPS)
    best = None
    best_key = None
    best_metrics = None
    for tau_low in neg_grid:
        for tau_high in pos_grid:
            if float(tau_high) <= float(tau_low):
                continue
            metrics = _routing_stats(probabilities, labels, float(tau_low), float(tau_high))
            recall = float(metrics["recall"])
            llm_rate = float(metrics["llm_call_rate"])
            if recall >= ROUTING_RECALL_FLOOR and llm_rate <= LLM_BUDGET:
                objective = float(metrics.get(ROUTING_OBJECTIVE) or 0.0)
                key = (objective, float(metrics["precision"]), float(metrics["accuracy"]), -llm_rate)
                if best_key is None or key > best_key:
                    best_key = key
                    best = (float(tau_low), float(tau_high))
                    best_metrics = metrics
    if best is not None:
        return best[0], best[1], "constrained_search", best_metrics

    fallback_low = float(choose_low_threshold(probabilities, labels, ROUTING_RECALL_FLOOR))
    fallback_high, tau_high_best_f1 = choose_best_f1_threshold(probabilities, labels, minimum=max(fallback_low, DIRECT_ACCEPT_MIN_PROBABILITY))
    fallback_metrics = _routing_stats(probabilities, labels, fallback_low, fallback_high)
    fallback_metrics["tau_high_best_f1"] = float(tau_high_best_f1)
    return fallback_low, float(fallback_high), "fallback_f1", fallback_metrics


def main() -> None:
    predictions = predict_feature_store(DATASET_NAME, "val", model_name=MODEL_NAME)
    labels = np.asarray(predictions["labels"], dtype=np.int32)
    fusion_scores = np.asarray(predictions["fusion_score"], dtype=np.float32)
    semantic_scores = np.asarray(predictions["semantic_score"], dtype=np.float32)
    graph_scores = np.asarray(predictions["graph_score"], dtype=np.float32)

    calibration = _calibration_payload(fusion_scores, labels)
    calibrated = np.asarray(calibration["calibrated"], dtype=np.float32)
    if ROUTING_MODE == "constrained":
        tau_low, tau_high, tau_strategy, routing_metrics = _choose_routing_thresholds(calibrated, labels)
    else:
        tau_low = choose_low_threshold(calibrated, labels, TARGET_RECALL)
        tau_high_minimum = max(float(tau_low), DIRECT_ACCEPT_MIN_PROBABILITY)
        if HIGH_RISK_THRESHOLD_STRATEGY == "precision":
            tau_high, tau_strategy = _choose_high_threshold(
                calibrated,
                labels,
                tau_low=tau_low,
                target_precision=HIGH_RISK_TARGET_PRECISION,
                minimum=tau_high_minimum,
            )
            tau_high_best_f1 = None
        else:
            tau_high, tau_high_best_f1 = choose_best_f1_threshold(
                calibrated,
                labels,
                minimum=tau_high_minimum,
            )
            tau_strategy = "max_f1"
        routing_metrics = _routing_stats(calibrated, labels, tau_low, tau_high)
        routing_metrics["tau_high_best_f1"] = float(tau_high_best_f1) if tau_high_best_f1 is not None else None

    low_predictions = (calibrated > tau_low).astype(int)
    high_predictions = (calibrated >= tau_high).astype(int)
    routing_proxy_predictions = _routing_proxy_predictions(calibrated, tau_low, tau_high)
    summary = {
        "dataset": DATASET_NAME,
        "model_name": MODEL_NAME,
        "target_recall": TARGET_RECALL,
        "routing_mode": ROUTING_MODE,
        "routing_objective": ROUTING_OBJECTIVE,
        "routing_inspect_proxy": ROUTING_INSPECT_PROXY,
        "routing_recall_floor": ROUTING_RECALL_FLOOR,
        "llm_budget": LLM_BUDGET,
        "calibration_method_requested": CALIBRATION_METHOD,
        "calibration_method": calibration["calibrator"]["method"],
        "high_risk_target_precision": HIGH_RISK_TARGET_PRECISION,
        "high_risk_threshold_strategy": HIGH_RISK_THRESHOLD_STRATEGY,
        "direct_accept_min_probability": DIRECT_ACCEPT_MIN_PROBABILITY,
        "tau_low": float(tau_low),
        "tau_high": float(tau_high),
        "tau_high_strategy": tau_strategy,
        "tau_high_best_f1": routing_metrics.get("tau_high_best_f1"),
        "calibrator": calibration["calibrator"],
        "calibration_metrics": calibration["calibration_metrics"],
        "val_metrics_uncalibrated": compute_binary_metrics(labels, (fusion_scores >= 0.5).astype(int), fusion_scores),
        "val_metrics_keep_for_llm": compute_binary_metrics(labels, low_predictions, calibrated),
        "val_metrics_high_risk": compute_binary_metrics(labels, high_predictions, calibrated),
        "val_metrics_direct_accept": compute_binary_metrics(labels, high_predictions, calibrated),
        "val_metrics_routing_proxy": compute_binary_metrics(labels, routing_proxy_predictions, calibrated),
        "branch_means": {
            "fusion_score_mean": float(np.mean(fusion_scores)),
            "semantic_score_mean": float(np.mean(semantic_scores)),
            "graph_score_mean": float(np.mean(graph_scores)),
        },
        "llm_budget_estimate": {
            "keep_ratio": float(np.mean(calibrated > tau_low)),
            "high_ratio": float(np.mean(calibrated >= tau_high)),
            "inspect_ratio": float(np.mean((calibrated > tau_low) & (calibrated < tau_high))),
        },
        "routing_metrics": routing_metrics,
    }
    output_path = MODELS_DIR / DATASET_NAME / f"calibration.{MODEL_NAME}.json"
    dump_json(output_path, summary)
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### File `06_build_demo_bank.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/06_build_demo_bank.py
import random
import os

from common import RETRIEVAL_DIR, SPLITS_DIR, dump_json, ensure_dir, get_record_code, iter_jsonl
from retrieval import (
    DEFAULT_EMBEDDING_BATCH_SIZE,
    DEFAULT_EMBEDDING_MAX_LENGTH,
    DEFAULT_RETRIEVAL_MODEL_REPO_ID,
    build_demo_bank,
    default_retrieval_model_dir,
    save_demo_bank,
)


DATASET_NAME = os.getenv("GRACE_DATASET", "devign")
MAX_EXAMPLES_PER_LABEL = int(os.getenv("GRACE_MAX_EXAMPLES_PER_LABEL", "4000"))
MAX_FEATURES = int(os.getenv("GRACE_TFIDF_MAX_FEATURES", "50000"))
SEED = 42
SEMANTIC_BACKEND = os.getenv("GRACE_RETRIEVAL_BACKEND", "auto")
SEMANTIC_MODEL_NAME = os.getenv("GRACE_RETRIEVAL_MODEL_ID", DEFAULT_RETRIEVAL_MODEL_REPO_ID)
SEMANTIC_MODEL_DIR = default_retrieval_model_dir(SEMANTIC_MODEL_NAME)
SEMANTIC_BATCH_SIZE = int(os.getenv("GRACE_RETRIEVAL_BATCH_SIZE", str(DEFAULT_EMBEDDING_BATCH_SIZE)))
SEMANTIC_MAX_LENGTH = int(os.getenv("GRACE_RETRIEVAL_MAX_LENGTH", str(DEFAULT_EMBEDDING_MAX_LENGTH)))
AUTO_DOWNLOAD_SEMANTIC_MODEL = os.getenv("GRACE_AUTO_DOWNLOAD_RETRIEVAL_MODEL", "").strip().lower() in {"1", "true", "yes", "on"}
GRAPH_BACKEND = os.getenv("GRACE_GRAPH_BACKEND", "auto")
PROGRESS_EVERY = int(os.getenv("GRACE_BUILD_PROGRESS_EVERY", "250"))


def _sample_records_by_label(split_path, max_examples_per_label: int, seed: int):
    rng = random.Random(seed)
    reservoirs = {0: [], 1: []}
    seen = {0: 0, 1: 0}
    for record in iter_jsonl(split_path):
        code = get_record_code(record)
        if not code:
            continue
        label = int(record["label"])
        canonical = {
            "record_id": record["record_id"],
            "dataset": record.get("dataset", DATASET_NAME),
            "label": label,
            "project": record.get("project", ""),
            "code": code,
            "code_hash": record.get("code_hash"),
        }
        seen[label] += 1
        bucket = reservoirs[label]
        if len(bucket) < max_examples_per_label:
            bucket.append(canonical)
            continue
        replacement_index = rng.randint(0, seen[label] - 1)
        if replacement_index < max_examples_per_label:
            bucket[replacement_index] = canonical
    return reservoirs[0] + reservoirs[1]


def main() -> None:
    train_path = SPLITS_DIR / DATASET_NAME / "train.jsonl"
    if not train_path.exists():
        raise FileNotFoundError(f"Missing train split for {DATASET_NAME}. Run 02_create_splits.py first.")
    records = _sample_records_by_label(train_path, max_examples_per_label=MAX_EXAMPLES_PER_LABEL, seed=SEED)
    print(
        f"Building demo bank for dataset={DATASET_NAME} with {len(records)} sampled records "
        f"(semantic={SEMANTIC_BACKEND}, graph={GRAPH_BACKEND})"
    )
    bank = build_demo_bank(
        records=records,
        max_examples_per_label=MAX_EXAMPLES_PER_LABEL,
        max_features=MAX_FEATURES,
        random_seed=SEED,
        semantic_backend=SEMANTIC_BACKEND,
        semantic_model_name=SEMANTIC_MODEL_NAME,
        semantic_model_dir=SEMANTIC_MODEL_DIR,
        semantic_batch_size=SEMANTIC_BATCH_SIZE,
        semantic_max_length=SEMANTIC_MAX_LENGTH,
        auto_download_semantic_model=AUTO_DOWNLOAD_SEMANTIC_MODEL,
        graph_backend=GRAPH_BACKEND,
        progress_every=PROGRESS_EVERY,
    )
    output_dir = ensure_dir(RETRIEVAL_DIR / DATASET_NAME)
    bank_path = output_dir / "demo_bank.joblib"
    save_demo_bank(bank_path, bank)
    summary = {
        "dataset": DATASET_NAME,
        "bank_path": str(bank_path),
        "total_examples": len(bank["records"]),
        "negative_examples": sum(1 for row in bank["records"] if row["label"] == 0),
        "positive_examples": sum(1 for row in bank["records"] if row["label"] == 1),
        "semantic_backend": bank.get("semantic_backend"),
        "semantic_notice": bank.get("semantic_notice"),
        "semantic_config": bank.get("semantic_config"),
        "graph_backend_requested": bank.get("graph_backend_requested"),
        "graph_backend_resolved": bank.get("graph_backend_resolved"),
        "graph_backend_notice": bank.get("graph_backend_notice"),
        "graph_backend_counts": bank.get("graph_backend_counts"),
    }
    dump_json(output_dir / "summary.json", summary)
    print(f"Saved demo bank for {DATASET_NAME} to {bank_path}")


if __name__ == "__main__":
    main()


### File `07_run_grace_hybrid.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/07_run_grace_hybrid.py
import json
import os
import time
from pathlib import Path

import numpy as np

from common import METRICS_DIR, MODELS_DIR, PREDICTIONS_DIR, RETRIEVAL_DIR, SPLITS_DIR, dump_json, ensure_dir, get_record_code, iter_jsonl
from graphs import get_graph_features
from hybrid_prefilter import DEFAULT_PREFILTER_MODEL_NAME, predict_feature_store
from local_llm_client import CACHE_SCHEMA_VERSION, DEFAULT_MODEL_REPO_ID, LocalVulnLLMClassifier, build_detection_prompt, default_local_model_dir
from localizer import locate_suspicious_slices
from metrics import apply_calibrator, apply_platt_scaler, bootstrap_f1_interval, compute_binary_metrics
from retrieval import DEMO_BANK_SCHEMA_VERSION, load_demo_bank, retrieve_examples


PREDICTION_SCHEMA_VERSION = 1


def _env_flag(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


def _env_int(name: str, default: int | None) -> int | None:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    lowered = value.strip().lower()
    if lowered == "none":
        return None
    return int(lowered)


DATASET_NAME = os.getenv("GRACE_DATASET", "devign")
PREFILTER_MODEL_NAME = os.getenv("GRACE_PREFILTER_MODEL_NAME", DEFAULT_PREFILTER_MODEL_NAME)
LLM_MODEL_NAME = os.getenv("GRACE_LOCAL_MODEL_ID", DEFAULT_MODEL_REPO_ID)
LOCAL_MODEL_DIR = default_local_model_dir(LLM_MODEL_NAME)
GRAPH_BACKEND = os.getenv("GRACE_GRAPH_BACKEND", "auto")
MAX_TEST_SAMPLES = _env_int("GRACE_MAX_TEST_SAMPLES", None)
TEST_CHUNK_SIZE = _env_int("GRACE_TEST_CHUNK_SIZE", None)
TEST_CHUNK_INDEX = _env_int("GRACE_TEST_CHUNK_INDEX", None)
INSPECT_DEMOS = int(os.getenv("GRACE_INSPECT_DEMOS", "3"))
HIGH_RISK_DEMOS = int(os.getenv("GRACE_HIGH_RISK_DEMOS", "5"))
DEMO_CHAR_LIMIT = int(os.getenv("GRACE_DEMO_CHAR_LIMIT", "800"))
MAX_NEW_TOKENS = int(os.getenv("GRACE_MAX_NEW_TOKENS", "160"))
LOAD_IN_4BIT = _env_flag("GRACE_LOAD_IN_4BIT", True)
AUTO_DOWNLOAD_MODEL = _env_flag("GRACE_AUTO_DOWNLOAD_MODEL", False)
CALL_LLM_FOR_INSPECT = _env_flag("GRACE_CALL_LLM_FOR_INSPECT", True)
CALL_LLM_FOR_HIGH = _env_flag("GRACE_CALL_LLM_FOR_HIGH", False)
RESUME = _env_flag("GRACE_RESUME", True)
VARIANT_SUFFIX = os.getenv("GRACE_VARIANT_OUTPUT_SUFFIX", "").strip().lower()
PREDICTION_FILE_STEM = os.getenv("GRACE_PREDICTION_FILE_STEM") or (f"grace_hybrid_predictions_{VARIANT_SUFFIX}" if VARIANT_SUFFIX else "grace_hybrid_predictions")
RUN_STATE_FILE_STEM = os.getenv("GRACE_RUN_STATE_FILE_STEM") or (f"grace_hybrid_run_state_{VARIANT_SUFFIX}" if VARIANT_SUFFIX else "grace_hybrid_run_state")
METRICS_FILE_STEM = os.getenv("GRACE_EVALUATION_FILE_STEM") or (f"grace_hybrid_evaluation_summary_{VARIANT_SUFFIX}" if VARIANT_SUFFIX else "grace_hybrid_evaluation_summary")
EVIDENCE_AWARE_VERIFIER = _env_flag("GRACE_EVIDENCE_AWARE_VERIFIER", False)


def _risk_band(probability: float, tau_low: float, tau_high: float) -> str:
    if probability <= tau_low:
        return "skip"
    if probability >= tau_high:
        return "high"
    return "inspect"


def _should_call_llm(risk_band: str) -> bool:
    if risk_band == "inspect":
        return CALL_LLM_FOR_INSPECT
    if risk_band == "high":
        return CALL_LLM_FOR_HIGH
    return False


def _direct_prefilter_prediction(risk_band: str) -> int:
    if risk_band == "high":
        return 1
    return 0


def _apply_saved_calibrator(score: float, calibration: dict) -> float:
    if "calibrator" in calibration:
        calibrated = apply_calibrator([score], calibration["calibrator"])
        return float(calibrated[0])
    if "platt_scaler" in calibration:
        calibrated = apply_platt_scaler([score], calibration["platt_scaler"])
        return float(calibrated[0])
    return float(score)


def _has_positive_evidence(result: dict) -> bool:
    vulnerable_lines = result.get("vulnerable_lines") or []
    if not vulnerable_lines:
        return False
    sink_or_api = str(result.get("sink_or_api") or "").strip()
    missing_guard = str(result.get("missing_guard") or "").strip()
    return bool(sink_or_api or missing_guard)


def _verified_llm_prediction(result: dict) -> tuple[int, str, str]:
    prediction = int(result.get("label_int", 0))
    reason = str(result.get("reason") or "").strip()
    if not EVIDENCE_AWARE_VERIFIER:
        return prediction, "disabled", reason
    if prediction == 1 and not _has_positive_evidence(result):
        return 0, "rejected_missing_evidence", f"evidence_verifier_rejected_positive: {reason}" if reason else "evidence_verifier_rejected_positive"
    return prediction, "accepted", reason


def _count_records(split_path: Path, limit: int | None = None) -> int:
    total = 0
    for record in iter_jsonl(split_path):
        if not get_record_code(record):
            continue
        total += 1
        if limit is not None and total >= limit:
            break
    return total


def _resolve_chunk_bounds(total_records: int) -> tuple[int, int] | None:
    if TEST_CHUNK_SIZE is None or TEST_CHUNK_INDEX is None:
        return None
    if TEST_CHUNK_SIZE <= 0:
        raise ValueError("GRACE_TEST_CHUNK_SIZE must be a positive integer.")
    if TEST_CHUNK_INDEX < 0:
        raise ValueError("GRACE_TEST_CHUNK_INDEX must be a non-negative integer.")
    start = TEST_CHUNK_INDEX * TEST_CHUNK_SIZE
    end = min(total_records, start + TEST_CHUNK_SIZE)
    return start, end


def _select_target_record_ids(split_path: Path, *, limit: int | None = None) -> tuple[set[str], int, dict | None]:
    total_records = _count_records(split_path, limit=limit)
    bounds = _resolve_chunk_bounds(total_records)
    if bounds is None:
        selected = set()
        seen = 0
        for record in iter_jsonl(split_path):
            if not get_record_code(record):
                continue
            selected.add(str(record["record_id"]))
            seen += 1
            if limit is not None and seen >= limit:
                break
        return selected, total_records, None

    start, end = bounds
    selected = set()
    seen = 0
    for record in iter_jsonl(split_path):
        if not get_record_code(record):
            continue
        if seen >= end:
            break
        if seen >= start:
            selected.add(str(record["record_id"]))
        seen += 1
        if limit is not None and seen >= limit:
            break
    chunk_context = {
        "chunk_index": TEST_CHUNK_INDEX,
        "chunk_size": TEST_CHUNK_SIZE,
        "start_offset": start,
        "end_offset_exclusive": end,
        "target_records_in_chunk": len(selected),
    }
    return selected, total_records, chunk_context


def _load_predictions(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _prepare_prediction_file(predictions_path: Path) -> list[dict]:
    if not RESUME or not predictions_path.exists():
        return []
    rows = _load_predictions(predictions_path)
    return [row for row in rows if row.get("schema_version") == PREDICTION_SCHEMA_VERSION]


def _build_run_signature(calibration: dict, bank: dict) -> dict:
    return {
        "dataset": DATASET_NAME,
        "variant_suffix": VARIANT_SUFFIX,
        "prefilter_model_name": PREFILTER_MODEL_NAME,
        "llm_model_name": LLM_MODEL_NAME,
        "graph_backend": GRAPH_BACKEND,
        "retrieval_backend": bank.get("semantic_backend"),
        "tau_low": float(calibration["tau_low"]),
        "tau_high": float(calibration["tau_high"]),
        "calibration_method": calibration.get("calibration_method"),
        "routing_mode": calibration.get("routing_mode"),
        "inspect_demos": INSPECT_DEMOS,
        "high_risk_demos": HIGH_RISK_DEMOS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "load_in_4bit": LOAD_IN_4BIT,
        "call_llm_for_inspect": CALL_LLM_FOR_INSPECT,
        "call_llm_for_high": CALL_LLM_FOR_HIGH,
    }


def _mean(values: list[float]) -> float:
    return float(sum(values) / max(len(values), 1))


def _summarize_predictions(
    predictions_path: Path,
    total_target_records: int,
    output_dir: Path,
    bank: dict,
    *,
    run_signature: dict,
    chunk_context: dict | None = None,
) -> dict:
    rows = _load_predictions(predictions_path)
    routing = {"skip": 0, "inspect": 0, "high": 0}
    decision_sources = {"prefilter": 0, "llm": 0}
    llm_cache_hits = 0
    llm_calls = 0
    for row in rows:
        routing[row["risk_band"]] = routing.get(row["risk_band"], 0) + 1
        decision_sources[row["decision_source"]] = decision_sources.get(row["decision_source"], 0) + 1
        if row.get("llm_cache_hit"):
            llm_cache_hits += 1
        if row.get("llm_called"):
            llm_calls += 1
    summary = {
        "dataset": DATASET_NAME,
        "schema_version": PREDICTION_SCHEMA_VERSION,
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "resolved_samples": len(rows),
        "target_samples": total_target_records,
        "complete": len(rows) == total_target_records,
        "evaluation_ready": len(rows) == total_target_records,
        "llm_calls": llm_calls,
        "llm_cache_hits": llm_cache_hits,
        "llm_call_ratio": float(llm_calls / max(len(rows), 1)),
        "routing": routing,
        "decision_sources": decision_sources,
        "retrieval_backend": bank.get("semantic_backend"),
        "graph_backend_requested": GRAPH_BACKEND,
        "predictions_path": str(predictions_path),
        "chunking": chunk_context,
        "run_signature": run_signature,
        "config": {
            "prefilter_model_name": PREFILTER_MODEL_NAME,
            "llm_model_name": LLM_MODEL_NAME,
            "inspect_demos": INSPECT_DEMOS,
            "high_risk_demos": HIGH_RISK_DEMOS,
            "max_new_tokens": MAX_NEW_TOKENS,
            "load_in_4bit": LOAD_IN_4BIT,
            "call_llm_for_inspect": CALL_LLM_FOR_INSPECT,
            "call_llm_for_high": CALL_LLM_FOR_HIGH,
        },
    }
    if rows:
        labels = [int(row["ground_truth"]) for row in rows]
        predictions = [int(row["prediction"]) for row in rows]
        probabilities = [float(row["calibrated_probability"]) for row in rows]
        graph_times = [float(row.get("graph_latency_ms") or 0.0) for row in rows]
        retrieval_times = [float(row.get("retrieval_latency_ms") or 0.0) for row in rows]
        llm_times = [float(row.get("llm_latency_ms") or 0.0) for row in rows]
        total_times = [float(row.get("record_runtime_ms") or 0.0) for row in rows]
        summary["timing_ms"] = {
            "graph_total": float(sum(graph_times)),
            "graph_mean": _mean(graph_times),
            "retrieval_total": float(sum(retrieval_times)),
            "retrieval_mean": _mean(retrieval_times),
            "llm_total": float(sum(llm_times)),
            "llm_mean": _mean(llm_times),
            "record_total": float(sum(total_times)),
            "record_mean": _mean(total_times),
        }
        summary.update(compute_binary_metrics(labels, predictions, probabilities))
        summary["bootstrap_f1"] = bootstrap_f1_interval(labels, predictions, iterations=500)
    dump_json(output_dir / f"{RUN_STATE_FILE_STEM}.json", summary)
    dump_json(METRICS_DIR / DATASET_NAME / f"{METRICS_FILE_STEM}.json", summary)
    return summary


def main() -> None:
    calibration_path = MODELS_DIR / DATASET_NAME / f"calibration.{PREFILTER_MODEL_NAME}.json"
    bank_path = RETRIEVAL_DIR / DATASET_NAME / "demo_bank.joblib"
    test_path = SPLITS_DIR / DATASET_NAME / "test.jsonl"
    if not calibration_path.exists():
        raise FileNotFoundError(f"Missing calibration file: {calibration_path}")
    if not bank_path.exists():
        raise FileNotFoundError(f"Missing demo bank: {bank_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing test split: {test_path}")

    calibration = json.loads(calibration_path.read_text(encoding="utf-8"))
    tau_low = float(calibration["tau_low"])
    tau_high = float(calibration["tau_high"])
    bank = load_demo_bank(bank_path)
    if bank.get("schema_version") != DEMO_BANK_SCHEMA_VERSION:
        raise RuntimeError("Demo bank schema mismatch. Rebuild `06_build_demo_bank.py`.")
    run_signature = _build_run_signature(calibration, bank)

    test_predictions = predict_feature_store(DATASET_NAME, "test", model_name=PREFILTER_MODEL_NAME)
    score_map = {
        record_id: {
            "fusion_score": float(fusion),
            "semantic_score": float(semantic),
            "graph_score": float(graph),
        }
        for record_id, fusion, semantic, graph in zip(
            test_predictions["record_ids"],
            test_predictions["fusion_score"],
            test_predictions["semantic_score"],
            test_predictions["graph_score"],
        )
    }

    client = None
    if CALL_LLM_FOR_INSPECT or CALL_LLM_FOR_HIGH:
        client = LocalVulnLLMClassifier(
            model_name=LLM_MODEL_NAME,
            model_dir=LOCAL_MODEL_DIR,
            max_new_tokens=MAX_NEW_TOKENS,
            load_in_4bit=LOAD_IN_4BIT,
            auto_download=AUTO_DOWNLOAD_MODEL,
        )
        client.prepare()

    output_dir = ensure_dir(PREDICTIONS_DIR / DATASET_NAME)
    predictions_path = output_dir / f"{PREDICTION_FILE_STEM}.jsonl"
    existing_rows = _prepare_prediction_file(predictions_path)
    processed_ids = {row["record_id"] for row in existing_rows}
    target_record_ids, total_target_records, chunk_context = _select_target_record_ids(test_path, limit=MAX_TEST_SAMPLES)
    chunk_target_records = len(target_record_ids)
    if not target_record_ids:
        print(
            json.dumps(
                {
                    "dataset": DATASET_NAME,
                    "message": "No records selected for this chunk configuration.",
                    "chunking": chunk_context,
                    "total_target_records": total_target_records,
                },
                ensure_ascii=False,
                indent=2,
            )
        )
        summary = _summarize_predictions(
            predictions_path,
            total_target_records,
            output_dir,
            bank,
            run_signature=run_signature,
            chunk_context=chunk_context,
        )
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        return

    run_state_path = output_dir / f"{RUN_STATE_FILE_STEM}.json"
    if RESUME and predictions_path.exists() and run_state_path.exists():
        try:
            previous_run_state = json.loads(run_state_path.read_text(encoding="utf-8"))
        except Exception:
            previous_run_state = {}
        previous_signature = previous_run_state.get("run_signature")
        if previous_signature and previous_signature != run_signature:
            print(
                json.dumps(
                    {
                        "message": "Existing predictions were produced by a different run signature. Starting a fresh prediction file.",
                        "previous_run_signature": previous_signature,
                        "current_run_signature": run_signature,
                    },
                    ensure_ascii=False,
                    indent=2,
                )
            )
            existing_rows = []
            processed_ids = set()

    already_done_in_chunk = len([record_id for record_id in target_record_ids if record_id in processed_ids])
    print(
        f"[config] dataset={DATASET_NAME} | model={PREFILTER_MODEL_NAME} | llm={LLM_MODEL_NAME} "
        f"| total_target_records={total_target_records} | chunk_target_records={chunk_target_records} "
        f"| chunk={chunk_context if chunk_context is not None else 'full'} "
        f"| graph={GRAPH_BACKEND} | retrieval={bank.get('semantic_backend')}"
    )

    file_mode = "a" if predictions_path.exists() and processed_ids else "w"
    with predictions_path.open(file_mode, encoding="utf-8") as handle:
        seen = 0
        chunk_processed = already_done_in_chunk
        for record in iter_jsonl(test_path):
            code = get_record_code(record)
            if not code:
                continue
            seen += 1
            if MAX_TEST_SAMPLES is not None and seen > MAX_TEST_SAMPLES:
                break
            if record["record_id"] not in target_record_ids:
                continue
            if record["record_id"] in processed_ids:
                continue

            record_started = time.perf_counter()
            scores = score_map[record["record_id"]]
            calibrated_probability = _apply_saved_calibrator(scores["fusion_score"], calibration)
            band = _risk_band(calibrated_probability, tau_low, tau_high)
            call_llm = _should_call_llm(band)
            direct_prediction = _direct_prefilter_prediction(band)

            graph_latency_ms = 0.0
            retrieval_latency_ms = 0.0
            llm_latency_ms = 0.0
            graph_features = None
            suspicious_context = None
            retrieved_examples = []

            if not call_llm:
                prefilter_reason = "prefilter_direct_positive" if direct_prediction == 1 else "prefilter_skip"
                payload = {
                    "schema_version": PREDICTION_SCHEMA_VERSION,
                    "resolution_status": "resolved",
                    "record_id": record["record_id"],
                    "dataset": record.get("dataset", DATASET_NAME),
                    "ground_truth": int(record["label"]),
                    "prediction": direct_prediction,
                    "decision_source": "prefilter",
                    "risk_band": band,
                    "llm_called": False,
                    "llm_cache_hit": False,
                    "prefilter_fusion_score": float(scores["fusion_score"]),
                    "prefilter_semantic_score": float(scores["semantic_score"]),
                    "prefilter_graph_score": float(scores["graph_score"]),
                    "calibrated_probability": calibrated_probability,
                    "retrieved_examples": [],
                    "graph_backend_used": None,
                    "graph_latency_ms": graph_latency_ms,
                    "retrieval_latency_ms": retrieval_latency_ms,
                    "llm_latency_ms": llm_latency_ms,
                    "record_runtime_ms": float(round((time.perf_counter() - record_started) * 1000.0, 3)),
                    "reason": prefilter_reason,
                }
            else:
                graph_started = time.perf_counter()
                graph_features = get_graph_features({**record, "code": code}, graph_backend=GRAPH_BACKEND)
                graph_latency_ms = float(round((time.perf_counter() - graph_started) * 1000.0, 3))

                suspicious_context = locate_suspicious_slices(
                    code,
                    semantic_score=scores["semantic_score"],
                    graph_score=scores["graph_score"],
                    fusion_score=calibrated_probability,
                    risk_band=band,
                )
                retrieval_started = time.perf_counter()
                retrieved_examples = retrieve_examples(
                    code,
                    bank,
                    total_k=HIGH_RISK_DEMOS if band == "high" else INSPECT_DEMOS,
                    calibrated_probability=calibrated_probability,
                    demo_char_limit=DEMO_CHAR_LIMIT,
                    query_record={**record, "code": code},
                    graph_backend=GRAPH_BACKEND,
                    query_graph_features=graph_features,
                )
                retrieval_latency_ms = float(round((time.perf_counter() - retrieval_started) * 1000.0, 3))
                prompt = build_detection_prompt(
                    {**record, "code": code},
                    retrieved_examples,
                    calibrated_probability,
                    band,
                    graph_features=graph_features,
                    suspicious_context=suspicious_context,
                    semantic_score=scores["semantic_score"],
                    graph_score=scores["graph_score"],
                    fusion_score=scores["fusion_score"],
                )
                llm_started = time.perf_counter()
                result = client.classify(prompt)
                llm_latency_ms = float(round((time.perf_counter() - llm_started) * 1000.0, 3))
                verified_prediction, evidence_status, verified_reason = _verified_llm_prediction(result)
                payload = {
                    "schema_version": PREDICTION_SCHEMA_VERSION,
                    "resolution_status": "resolved",
                    "record_id": record["record_id"],
                    "dataset": record.get("dataset", DATASET_NAME),
                    "ground_truth": int(record["label"]),
                    "prediction": int(verified_prediction),
                    "decision_source": "llm",
                    "risk_band": band,
                    "llm_called": True,
                    "llm_cache_hit": bool(result.get("cached")),
                    "prefilter_fusion_score": float(scores["fusion_score"]),
                    "prefilter_semantic_score": float(scores["semantic_score"]),
                    "prefilter_graph_score": float(scores["graph_score"]),
                    "calibrated_probability": calibrated_probability,
                    "retrieved_examples": [row["record_id"] for row in retrieved_examples],
                    "graph_backend_used": graph_features.get("backend"),
                    "graph_latency_ms": graph_latency_ms,
                    "retrieval_latency_ms": retrieval_latency_ms,
                    "llm_latency_ms": llm_latency_ms,
                    "record_runtime_ms": float(round((time.perf_counter() - record_started) * 1000.0, 3)),
                    "reason": verified_reason or result["reason"],
                    "llm_label": result["label"],
                    "llm_confidence": float(result["confidence"]),
                    "llm_evidence_status": evidence_status,
                    "llm_cwe_family": result.get("cwe_family"),
                    "llm_vulnerable_lines": result.get("vulnerable_lines"),
                    "llm_sink_or_api": result.get("sink_or_api"),
                    "llm_missing_guard": result.get("missing_guard"),
                    "suspicious_top_lines": suspicious_context.get("top_lines"),
                }

            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")
            handle.flush()
            processed_ids.add(record["record_id"])
            chunk_processed += 1
            print(
                f"[progress] global={len(processed_ids)}/{total_target_records} "
                f"| chunk={chunk_processed}/{chunk_target_records} | record_id={record['record_id']} "
                f"| band={band} | decision={payload['decision_source']} | calibrated={calibrated_probability:.4f}"
            )

    summary = _summarize_predictions(
        predictions_path,
        total_target_records,
        output_dir,
        bank,
        run_signature=run_signature,
        chunk_context=chunk_context,
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### File `08_evaluate_predictions.py`


In [ ]:
%%writefile GRACE-improve/baseline/baseline2/08_evaluate_predictions.py
import json
import os
from pathlib import Path

from evaluate_predictions import evaluate_prediction_artifacts, write_evaluation_summary


DATASET_NAME = os.getenv("GRACE_DATASET", "devign")
VARIANT_SUFFIX = os.getenv("GRACE_VARIANT_OUTPUT_SUFFIX", "").strip().lower()
PREDICTION_FILE_STEM = os.getenv("GRACE_PREDICTION_FILE_STEM") or (f"grace_hybrid_predictions_{VARIANT_SUFFIX}" if VARIANT_SUFFIX else "grace_hybrid_predictions")
RUN_STATE_FILE_STEM = os.getenv("GRACE_RUN_STATE_FILE_STEM") or (f"grace_hybrid_run_state_{VARIANT_SUFFIX}" if VARIANT_SUFFIX else "grace_hybrid_run_state")
METRICS_FILE_STEM = os.getenv("GRACE_EVALUATION_FILE_STEM") or (f"grace_hybrid_evaluation_summary_{VARIANT_SUFFIX}" if VARIANT_SUFFIX else "grace_hybrid_evaluation_summary")
PREDICTIONS_PATH = Path(os.getenv("GRACE_PREDICTIONS_PATH", f"GRACE-improve/baseline/baseline2/artifacts/predictions/{DATASET_NAME}/{PREDICTION_FILE_STEM}.jsonl"))
RUN_STATE_PATH = Path(os.getenv("GRACE_RUN_STATE_PATH", f"GRACE-improve/baseline/baseline2/artifacts/predictions/{DATASET_NAME}/{RUN_STATE_FILE_STEM}.json"))
BASELINE_COMPARE_PATH = Path(os.getenv("GRACE_BASELINE_COMPARE_PATH")) if os.getenv("GRACE_BASELINE_COMPARE_PATH") else None


def main() -> None:
    _, _, metrics = evaluate_prediction_artifacts(
        PREDICTIONS_PATH,
        RUN_STATE_PATH,
        dataset_name=DATASET_NAME,
        baseline_compare_path=BASELINE_COMPARE_PATH,
        expected_schema_version=1,
    )
    output_path = write_evaluation_summary(metrics, dataset_name=DATASET_NAME, filename=f"{METRICS_FILE_STEM}.json")
    payload = {
        "output_path": str(output_path),
        "metrics": metrics,
    }
    print(json.dumps(payload, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## Runtime Sanity Check


In [ ]:
import importlib
import json

versions = {}
for module_name in ['torch', 'tensorflow', 'transformers', 'bitsandbytes', 'pandas', 'numpy', 'sklearn']:
    try:
        module = importlib.import_module(module_name)
        versions[module_name] = getattr(module, '__version__', 'unknown')
    except Exception as exc:
        versions[module_name] = f'not available: {exc}'

try:
    import torch
    versions['cuda_available'] = bool(torch.cuda.is_available())
    versions['cuda_device_count'] = int(torch.cuda.device_count())
    versions['cuda_device_name'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    pass

report = {
    'versions': versions,
    'top_level_kaggle_inputs': [str(path) for path in sorted(KAGGLE_INPUT_ROOT.iterdir())] if KAGGLE_INPUT_ROOT.exists() else [],
}
print(json.dumps(report, indent=2))


## Step 00 - Verify Assets


In [ ]:
run_baseline2('00_verify_assets.py')


## Step 01 - Prepare Datasets


In [ ]:
run_baseline2('01_prepare_datasets.py')


## Step 02 - Create Splits


In [ ]:
run_baseline2('02_create_splits.py')


## Step 03 - Build Feature Store


In [ ]:
run_baseline2('03_build_feature_store.py')


## Step 04 - Train Hybrid Prefilter


In [ ]:
prefilter_dir = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'models' / DATASET_NAME / PREFILTER_MODEL_NAME
prefilter_ready = (prefilter_dir / 'config.json').exists() and (prefilter_dir / 'weights.weights.h5').exists()
if prefilter_ready:
    print(f'Skipping training because prefilter artifacts already exist at {prefilter_dir}')
else:
    run_baseline2('04_train_hybrid_prefilter.py')


## Step 05 - Calibrate Budget Controller


In [ ]:
import json

calibration_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'models' / DATASET_NAME / f'calibration.{PREFILTER_MODEL_NAME}.json'
calibration_matches = False
if calibration_path.exists():
    try:
        existing_calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
        calibration_matches = (
            float(existing_calibration.get('target_recall', -1.0)) == float(TARGET_RECALL)
            and float(existing_calibration.get('direct_accept_min_probability', -1.0)) == float(DIRECT_ACCEPT_MIN_PROBABILITY)
            and str(existing_calibration.get('high_risk_threshold_strategy', '')).strip().lower() == str(HIGH_RISK_THRESHOLD_STRATEGY).strip().lower()
            and float(existing_calibration.get('high_risk_target_precision', -1.0)) == float(HIGH_RISK_TARGET_PRECISION)
        )
    except Exception:
        calibration_matches = False
if calibration_matches:
    print(f'Skipping calibration because artifact already matches config at {calibration_path}')
else:
    run_baseline2('05_calibrate_budget_controller.py')


## Step 06 - Build Demo Bank


In [ ]:
demo_bank_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'retrieval' / DATASET_NAME / 'demo_bank.joblib'
if demo_bank_path.exists():
    print(f'Skipping demo bank build because artifact already exists at {demo_bank_path}')
else:
    run_baseline2('06_build_demo_bank.py')


## Step 07 - Run GRACE Hybrid


In [ ]:
import json
import math

from common import get_record_code, iter_jsonl

variant_suffix = globals().get('GRACE_VARIANT_OUTPUT_SUFFIX', '')
prediction_file_stem = globals().get('GRACE_PREDICTION_FILE_STEM') or (f'grace_hybrid_predictions_{variant_suffix}' if variant_suffix else 'grace_hybrid_predictions')
run_state_file_stem = globals().get('GRACE_RUN_STATE_FILE_STEM') or (f'grace_hybrid_run_state_{variant_suffix}' if variant_suffix else 'grace_hybrid_run_state')
test_path = WORKING_CODE_ROOT / 'baseline' / 'artifacts' / 'splits' / DATASET_NAME / 'test.jsonl'
predictions_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'predictions' / DATASET_NAME / f'{prediction_file_stem}.jsonl'
run_state_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'predictions' / DATASET_NAME / f'{run_state_file_stem}.json'


def load_test_record_ids(path):
    record_ids = []
    for record in iter_jsonl(path):
        if get_record_code(record):
            record_ids.append(str(record['record_id']))
    return record_ids


def load_prediction_ids(path):
    if not path.exists():
        return set()
    rows = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return {str(row['record_id']) for row in rows if row.get('schema_version') == 1}


all_test_ids = load_test_record_ids(test_path)
processed_ids = load_prediction_ids(predictions_path)
remaining_ids = [record_id for record_id in all_test_ids if record_id not in processed_ids]

if not all_test_ids:
    raise RuntimeError(f'No valid test records found at {test_path}')

if TEST_CHUNK_SIZE is None or TEST_CHUNK_SIZE <= 0:
    print('Chunking disabled. Running Step 07 on the full unresolved test set.')
    run_baseline2(
        '07_run_grace_hybrid.py',
        extra_env={
            'GRACE_MAX_TEST_SAMPLES': None,
            'GRACE_TEST_CHUNK_SIZE': None,
            'GRACE_TEST_CHUNK_INDEX': None,
        },
    )
else:
    num_chunks = int(math.ceil(len(all_test_ids) / TEST_CHUNK_SIZE))
    remaining_chunk_indices = []
    for chunk_index in range(num_chunks):
        chunk_ids = all_test_ids[chunk_index * TEST_CHUNK_SIZE : (chunk_index + 1) * TEST_CHUNK_SIZE]
        if any(record_id not in processed_ids for record_id in chunk_ids):
            remaining_chunk_indices.append(chunk_index)

    status = {
        'dataset': DATASET_NAME,
        'total_test_records': len(all_test_ids),
        'processed_records': len(processed_ids),
        'remaining_records': len(remaining_ids),
        'test_chunk_size': TEST_CHUNK_SIZE,
        'num_chunks': num_chunks,
        'remaining_chunk_indices': remaining_chunk_indices,
        'run_all_test_chunks_in_one_run': RUN_ALL_TEST_CHUNKS_IN_ONE_RUN,
        'predictions_path': str(predictions_path),
        'run_state_path': str(run_state_path),
    }
    print(json.dumps(status, indent=2))

    if not remaining_chunk_indices:
        print('All test chunks are already processed. Skipping Step 07.')
    else:
        chunk_indices_to_run = remaining_chunk_indices if RUN_ALL_TEST_CHUNKS_IN_ONE_RUN else [remaining_chunk_indices[0]]
        for chunk_index in chunk_indices_to_run:
            print(f'Running chunk {chunk_index + 1}/{num_chunks} with chunk_size={TEST_CHUNK_SIZE}')
            run_baseline2(
                '07_run_grace_hybrid.py',
                extra_env={
                    'GRACE_MAX_TEST_SAMPLES': None,
                    'GRACE_TEST_CHUNK_SIZE': TEST_CHUNK_SIZE,
                    'GRACE_TEST_CHUNK_INDEX': chunk_index,
                },
            )


## Step 08 - Evaluate Predictions


In [ ]:
import json

variant_suffix = globals().get('GRACE_VARIANT_OUTPUT_SUFFIX', '')
prediction_file_stem = globals().get('GRACE_PREDICTION_FILE_STEM') or (f'grace_hybrid_predictions_{variant_suffix}' if variant_suffix else 'grace_hybrid_predictions')
run_state_file_stem = globals().get('GRACE_RUN_STATE_FILE_STEM') or (f'grace_hybrid_run_state_{variant_suffix}' if variant_suffix else 'grace_hybrid_run_state')
run_state_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'predictions' / DATASET_NAME / f'{run_state_file_stem}.json'
if not run_state_path.exists():
    print(f'Skipping evaluation because run_state does not exist yet: {run_state_path}')
else:
    run_state = json.loads(run_state_path.read_text(encoding='utf-8'))
    if bool(run_state.get('complete')):
        run_baseline2('08_evaluate_predictions.py')
    else:
        payload = {
            'message': 'Skipping evaluation because not all test chunks are complete yet.',
            'resolved_samples': run_state.get('resolved_samples'),
            'target_samples': run_state.get('target_samples'),
            'chunking': run_state.get('chunking'),
            'predictions_path': run_state.get('predictions_path'),
        }
        print(json.dumps(payload, indent=2))


## Final Summary


In [ ]:
import json
import math

variant_suffix = globals().get('GRACE_VARIANT_OUTPUT_SUFFIX', '')
prediction_file_stem = globals().get('GRACE_PREDICTION_FILE_STEM') or (f'grace_hybrid_predictions_{variant_suffix}' if variant_suffix else 'grace_hybrid_predictions')
run_state_file_stem = globals().get('GRACE_RUN_STATE_FILE_STEM') or (f'grace_hybrid_run_state_{variant_suffix}' if variant_suffix else 'grace_hybrid_run_state')
metrics_file_stem = globals().get('GRACE_EVALUATION_FILE_STEM') or (f'grace_hybrid_evaluation_summary_{variant_suffix}' if variant_suffix else 'grace_hybrid_evaluation_summary')
metrics_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'metrics' / DATASET_NAME / f'{metrics_file_stem}.json'
run_state_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'predictions' / DATASET_NAME / f'{run_state_file_stem}.json'
predictions_path = WORKING_CODE_ROOT / 'baseline' / 'baseline2' / 'artifacts' / 'predictions' / DATASET_NAME / f'{prediction_file_stem}.jsonl'

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    preview = {
        'dataset': metrics.get('dataset'),
        'samples': metrics.get('samples'),
        'accuracy': metrics.get('accuracy'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'f1': metrics.get('f1'),
        'roc_auc': metrics.get('roc_auc'),
        'pr_auc': metrics.get('pr_auc'),
        'llm_calls': metrics.get('llm_calls'),
        'llm_call_ratio': metrics.get('llm_call_ratio'),
        'routing': metrics.get('routing'),
        'decision_sources': metrics.get('decision_sources'),
        'metrics_path': str(metrics_path),
        'run_state_path': str(run_state_path),
        'predictions_path': str(predictions_path),
    }
    print(json.dumps(preview, indent=2))
elif run_state_path.exists():
    run_state = json.loads(run_state_path.read_text(encoding='utf-8'))
    target_samples = int(run_state.get('target_samples') or 0)
    resolved_samples = int(run_state.get('resolved_samples') or 0)
    remaining = max(0, target_samples - resolved_samples)
    chunk_size = int((run_state.get('chunking') or {}).get('chunk_size') or TEST_CHUNK_SIZE or 0)
    remaining_chunks_estimate = int(math.ceil(remaining / chunk_size)) if chunk_size > 0 else None
    preview = {
        'message': 'Evaluation summary is not available yet because full test-set chunking is still in progress.',
        'dataset': run_state.get('dataset'),
        'resolved_samples': resolved_samples,
        'target_samples': target_samples,
        'remaining_samples': remaining,
        'remaining_chunks_estimate': remaining_chunks_estimate,
        'chunking': run_state.get('chunking'),
        'predictions_path': str(predictions_path),
        'run_state_path': str(run_state_path),
    }
    print(json.dumps(preview, indent=2))
else:
    raise FileNotFoundError(f'Neither metrics nor run_state file exists yet under {predictions_path.parent}')
